# Query, key and value subspaces of attention heads: an invariant, chance-calibrated comparison across pre-trained Transformers

This notebook produces every number, table and figure of the paper. Each number in the paper source is a
placeholder `[[RESULT:id]]`; the cell that computes it calls `R.set("id", ...)` and is labelled with
`R.cell("<section>.<cell>")`, and the paper lists the cells it uses. All results are written to
`outputs/results.json` (id, producing cell, value, LaTeX text), figures to `outputs/figures` and tables to
`outputs/tables`. If the paper source is placed at `paper/qkv_geometry_pretrained.tex`, the last section (Z) also writes
`paper/qkv_geometry_pretrained_filled.tex`, where placeholders without a value appear as yellow `\pending{id}` boxes.

**Sections**

| Section | Content |
|---|---|
| 0–1 | Configuration, results registry, geometry library (weight extraction, principal angles, nulls, statistics, training helpers) |
| A | Model-free checks: gauge invariance, invariance table, analytical null, RoPE proposition, toy figures, validation of the weight extraction |
| B0 | The twelve reference checkpoints: raw versus calibrated overlaps, ordering tests, coupling, depth profiles; comparison of similarity measures on trained weights |
| B1 | Seed noise floor of the 25 MultiBERTs |
| B2 | Model size at a fixed recipe (24 BERT miniatures, Pythia sizes), threshold or gradual change, distilled models, Pythia training checkpoints |
| B3 | Rotary embeddings: effective bilinear forms per offset and per frequency plane; activation-level analysis |
| B6 | Broader architectures (encoder-decoders, mixture-of-experts, 7-8B GQA models) and family-level tests |
| Z | Results summary, archive, fill of the paper source |

**Running on Kaggle.** *Settings → Accelerator → GPU T4 x2*; *Settings → Internet → On*; *Add-ons → Secrets*:
add a secret `HF_TOKEN` (a Hugging Face token with access to the gated Llama checkpoints) and tick it for this
notebook; then *Save Version → Save & Run All*. The first cells stop at once if the GPU, Internet or the token
is missing. A session stops cleanly before Kaggle's 12-hour limit; the last cell then prints `STILL TO RUN`.
Attach the output of the finished version with *Add input → Your Work → this notebook* and run again:
completed sections are skipped and interrupted trainings resume from their last checkpoint.

**Running elsewhere.** `pip install -r requirements.txt`, `export HF_TOKEN=...`, then execute the notebook
(e.g. `jupyter nbconvert --to notebook --execute qkv_geometry_pretrained.ipynb --ExecutePreprocessor.timeout=-1`).
`QKV_SECTIONS` selects sections (e.g. `QKV_SECTIONS="B1,Z"`); results accumulate in `outputs/results.json`.
`QKV_SMOKE=1` replaces every checkpoint by a tiny random model of the same architecture and every corpus by
random tokens: it runs all code paths on a CPU in minutes (outputs in `outputs_smoke/`), but its numbers are
meaningless. Never write the token in the notebook.

### 0.0 Run configuration (the only cell to edit)
* `RUN_SECTIONS`: sections run in this session (default: all). A Kaggle session lasts at most 12 h; the
  notebook stops cleanly before the limit and the next session continues where it stopped.
* `RUN_RESUME_FROM`: outputs of earlier sessions. With `auto` (default), every `outputs/` folder containing a
  `results.json` under `/kaggle/input` is used (attach the output of the previous version of this
  notebook with *Add input*): results are merged and caches (geometry, tokenised corpora, trained and
  partially trained models) are reused, so nothing is recomputed.
* `HF_TOKEN` is read from the Kaggle secret of that name when it is attached (*Add-ons → Secrets*);
  never write the token in the notebook.

In [ ]:
import os
NOTEBOOK = "Query, key and value subspaces of attention heads: an invariant, chance-calibrated comparison across pre-trained Transformers (notebook, build 2026-11-25)"
PAPER_TEX = "qkv_geometry_pretrained.tex"   # paper/qkv_geometry_pretrained.tex with [[RESULT:id]] placeholders is filled by section Z if present
RUN_SECTIONS = None      # e.g. "B3,Z"; None: environment variable QKV_SECTIONS if set, otherwise all sections
RUN_RESUME_FROM = None   # e.g. "/kaggle/input/<notebook>/outputs"; None: QKV_RESUME_FROM if set, otherwise "auto"
ALLOW_MISSING_GATED = False   # True: run even if HF_TOKEN gives no access to the gated checkpoints (Llama);
                              # they are then skipped and listed as missing. False: stop at once instead.
if ALLOW_MISSING_GATED:
    os.environ["QKV_ALLOW_MISSING_GATED"] = "1"
if RUN_SECTIONS is not None:
    os.environ["QKV_SECTIONS"] = RUN_SECTIONS
if RUN_RESUME_FROM is not None:
    os.environ["QKV_RESUME_FROM"] = RUN_RESUME_FROM
os.environ.setdefault("QKV_RESUME_FROM", "auto")
_why = ""
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except ImportError:
        _why = "no environment variable HF_TOKEN"
    except Exception as e:
        _why = (f"the Kaggle secret HF_TOKEN could not be read ({type(e).__name__}): Add-ons -> Secrets, "
                "tick HF_TOKEN for this notebook")
print(NOTEBOOK)
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else f"NOT set ({_why or 'empty secret'})")

In [ ]:
import os, sys, json, math, time, re, gc, itertools, warnings, hashlib, random, functools, shutil, subprocess
import copy, threading, queue, collections
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd
import scipy
import scipy.stats as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SMOKE = os.environ.get("QKV_SMOKE", "0") == "1"
SECTIONS = os.environ.get("QKV_SECTIONS", "ALL").upper().replace(" ", "").split(",")
# Time budget: a Kaggle session is stopped after 12 h and a stopped "Save & Run All" keeps no output.
# Past QKV_TIME_BUDGET_H (default 11.25 h on Kaggle, unlimited elsewhere) long jobs save their state and
# stop, the remaining sections are skipped and section Z still archives everything for the next session.
T_START = time.time()
TIME_BUDGET_S = float(os.environ.get("QKV_TIME_BUDGET_H", "11.25" if Path("/kaggle/working").exists() else "inf")) * 3600

class TimeBudgetExceeded(Exception):
    pass

def time_left() -> float:
    return TIME_BUDGET_S - (time.time() - T_START)

def check_budget(what: str = ""):
    if time_left() < 0:
        raise TimeBudgetExceeded(f"time budget of {TIME_BUDGET_S / 3600:.2f} h reached {what}".strip())

# ---------- network: every section except A and Z loads checkpoints from the Hugging Face Hub ----------
import requests
NET_FREE_SECTIONS = {"A", "Z"}
HUB_PROBE = os.environ.get("HF_ENDPOINT", "https://huggingface.co").rstrip("/") + "/api/models/openai-community/gpt2"
NET_DOWN = threading.Event()          # set when the Hub was found unreachable; sections re-check before starting

class NetworkUnavailable(RuntimeError):
    """The Hub cannot be reached. Never treated as a missing checkpoint: the cell stops and its results stay
    pending, so that no statistic is computed on a set of models silently reduced by a network failure."""

_NET_TYPES = {"ConnectError", "ConnectTimeout", "ReadTimeout", "WriteTimeout", "PoolTimeout", "NetworkError",
              "RemoteProtocolError", "TimeoutException", "ConnectionError", "Timeout", "ReadTimeoutError",
              "MaxRetryError", "NewConnectionError", "ProtocolError", "gaierror", "LocalEntryNotFoundError",
              "OfflineModeIsEnabled", "ChunkedEncodingError", "IncompleteRead", "SSLError"}
_NET_TEXT = re.compile(r"name resolution|temporary failure|could ?n[o']t connect|failed to establish|"
                       r"connection (refused|reset|aborted)|network is unreachable|timed out|max retries exceeded|"
                       r"remote end closed|cannot find the requested files in the local cache|offline mode|"
                       r"HTTP 5\d\d|\b(502|503|504) (server|bad gateway|service|gateway)", re.I)

def is_network_error(e) -> bool:
    """True for DNS, connection, timeout, 429 and 5xx errors (anywhere in the exception chain)."""
    for _ in range(8):
        if e is None:
            return False
        if isinstance(e, (NetworkUnavailable, ConnectionError, TimeoutError)) or any(
                c.__name__ in _NET_TYPES for c in type(e).__mro__):          # e.g. requests ProxyError, httpx errors
            return True
        if getattr(getattr(e, "response", None), "status_code", None) in (429, 500, 502, 503, 504):
            return True
        if _NET_TEXT.search(str(e)):
            return True
        e = e.__cause__ or e.__context__
    return False

def _chain(e, n=8):
    while e is not None and n > 0:
        yield e
        e = e.__cause__ or e.__context__; n -= 1

def is_resource_error(e) -> bool:
    """GPU out of memory or full disk (a property of the run, not of the checkpoint)."""
    return any(type(x).__name__ in ("OutOfMemoryError",) or re.search(r"out of memory|No space left on device|not enough disk",
               str(x), re.I) for x in _chain(e))

_UNAVAILABLE = {"GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError", "EntryNotFoundError",
                "RemoteEntryNotFoundError", "FileNotFoundError"}
def is_unavailable_checkpoint(e) -> bool:
    """The checkpoint itself cannot be obtained (absent, gated without access, no weight file)."""
    return any(type(x).__name__ in _UNAVAILABLE or isinstance(x, PermissionError) or re.search(
        r"is not a valid model identifier|does not appear to have a file named|gated", str(x)) for x in _chain(e))

def hub_reachable(tries: int = 3, wait: float = 10):
    """(True, "") if the Hub answers, else (False, reason)."""
    why = ""
    for i in range(tries):
        try:
            r = requests.get(HUB_PROBE, timeout=20)
            if r.status_code < 500:
                return True, ""
            why = f"HTTP {r.status_code}"
        except Exception as e:
            why = f"{type(e).__name__}: {str(e)[:160]}"
        if i < tries - 1:
            time.sleep(wait)
    return False, why

def hub_retry(fn, what: str, waits=(15, 60, 180)):
    """fn() with retries on network errors. NetworkUnavailable if the Hub stays unreachable (about 4 min);
    any other error (absent repository, gated without access, ...) is raised at once."""
    last = None
    for i in range(len(waits) + 1):
        try:
            return fn()
        except NetworkUnavailable:
            raise
        except Exception as e:
            if not is_network_error(e):
                raise
            last = e
        if i == len(waits) or NET_DOWN.is_set():
            break
        print(f"[network] {what}: {type(last).__name__}: {str(last)[:120]}; retry {i + 1}/{len(waits)} in {waits[i]} s",
              flush=True)
        time.sleep(waits[i])
    NET_DOWN.set()
    raise NetworkUnavailable(f"{what}: the Hugging Face Hub cannot be reached ({type(last).__name__}: {str(last)[:160]}). "
                             "On Kaggle: Settings -> Internet -> On, then run again.") from last

def _needs_network() -> bool:
    return not SMOKE and os.environ.get("QKV_ALLOW_OFFLINE", "0") != "1" and (
        "ALL" in SECTIONS or bool(set(SECTIONS) - NET_FREE_SECTIONS))

def require_network():
    """Stop at once (instead of skipping every checkpoint for hours) when the Hub cannot be reached."""
    if not _needs_network():
        return
    ok, why = hub_reachable()
    if not ok:
        raise RuntimeError("The Hugging Face Hub (huggingface.co) cannot be reached (" + why + "), so no checkpoint can "
                           "be loaded. On Kaggle: Settings -> Internet -> On (right-hand panel: Session options -> "
                           "Internet; the account must be phone-verified), then run again. "
                           "(QKV_ALLOW_OFFLINE=1 overrides.)")
    print("Hugging Face Hub: reachable")

GPU_SECTIONS = {"B2", "B3", "B4", "B5", "B7", "B8"}     # training or full forward passes

def require_gpu():
    """Stop at once (instead of running for 12 h on a CPU) when sections that need a GPU are requested
    but no GPU is visible. Kaggle: Settings -> Accelerator -> GPU T4 x2. QKV_ALLOW_CPU=1 overrides."""
    if SMOKE or torch.cuda.is_available() or os.environ.get("QKV_ALLOW_CPU", "0") == "1":
        return
    wanted = sorted(GPU_SECTIONS) if "ALL" in SECTIONS else sorted(GPU_SECTIONS & set(SECTIONS))
    if wanted:
        raise RuntimeError("No GPU is visible, but sections " + ", ".join(wanted) + " train models or run full "
                           "forward passes and would take days on a CPU. On Kaggle: Settings -> Accelerator -> "
                           "GPU T4 x2, then run again. (Set QKV_ALLOW_CPU=1 to override.)")

# Cells of each section in this notebook (written by the build script). A section whose cells all completed
# without error in an earlier session (outputs/cells_done.json, carried over by the resume) is skipped, so the
# same notebook can be run session after session until everything is done. QKV_RERUN=1 recomputes.
SECTION_CELLS: Dict[str, List[str]] = {'A': ['A.1', 'A.2', 'A.3', 'A.4', 'A.5', 'A.6'], 'B0': ['B0.1', 'B0.2', 'B0.3'], 'B1': ['B1.1'], 'B2': ['B2.1', 'B2.2', 'B2.3', 'B2.4'], 'B3': ['B3.1', 'B3.2'], 'B6': ['B6.1', 'B6.2'], 'Z': ['Z.1']}
CELLS_DONE: Dict[str, object] = {}
_SKIP_SAID = set()
# Version of a cell's code; a completion recorded for an older version does not count. 2026-10-27: B2.5 (learning-
# rate warm-up, three seeds), B7.2 and B8.3 (SST-2 loading fixed; a failed fine-tune now stops the cell), B7.1 and
# B8.2 (pair names of the extreme values reported; B7.1: random control of the same pair).
CELL_VERSION = {"B2.5": 2, "B7.1": 2, "B7.2": 2, "B8.2": 2, "B8.3": 2}

def cell_is_done(cell: str) -> bool:
    e = CELLS_DONE.get(cell)
    if e is None:
        return False
    v = e.get("version", 1) if isinstance(e, dict) else 1
    return v == CELL_VERSION.get(cell, 1)

def section_done(name: str) -> bool:
    cells = SECTION_CELLS.get(name.upper(), [])
    return bool(cells) and all(cell_is_done(c) for c in cells) and os.environ.get("QKV_RERUN", "0") != "1"

def mark_cell_done(cell: str):
    CELLS_DONE[cell] = dict(time=time.strftime("%Y-%m-%d %H:%M:%S"), version=CELL_VERSION.get(cell, 1))
    (OUT / "cells_done.json").write_text(json.dumps(CELLS_DONE, indent=1))

def section_enabled(name: str) -> bool:
    on = "ALL" in SECTIONS or name.upper() in SECTIONS
    if on and name.upper() != "Z" and section_done(name):
        if name.upper() not in _SKIP_SAID:
            _SKIP_SAID.add(name.upper())
            print(f"[resume] section {name} already completed in an earlier session "
                  f"({', '.join(SECTION_CELLS[name.upper()])}): skipped, its results are kept")
        return False
    if on and name.upper() != "Z" and time_left() < 0:
        if ("budget", name.upper()) not in _SKIP_SAID:
            _SKIP_SAID.add(("budget", name.upper()))
            print(f"[time budget] section {name} skipped (run it in the next session)")
        return False
    if on and name.upper() not in NET_FREE_SECTIONS and NET_DOWN.is_set():
        ok, why = hub_reachable(tries=4, wait=30)
        if not ok:
            print(f"[network] section {name} skipped: the Hugging Face Hub is still unreachable ({why})")
            return False
        NET_DOWN.clear()
        print("[network] the Hugging Face Hub is reachable again")
    return on

SEED = 20260105
PROJECT_DIR = Path(os.environ.get("QKV_PROJECT_DIR", ".")).resolve()
OUT = PROJECT_DIR / ("outputs_smoke" if SMOKE else "outputs")
FIG, TAB, CACHE = OUT / "figures", OUT / "tables", OUT / "cache"
for _d in (FIG, TAB, CACHE, CACHE / "geometry", CACHE / "data"):
    _d.mkdir(parents=True, exist_ok=True)
def _resume_sources():
    spec = os.environ.get("QKV_RESUME_FROM", "").strip()
    if not spec or (SMOKE and os.environ.get("QKV_TEST_RESUME", "0") != "1"):
        return []
    if spec == "auto":
        base = Path("/kaggle/input")
        cands = [r.parent for pat in ("*/outputs/results.json", "*/*/outputs/results.json", "*/*/*/outputs/results.json")
                 for r in base.glob(pat)] if base.exists() else []
    else:
        cands = [Path(x.strip()) for x in spec.split(",") if x.strip()]
    cands = [c for c in dict.fromkeys(cands) if (c / "results.json").exists() and c.resolve() != OUT.resolve()]
    n_res = lambda c: len(json.loads((c / "results.json").read_text()))
    return sorted(cands, key=lambda c: ((c / "results.json").stat().st_mtime, n_res(c)))    # older first

_done_marker = OUT / ".resumed_from.json"
_done = json.loads(_done_marker.read_text()) if _done_marker.exists() else []
_srcs = [c for c in _resume_sources() if str(c) not in _done]
if _srcs:
    _here = json.loads((OUT / "results.json").read_text()) if (OUT / "results.json").exists() else {}
    _merged = {}
    for _src in _srcs:                    # older sources first, newer ones override
        _prev = json.loads((_src / "results.json").read_text())
        _merged.update(_prev)
        shutil.copytree(_src, OUT, dirs_exist_ok=True)
        _done.append(str(_src))
        print(f"resumed {len(_prev)} results and the caches from {_src}")
    _merged.update(_here)                 # results already produced in this session take precedence
    (OUT / "results.json").write_text(json.dumps(_merged, indent=1))
    _done_marker.write_text(json.dumps(_done))
    _cd = {}
    for _src in _srcs:
        if (_src / "cells_done.json").exists():
            _cd.update(json.loads((_src / "cells_done.json").read_text()))
    if _cd:
        (OUT / "cells_done.json").write_text(json.dumps(_cd, indent=1))
if (OUT / "cells_done.json").exists():
    CELLS_DONE.update(json.loads((OUT / "cells_done.json").read_text()))
    print(f"cells completed in earlier sessions: {', '.join(sorted(c for c in CELLS_DONE if cell_is_done(c))) or 'none'}")
    _redo = sorted(c for c in CELLS_DONE if not cell_is_done(c))
    if _redo:
        print(f"cells run again because their code changed since: {', '.join(_redo)}")

def result_cache(kind: str, *key) -> Path:
    """Disk cache of an expensive per-model computation, kept in outputs/cache so that a later session (after a
    resume) reuses it instead of recomputing it. The key must contain every setting the result depends on."""
    h = hashlib.sha1("|".join(map(str, key)).encode()).hexdigest()[:16]
    d = CACHE / "results" / kind
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{h}.pkl"
PAPER_DIR = PROJECT_DIR / "paper"
# Weight cache. On Kaggle, /kaggle/working is limited (20 GB) and counts towards the saved output, so the
# Hub cache defaults to /tmp there. Large checkpoints are not stored at all: attention tensors are streamed
# from the safetensors files by HTTP range requests (see 1.2).
_ON_KAGGLE = Path("/kaggle/working").exists()
HF_CACHE = Path(os.environ.get("QKV_HF_CACHE", os.environ.get("HF_HUB_CACHE",
                "/tmp/hf_cache" if _ON_KAGGLE else str(PROJECT_DIR / "hf_cache"))))
HF_CACHE.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
# Mixed precision for the training experiments: bf16 where supported (A100/H100/L4), otherwise fp16 with
# loss scaling (T4/P100, e.g. on Kaggle); geometry itself is always computed in float64.
AMP_DTYPE = torch.bfloat16 if GPU_BF16 else (torch.float16 if DEVICE == "cuda" else None)

def amp_ctx():
    return torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None)

def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

def amp_step(loss, opt, params, scaler, clip=1.0):
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, clip)
    scaler.step(opt)
    scaler.update()

# Budget knobs (full run / smoke run). Every experiment reads its sizes from here.
CFG = dict(
    n_boot=2000 if not SMOKE else 30,          # bootstrap replicates
    n_perm=10000 if not SMOKE else 50,         # permutation replicates
    n_null_mc=20000 if not SMOKE else 200,     # Monte Carlo draws for null checks
    eval_tokens=65536 if not SMOKE else 512,   # WikiText-103 tokens for perplexity
    act_tokens=10240 if not SMOKE else 256,    # tokens for activation-level analysis
    ctx_eval=512 if not SMOKE else 32,
    rope_offsets=[0, 1, 4, 16, 64, 256],
    topr=8 if not SMOKE else 2,                # dominant-subspace rank for B_delta analysis
    # from-scratch models (B3/B4)
    scratch=dict(d=384, L=8, H=6, ctx=256, batch=64, steps=int(os.environ.get("QKV_SCRATCH_STEPS", 10000)), lr=6e-4, warmup=500,
                 train_tokens=170_000_000, val_tokens=1_000_000, log_every=500, ckpt_every=1000, micro=16) if not SMOKE else
            dict(d=32, L=2, H=2, ctx=16, batch=4, steps=6, lr=1e-3, warmup=2,
                 train_tokens=4000, val_tokens=600, log_every=3, ckpt_every=2, micro=2),
    # micro: sequences per forward/backward pass; gradients of batch/micro passes are accumulated, so every
    # optimiser step still uses `batch` sequences (the 50k-way output layer does not fit a 16 GB GPU at 64 x 256)
    scratch_seeds=[0, 1, 2, 3, 4] if not SMOKE else [0, 1, 2],   # 5 vs 5 runs: smallest exact two-sided p = 2/252
    scratch_gqa_kv=2 if not SMOKE else 1,                          # GQA condition: 6 query heads share 2 KV heads
    reg_lambdas=[-0.1, -0.03, 0.0, 0.03, 0.1] if not SMOKE else [-0.1, 0.0, 0.1],
    reg_seeds=[0, 1] if not SMOKE else [0],
    surgery_t=[0.0, 0.25, 0.5, 0.75, 1.0] if not SMOKE else [0.0, 1.0],
    surgery_random_draws=39 if not SMOKE else 4,     # random draws at t < 1 (figure: mean and 2.5-97.5% range)
    surgery_random_draws_t1=399 if not SMOKE else 9,  # draws at t = 1, where the test is made: smallest two-sided
                                                      # p = 2/400 = 0.005, i.e. 0.025 after Holm over 5 models
    head_seq_len=256 if not SMOKE else 24,
    head_n_seq=32 if not SMOKE else 2,
    induction_T=50 if not SMOKE else 8,
    glue_epochs=2 if not SMOKE else 1,
    glue_max_train=None if not SMOKE else 16,
    glue_max_eval=None if not SMOKE else 16,
    mlm_eval_tokens=100_000 if not SMOKE else 512,
    distill=dict(steps=5000, batch=64, ctx=128, lr=5e-4, warmup=500, mu=1.0, ckpt_every=250) if not SMOKE else
            dict(steps=3, batch=2, ctx=16, lr=1e-3, warmup=1, mu=1.0, ckpt_every=1),
    distill_seeds=[0, 1, 2] if not SMOKE else [0, 1],
)

def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed % (2**32 - 1)); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def stable_hash(*parts) -> int:
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:8], 16)

seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = False   # geometry is computed in float64 anyway

# ---------- several GPUs: one independent job per GPU at a time ----------
def _visible_gpus():
    if os.environ.get("QKV_TEST_DEVICES"):          # tests only, e.g. "cpu,cpu" exercises the parallel code paths
        return os.environ["QKV_TEST_DEVICES"].split(",")
    if not torch.cuda.is_available():
        return []
    sel = os.environ.get("QKV_GPUS", "all").strip()
    ids = range(torch.cuda.device_count()) if sel == "all" else [int(i) for i in sel.split(",") if i.strip()]
    return [f"cuda:{i}" for i in ids]

GPUS = _visible_gpus()
MAIN_DEVICE = torch.device("cuda", torch.cuda.current_device()) if DEVICE == "cuda" else torch.device("cpu")
_INIT_LOCK = threading.RLock()        # model construction + seeding happen one at a time
STOP_EVENT = threading.Event()        # set on interruption; long jobs check it and stop

def seed_local(seed: int, device=None):
    """Like seed_everything, but reseeds only the generator of `device` among the GPUs, so that a job
    starting on one GPU does not perturb the random stream (dropout) of a job running on another.
    For the job's own device the seeding is identical to seed_everything."""
    random.seed(seed); np.random.seed(seed % (2**32 - 1)); torch.default_generator.manual_seed(seed)
    dev = torch.device(device) if device is not None else MAIN_DEVICE
    if dev.type == "cuda":
        with torch.cuda.device(dev):
            torch.cuda.manual_seed(seed)

def model_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except StopIteration:
        return MAIN_DEVICE

class Progress:
    """Prints `[label] started`, a first estimate after `first` seconds, then
    `[label] i/total, elapsed, estimated time left` at most every `every` seconds."""
    def __init__(self, label: str, total: int, every: float = 300, start: int = 0, first: float = 60):
        self.label, self.total, self.every, self.start, self.first = label, total, every, start, first
        self.t0 = self.last = time.time()
        self.n_printed = 0
        print(f"  [{label}] started ({total - start} steps)", flush=True)
    def __call__(self, i: int, extra: str = ""):
        now = time.time()
        due = (now - self.t0 >= self.first) if self.n_printed == 0 else (now - self.last >= self.every)
        if not due and i < self.total:
            return
        self.last = now; self.n_printed += 1
        el = now - self.t0
        left = el / max(i - self.start, 1) * (self.total - i)
        print(f"  [{self.label}] {i}/{self.total}  {el / 60:.1f} min elapsed, ~{left / 60:.0f} min left {extra}", flush=True)

def free_gpu_memory():
    """Collect garbage and return cached blocks of every GPU to the driver."""
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()

def release_exception(e: BaseException):
    """Drop the local variables of the frames of a caught exception (models, activations, optimiser state),
    which would otherwise stay on the GPU for as long as the exception object is referenced."""
    import traceback as _tb
    _tb.clear_frames(e.__traceback__)
    free_gpu_memory()

def run_parallel(fn, jobs, label: str = "jobs", devices=None, return_exceptions: bool = False):
    """fn(job, device) for every job, at most one job per device at a time (one worker thread per
    device); results in job order.
    With fewer than two devices this is a plain loop on the main device. Jobs seed themselves with
    seed_local under _INIT_LOCK, so a result does not depend on which GPU ran it or when."""
    devices = list(GPUS if devices is None else devices)
    jobs = list(jobs)
    STOP_EVENT.clear()
    def call(j, dev):
        try:
            return fn(j, dev)
        except (TimeBudgetExceeded, NetworkUnavailable) as e:
            release_exception(e)
            raise                           # never turned into a per-job failure: the cell must stop
        except Exception as e:
            release_exception(e)
            if not return_exceptions:
                raise
            print(f"[{label}] job {j} failed: {repr(e)[:200]}", flush=True)
            return e
    if len(devices) < 2 or len(jobs) < 2:
        return [call(j, MAIN_DEVICE) for j in jobs]
    free = queue.Queue()
    for d in devices:
        free.put(d)
    tls = threading.local()
    def bind():                             # each worker thread owns one device for its whole life
        tls.dev = free.get()
        if str(tls.dev).startswith("cuda"):
            torch.cuda.set_device(tls.dev)
    def work(j):
        return call(j, torch.device(tls.dev))
    t0 = time.time()
    print(f"[parallel] {len(jobs)} {label} on {len(devices)} devices ({', '.join(map(str, devices))})", flush=True)
    ex = ThreadPoolExecutor(len(devices), thread_name_prefix="qkv-job", initializer=bind)
    out, err = [], None
    try:
        futs = [ex.submit(work, j) for j in jobs]
        for f in futs:                      # a failed job does not interrupt the others
            try:
                out.append(f.result())
            except Exception as e:
                err = err or e; out.append(e)
    except BaseException:                   # interruption: running jobs stop at their next check
        STOP_EVENT.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown(wait=True)
    free_gpu_memory()
    print(f"[parallel] {label} done in {time.time() - t0:.0f} s", flush=True)
    if err is not None:
        raise err
    return out

if len(GPUS) > 1:
    print(f"{len(GPUS)} devices for parallel jobs: " + ", ".join(
        f"{g} ({torch.cuda.get_device_name(g)})" if g.startswith("cuda") else g for g in GPUS))
def disk_free_gb(path) -> float:
    return shutil.disk_usage(str(path)).free / 1e9

print(f"SMOKE={SMOKE}  SECTIONS={SECTIONS}  DEVICE={DEVICE}  torch={torch.__version__}  OUT={OUT}")
require_gpu()
print(f"free disk: outputs {disk_free_gb(OUT):.1f} GB ({OUT}), weight cache {disk_free_gb(HF_CACHE):.1f} GB ({HF_CACHE})")
require_network()

### 0.2 Results registry and export helpers
`R.cell(id)` declares the producing cell; `R.set(rid, value, fmt)` stores a value and its LaTeX
rendering. Results persist in `outputs/results.json`, so sections can be run in separate sessions.

In [ ]:
def latex_safe(s: str) -> str:
    """Escape LaTeX-special characters outside $...$ (underscores in model names, bare % and &, #)."""
    parts = s.split("$")
    for i in range(0, len(parts), 2):
        parts[i] = re.sub(r"(?<!\\)([_%&#])", r"\\\1", parts[i])
        parts[i] = re.sub(r"(?<![\w\-\\.])-(?=\.?\d)", "\x00", parts[i])   # numeric minus -> $-$
    return "$".join(parts).replace("\x00", "$-$")

CONTINUE_ON_ERROR = os.environ.get("QKV_CONTINUE_ON_ERROR", "1") == "1"
FAILED: Dict[str, str] = {}

class guard:
    """Context manager wrapped around every experiment cell: with QKV_CONTINUE_ON_ERROR=1 (default) an
    error is reported and recorded, and the notebook continues with the next experiment."""
    def __enter__(self):
        R._cell = None
        return self
    def __exit__(self, et, ev, tb):
        if et is None:
            if getattr(R, "_cell", None):
                mark_cell_done(R._cell)
            free_gpu_memory()
            return False
        if not CONTINUE_ON_ERROR or issubclass(et, KeyboardInterrupt):
            return False
        import traceback
        cell = getattr(R, "_cell", None) or "?"
        FAILED[cell] = f"{et.__name__}: {ev}"
        if not issubclass(et, TimeBudgetExceeded):
            print(f"[error in cell {cell}] {et.__name__}: {ev}")
        if issubclass(et, TimeBudgetExceeded):
            print(f"[time budget] cell {cell} stopped; its finished parts are saved and it is completed in the next "
                  "session")
        elif is_network_error(ev):
            NET_DOWN.set()
            print(f"[network] cell {cell} stopped by a network failure: its results stay pending and no checkpoint "
                  "was recorded as missing. Run the notebook again (finished work is reused).")
        else:
            traceback.print_exception(et, ev, tb, limit=4)
        traceback.clear_frames(tb)          # free what the failed cell held (GPU tensors included)
        free_gpu_memory()
        return True

def mem_status() -> str:
    """Host RAM and GPU memory in use (printed at the start of every cell: if Kaggle restarts the kernel for
    lack of memory, the last header shows which cell was running and how much memory was already used)."""
    parts = []
    try:
        import psutil
        vm = psutil.virtual_memory()
        parts.append(f"RAM {(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f} GB")
    except Exception:
        pass
    if torch.cuda.is_available():
        parts.append("GPU " + ", ".join(f"{torch.cuda.memory_allocated(i) / 1e9:.1f}" for i in range(torch.cuda.device_count())) + " GB")
    return "(" + "; ".join(parts) + ")" if parts else ""

class Results:
    def __init__(self, path: Path):
        self.path = path
        self.store = json.loads(path.read_text()) if path.exists() else {}
        self._cell = None

    def cell(self, cell_id: str):
        self._cell = cell_id
        print(f"--- cell {cell_id} --- {mem_status()}", flush=True)

    def set(self, rid: str, value, fmt: Optional[str] = None, note: str = ""):
        assert re.fullmatch(r"[a-z0-9][a-z0-9.\-]*", rid), f"bad result id {rid}"
        if isinstance(value, (np.floating, np.integer)):
            value = value.item()
        if isinstance(value, str):
            text = latex_safe(value)
        elif value is None or (isinstance(value, float) and not np.isfinite(value)):
            text = r"n/a"
        elif fmt is not None:
            text = latex_safe(fmt.format(value))
        else:
            text = str(value)
        self.store[rid] = dict(value=value if not isinstance(value, float) or np.isfinite(value) else None,
                               text=text, cell=self._cell, note=note, smoke=SMOKE,
                               time=time.strftime("%Y-%m-%d %H:%M:%S"))
        self.path.write_text(json.dumps(self.store, indent=1, default=str))
        print(f"  [result] {rid} = {text[:220]}")

    def get(self, rid, default=None):
        return self.store.get(rid, {}).get("value", default)

    def text(self, rid):
        return self.store[rid]["text"]

R = Results(OUT / "results.json")

def savefig(fig, name: str):
    from matplotlib.ticker import NullFormatter
    for a in fig.axes:          # no minor tick labels on log axes (they collide on short decades)
        if a.get_xscale() == "log":
            a.xaxis.set_minor_formatter(NullFormatter())
        if a.get_yscale() == "log":
            a.yaxis.set_minor_formatter(NullFormatter())
    p = FIG / f"{name}.pdf"
    fig.savefig(p, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("figure ->", p)

def save_table(df: pd.DataFrame, name: str, caption: str = "", label: str = "", col_format=None,
               float_fmt="{:.3f}", escape=False):
    """Write a booktabs tabular (no float environment; the paper provides caption/label)."""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: "--" if pd.isna(v) else latex_safe(float_fmt.format(v)))
        elif df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].map(lambda v: latex_safe(v) if isinstance(v, str) else v)
    df.columns = [latex_safe(str(c)) for c in df.columns]
    tex = df.to_latex(index=False, escape=escape, column_format=col_format)
    (TAB / f"{name}.tex").write_text(tex)
    df.to_csv(TAB / f"{name}.csv", index=False)
    print("table ->", TAB / f"{name}.tex")

def fmt_num(x, nd=3):
    return "n/a" if x is None or not np.isfinite(x) else f"{x:.{nd}f}"

def fmt_ci(m, lo, hi, nd=3):
    return f"{m:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return "n/a"
    if p < 1e-4:
        return r"$p<10^{-4}$"
    return f"$p={p:.3f}$" if p < 0.1 else f"$p={p:.2f}$"

def fmt_sci(x, nd=1):
    if x is None or not np.isfinite(x):
        return "n/a"
    if x == 0:
        return "0"
    e = int(np.floor(np.log10(abs(x)))); m = x / 10**e
    if abs(float(f"{m:.{nd}f}")) >= 10:          # 9.96e-17 must print as 1.0e-16, not 10.0e-17
        e += 1; m = x / 10**e
    return rf"${m:.{nd}f}\times10^{{{e}}}$"

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
                     "legend.fontsize": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42})
PAIR_COLORS = {"QK": "#4C72B0", "KV": "#DD8452", "QV": "#55A868"}
PAIRS = ["QK", "KV", "QV"]

## 1. Geometry library
### 1.1 Model registry
Static metadata only (architecture family, positional encoding, attention type, training
procedure, lineage). Sizes and parameter counts are always read from the checkpoints.
`pe`: `abs` (learned/sinusoidal absolute), `relbias` (T5 relative bias), `rope` (full rotary),
`prope` (partial rotary), `nope`. `procedure`: how the checkpoint was obtained from its `parent`.

In [ ]:
REG: Dict[str, dict] = {}

def reg(mid, arch, family, pe, attn="MHA", procedure="pretrain", parent=None, tags=(), **kw):
    REG[mid] = dict(arch=arch, family=family, pe=pe, attn=attn, procedure=procedure,
                    parent=parent, tags=set(tags), **kw)

# --- the 12 reference checkpoints (those of the preliminary analysis)
reg("google-bert/bert-base-uncased", "bert", "BERT", "abs", tags=["orig", "distill-ref"], smoke_size="base")
reg("distilbert/distilbert-base-uncased", "distilbert", "DistilBERT", "abs", procedure="distill",
    parent="google-bert/bert-base-uncased", tags=["orig", "distilled"])
reg("prajjwal1/bert-tiny", "bert", "BERT", "abs", tags=["orig"], L=2, H=128)
reg("openai-community/gpt2", "gpt2", "GPT-2", "abs", tags=["orig", "surgery", "heads", "act"])
reg("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", "llama", "TinyLlama", "rope", "GQA",
    tags=["orig", "act", "rope"])
reg("microsoft/phi-2", "phi", "Phi", "prope", tags=["orig", "act", "rope"])
reg("Qwen/Qwen2.5-0.5B", "qwen2", "Qwen2.5", "rope", "GQA", tags=["orig", "surgery", "heads", "act", "rope"])
reg("Qwen/Qwen2.5-0.5B-Instruct", "qwen2", "Qwen2.5", "rope", "GQA", procedure="sft+rl",
    parent="Qwen/Qwen2.5-0.5B", tags=["orig"])
reg("Qwen/Qwen2.5-1.5B", "qwen2", "Qwen2.5", "rope", "GQA", tags=["orig"])
reg("Qwen/Qwen2.5-1.5B-Instruct", "qwen2", "Qwen2.5", "rope", "GQA", procedure="sft+rl",
    parent="Qwen/Qwen2.5-1.5B", tags=["orig"])
reg("mistralai/Mistral-7B-v0.1", "mistral", "Mistral", "rope", "GQA", tags=["orig", "large", "rope"])
reg("mistralai/Mistral-7B-Instruct-v0.1", "mistral", "Mistral", "rope", "GQA", procedure="sft",
    parent="mistralai/Mistral-7B-v0.1", tags=["orig", "large"])

# --- B2: the 24 BERT miniatures (Turc et al., 2019): same recipe, capacity varies
MINIS = []
for L_ in (2, 4, 6, 8, 10, 12):
    for H_ in (128, 256, 512, 768):
        mid = f"google/bert_uncased_L-{L_}_H-{H_}_A-{H_ // 64}"
        reg(mid, "bert", "BERT-miniature", "abs", tags=["minis"], L=L_, H=H_)
        MINIS.append(mid)

# --- B2: distilled families
DISTILLED = ["distilbert/distilbert-base-uncased", "huawei-noah/TinyBERT_General_4L_312D",
             "huawei-noah/TinyBERT_General_6L_768D", "microsoft/MiniLM-L12-H384-uncased",
             "nreimers/MiniLM-L6-H384-uncased", "google/mobilebert-uncased"]
reg("huawei-noah/TinyBERT_General_4L_312D", "bert", "TinyBERT", "abs", procedure="distill",
    parent="google-bert/bert-base-uncased", tags=["distilled"], smoke_size="tiny")
reg("huawei-noah/TinyBERT_General_6L_768D", "bert", "TinyBERT", "abs", procedure="distill",
    parent="google-bert/bert-base-uncased", tags=["distilled"], smoke_size="base6")
reg("microsoft/MiniLM-L12-H384-uncased", "bert", "MiniLM", "abs", procedure="distill",
    parent="microsoft/unilm-v2-base", tags=["distilled"], smoke_size="mini12")
reg("nreimers/MiniLM-L6-H384-uncased", "bert", "MiniLM", "abs", procedure="distill",
    parent="microsoft/MiniLM-L12-H384-uncased", tags=["distilled"], smoke_size="mini6")
reg("google/mobilebert-uncased", "mobilebert", "MobileBERT", "abs", procedure="distill",
    parent="IB-BERT-large", tags=["distilled"])

# --- B1: seed variability
MULTIBERTS = [f"google/multiberts-seed_{s}" for s in range(25)]
for s, mid in enumerate(MULTIBERTS):
    reg(mid, "bert", "MultiBERTs", "abs", tags=["multiberts"], seed=s, smoke_size="base")
PYTHIA_SIZES = ["14m", "31m", "70m", "160m", "410m", "1b", "1.4b", "2.8b"]
for sz in PYTHIA_SIZES:
    reg(f"EleutherAI/pythia-{sz}", "neox", "Pythia", "prope", tags=["pythia"], size=sz)
POLYPYTHIA_SIZES = ["70m", "160m", "410m"]
POLYPYTHIAS = {sz: [f"EleutherAI/pythia-{sz}"] + [f"EleutherAI/pythia-{sz}-seed{s}" for s in range(1, 10)]
               for sz in POLYPYTHIA_SIZES}
UNSTABLE_DOCUMENTED = {"410m": [3, 4]}     # PolyPythias runs with loss spikes (van der Wal et al., 2025, Sec. 3)
for sz, mids in POLYPYTHIAS.items():
    for s, mid in enumerate(mids[1:], start=1):
        reg(mid, "neox", "Pythia", "prope", tags=["polypythias"], size=sz, seed=s)
PYTHIA_CHECKPOINT_SIZES = ["70m", "160m", "410m"]
PYTHIA_STEPS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1000, 2000, 4000, 8000, 16000, 32000,
                64000, 100000, 143000]
if SMOKE:
    PYTHIA_STEPS = [0, 512, 143000]

# --- B6: broader coverage
reg("google-t5/t5-base", "t5", "T5", "relbias", tags=["broad"])
reg("facebook/bart-base", "bart", "BART", "abs", tags=["broad"])
reg("allenai/OLMoE-1B-7B-0924", "olmoe", "OLMoE", "rope", tags=["broad", "moe", "large"])
reg("Qwen/Qwen1.5-MoE-A2.7B", "qwen2_moe", "Qwen1.5-MoE", "rope", tags=["broad", "moe", "large"])
reg("meta-llama/Llama-3.1-8B", "llama", "Llama-3", "rope", "GQA", tags=["broad", "large", "rope"])
reg("Qwen/Qwen2.5-7B", "qwen2", "Qwen2.5", "rope", "GQA", tags=["broad", "large", "rope"])

# --- B7: fine-tuning pairs (child -> parent); procedure in {sft, sft+rl, sft+dpo, rlhf, rlvr, domain, task}
reg("meta-llama/Llama-3.2-1B", "llama", "Llama-3", "rope", "GQA", tags=["ft-base", "heads", "act", "rope"])
reg("meta-llama/Llama-3.2-1B-Instruct", "llama", "Llama-3", "rope", "GQA", procedure="sft+rl",
    parent="meta-llama/Llama-3.2-1B")
reg("HuggingFaceH4/mistral-7b-sft-beta", "mistral", "Mistral", "rope", "GQA", procedure="sft",
    parent="mistralai/Mistral-7B-v0.1", tags=["large"])
reg("HuggingFaceH4/zephyr-7b-beta", "mistral", "Mistral", "rope", "GQA", procedure="sft+dpo",
    parent="HuggingFaceH4/mistral-7b-sft-beta", tags=["large"])
reg("meta-llama/Llama-2-7b-hf", "llama", "Llama-2", "rope", "MHA", tags=["large"])
reg("meta-llama/Llama-2-7b-chat-hf", "llama", "Llama-2", "rope", "MHA", procedure="sft+rlhf",
    parent="meta-llama/Llama-2-7b-hf", tags=["large"])
reg("codellama/CodeLlama-7b-hf", "llama", "Llama-2", "rope", "MHA", procedure="domain",
    parent="meta-llama/Llama-2-7b-hf", tags=["large"])
reg("allenai/OLMo-2-0425-1B", "olmo2", "OLMo-2", "rope", tags=["ft-base"])
reg("allenai/OLMo-2-0425-1B-SFT", "olmo2", "OLMo-2", "rope", procedure="sft", parent="allenai/OLMo-2-0425-1B")
reg("allenai/OLMo-2-0425-1B-DPO", "olmo2", "OLMo-2", "rope", procedure="dpo", parent="allenai/OLMo-2-0425-1B-SFT")
reg("allenai/OLMo-2-0425-1B-Instruct", "olmo2", "OLMo-2", "rope", procedure="rlvr",
    parent="allenai/OLMo-2-0425-1B-DPO")
reg("google-bert/bert-base-cased", "bert", "BERT", "abs", smoke_size="base")
reg("dmis-lab/biobert-base-cased-v1.2", "bert", "BERT", "abs", procedure="domain",
    parent="google-bert/bert-base-cased", smoke_size="base")
reg("textattack/bert-base-uncased-SST-2", "bert", "BERT", "abs", procedure="task",
    parent="google-bert/bert-base-uncased", smoke_size="base")
reg("textattack/bert-base-uncased-MNLI", "bert", "BERT", "abs", procedure="task",
    parent="google-bert/bert-base-uncased", smoke_size="base")
# independent same-architecture checkpoint (negative control for lineage detection)
reg("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract", "bert", "BERT", "abs",
    tags=["negative"], smoke_size="base")

FT_PAIRS = [(c, REG[c]["parent"]) for c in REG
            if REG[c]["procedure"] in {"sft", "sft+rl", "sft+dpo", "dpo", "rlhf", "sft+rlhf", "rlvr", "domain", "task"}
            and REG[c]["parent"] in REG]
ORIG12 = [m for m in REG if "orig" in REG[m]["tags"]]
BROAD = [m for m in REG if "broad" in REG[m]["tags"]]
print(len(REG), "registered checkpoints;", len(FT_PAIRS), "fine-tuning pairs;", len(MINIS), "miniatures")

### 1.2 Weight access with bounded disk use (streamed safetensors, `.bin` fallback, smoke models)
`WeightSource` never instantiates the network. For safetensors checkpoints it reads the file header
and then fetches **only the tensors that are requested** (query/key/value projections and norm gains)
with HTTP range requests, so a 7–8B checkpoint costs ~1–2 GB of transfer and no disk. Large tensors are
fetched as parallel byte-range chunks (`QKV_HTTP_THREADS`, default 8) and the tensors of layer $\ell+1$
are prefetched in the background while layer $\ell$ is analysed. If the server
does not honour range requests, the shards are downloaded one at a time (at most two open per
checkpoint) and deleted when no longer needed. `.bin` checkpoints are downloaded and deleted after use.
Full-model loads (`load_model`) free disk space beforehand and evict the files afterwards.
Eviction is controlled by `QKV_EVICT` (`auto`: evict when less than 100 GB are free; `1`: always; `0`: never).

In [ ]:
import struct
import requests
import transformers
from transformers import AutoConfig
from huggingface_hub import HfApi, hf_hub_download, hf_hub_url
from huggingface_hub.utils import build_hf_headers
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()            # keeps the notebook small; downloads are logged by the cells
except Exception:
    pass
# Library messages that are expected and harmless here are silenced (errors are still shown): the
# BertForMaskedLM "load report" (pooler / next-sentence weights unused by the MLM model) and the
# "sequence longer than 512" notice of the tokenizer (corpora are tokenised as one stream and cut
# into short windows afterwards).
transformers.logging.set_verbosity_error()
try:
    import datasets as _ds
    _ds.disable_progress_bars(); _ds.logging.set_verbosity_error()
except Exception:
    pass

QKV_EVICT = os.environ.get("QKV_EVICT", "auto")
ACTIVE_REPOS = set()

def evict_enabled() -> bool:
    if QKV_EVICT == "auto":
        return disk_free_gb(HF_CACHE) < 100
    return QKV_EVICT == "1"

def _evict_path(p):
    """Delete a cached file (the blob behind the snapshot symlink, then the symlink)."""
    try:
        real = os.path.realpath(p)
        if os.path.exists(real):
            os.remove(real)
        if os.path.lexists(p):
            os.remove(p)
    except OSError:
        pass

def evict_repo(repo_id: str):
    """Remove every cached revision of a repository from HF_CACHE."""
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir(HF_CACHE)
        hashes = [r.commit_hash for repo in info.repos if repo.repo_id == repo_id for r in repo.revisions]
        if hashes:
            info.delete_revisions(*hashes).execute()
    except Exception as e:
        print("[evict]", repo_id, repr(e)[:120])

def ensure_space(need_bytes: float, margin: float = 2e9):
    """Free disk space in HF_CACHE (least recently used repositories first) before a download."""
    if disk_free_gb(HF_CACHE) * 1e9 >= need_bytes + margin:
        return
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir(HF_CACHE)
        repos = sorted([r for r in info.repos if r.repo_id not in ACTIVE_REPOS], key=lambda r: r.last_accessed)
        for repo in repos:
            info.delete_revisions(*[r.commit_hash for r in repo.revisions]).execute()
            print(f"[disk] evicted {repo.repo_id} ({repo.size_on_disk / 1e9:.1f} GB)")
            if disk_free_gb(HF_CACHE) * 1e9 >= need_bytes + margin:
                return
    except Exception as e:
        print("[disk] eviction failed:", repr(e)[:150])
    if disk_free_gb(HF_CACHE) * 1e9 < need_bytes + margin:
        raise OSError(f"not enough disk in {HF_CACHE}: need {need_bytes / 1e9:.1f} GB, "
                      f"free {disk_free_gb(HF_CACHE):.1f} GB (set QKV_HF_CACHE to a larger volume)")

def repo_files(repo: str, revision: Optional[str] = None) -> Dict[str, int]:
    """File name -> size for a Hub repository at a revision (cached on disk)."""
    cp = CACHE / "hub_meta" / (hashlib.sha1(f"{repo}|{revision}".encode()).hexdigest()[:16] + ".json")
    if cp.exists():
        return json.loads(cp.read_text())
    info = HfApi().model_info(repo, revision=revision, files_metadata=True)
    out = {s.rfilename: int(s.size or 0) for s in info.siblings}
    cp.parent.mkdir(parents=True, exist_ok=True)
    cp.write_text(json.dumps(out))
    return out

class RangeUnsupported(Exception):
    pass

import threading
from concurrent.futures import ThreadPoolExecutor
HTTP_THREADS = int(os.environ.get("QKV_HTTP_THREADS", "8"))      # parallel range requests per tensor
HTTP_CHUNK = 16 * 2 ** 20                                          # bytes per range request
_TLS = threading.local()
_CHUNK_POOL = ThreadPoolExecutor(max(1, HTTP_THREADS), thread_name_prefix="qkv-range")
_PREFETCH_POOL = ThreadPoolExecutor(2, thread_name_prefix="qkv-prefetch")
NET = {"bytes": 0, "requests": 0}                                   # transfer counter (progress messages)
_NET_LOCK = threading.Lock()

def _session():
    s_ = getattr(_TLS, "session", None)
    if s_ is None:
        s_ = _TLS.session = requests.Session()
    return s_

def _range_get_one(url: str, start: int, end: int, tries: int = 6) -> bytes:
    headers = dict(build_hf_headers())
    headers["Range"] = f"bytes={start}-{end}"
    last = None
    for i in range(tries):
        try:
            r = _session().get(url, headers=headers, stream=True, timeout=180, allow_redirects=True)
            if r.status_code == 206:
                data = r.content
                if len(data) == end - start + 1:
                    with _NET_LOCK:
                        NET["bytes"] += len(data); NET["requests"] += 1
                    return data
                last = IOError(f"short read {len(data)} / {end - start + 1}")
            else:
                r.close()
                if r.status_code == 200:
                    raise RangeUnsupported(url)
                if r.status_code in (401, 403, 404):
                    raise PermissionError(f"HTTP {r.status_code} for {url} (gated model? set HF_TOKEN)")
                last = IOError(f"HTTP {r.status_code}")
                if r.status_code == 429:
                    print(f"[http] rate-limited by the Hub (429); retry {i + 1}/{tries}", flush=True)
        except (RangeUnsupported, PermissionError):
            raise
        except Exception as e:
            last = e
        time.sleep(min(60, 2 ** i))
    raise IOError(f"range request failed for {url}: {last!r}") from (last if isinstance(last, BaseException) else None)

def range_get(url: str, start: int, end: int, tries: int = 6) -> bytes:
    """Bytes [start, end] (inclusive) of a remote file; raises RangeUnsupported if the server ignores Range.
    Spans larger than HTTP_CHUNK are fetched as parallel chunks."""
    if end - start + 1 <= HTTP_CHUNK or HTTP_THREADS <= 1:
        return _range_get_one(url, start, end, tries)
    bounds = [(a, min(a + HTTP_CHUNK, end + 1) - 1) for a in range(start, end + 1, HTTP_CHUNK)]
    futs = [_CHUNK_POOL.submit(_range_get_one, url, a, b, tries) for a, b in bounds]
    return b"".join(f.result() for f in futs)

_LAYER_RX = re.compile(r"(^|\.)(\d+)\.")

def next_layer_key(k: str) -> Optional[str]:
    """Same tensor one block further (first integer index incremented), e.g. layers.3.q -> layers.4.q."""
    m = _LAYER_RX.search(k)
    if not m:
        return None
    return k[:m.start(2)] + str(int(m.group(2)) + 1) + k[m.end(2):]

ST_DTYPES = {"F64": torch.float64, "F32": torch.float32, "F16": torch.float16, "BF16": torch.bfloat16,
             "I64": torch.int64, "I32": torch.int32, "I16": torch.int16, "I8": torch.int8, "U8": torch.uint8,
             "BOOL": torch.bool}

def st_header(repo: str, revision: Optional[str], filename: str, reader=None) -> dict:
    """Parsed safetensors header of a remote file (two small range requests; cached on disk)."""
    cp = CACHE / "st_headers" / (hashlib.sha1(f"{repo}|{revision}|{filename}".encode()).hexdigest()[:16] + ".json")
    if cp.exists():
        return json.loads(cp.read_text())
    reader = reader or (lambda s_, e_: range_get(hf_hub_url(repo, filename, revision=revision), s_, e_))
    n = struct.unpack("<Q", reader(0, 7))[0]
    hdr = json.loads(reader(8, 8 + n - 1))
    obj = {"n": n, "header": hdr}
    cp.parent.mkdir(parents=True, exist_ok=True)
    cp.write_text(json.dumps(obj))
    return obj

def st_tensor(reader, n: int, meta: dict) -> torch.Tensor:
    s_, e_ = meta["data_offsets"]
    dt = ST_DTYPES[meta["dtype"]]
    if e_ == s_:
        return torch.empty(meta["shape"], dtype=dt)
    data = reader(8 + n + s_, 8 + n + e_ - 1)
    return torch.frombuffer(bytearray(data), dtype=dt).reshape(meta["shape"]).clone()

SMOKE_VOCAB = 256

def _smoke_dims(mid: str):
    """Tiny but architecture-faithful configurations for the smoke test."""
    info = REG.get(mid, {"arch": "bert", "tags": set()})
    a = info["arch"]
    size = info.get("smoke_size", info.get("size", "small"))
    if a == "bert":
        if "L" in info:   # miniature grid: same depth, width / 8, d_head 8
            return dict(hidden_size=info["H"] // 8, num_attention_heads=max(1, info["H"] // 64),
                        num_hidden_layers=info["L"], intermediate_size=info["H"] // 2)
        table = {"base": (96, 12, 4), "base6": (96, 12, 3), "tiny": (40, 4, 2), "mini12": (48, 6, 4),
                 "mini6": (48, 6, 2), "small": (32, 4, 2)}
        h, n, L = table.get(size, (32, 4, 2))
        return dict(hidden_size=h, num_attention_heads=n, num_hidden_layers=L, intermediate_size=2 * h)
    if a == "neox":
        table = {"14m": (32, 4, 2), "31m": (40, 4, 2), "70m": (48, 4, 3), "160m": (64, 4, 3),
                 "410m": (96, 6, 3), "1b": (128, 4, 3), "1.4b": (128, 8, 3), "2.8b": (160, 8, 3)}
        h, n, L = table.get(size, (64, 4, 3))
        return dict(hidden_size=h, num_attention_heads=n, num_hidden_layers=L, intermediate_size=2 * h)
    return {}

def smoke_config(mid: str):
    info = REG.get(mid, {"arch": "bert", "attn": "MHA"})
    a, gqa = info["arch"], info.get("attn") == "GQA"
    T = transformers
    if a == "bert":
        c = T.BertConfig(vocab_size=SMOKE_VOCAB, max_position_embeddings=512, **_smoke_dims(mid))
    elif a == "distilbert":
        c = T.DistilBertConfig(vocab_size=SMOKE_VOCAB, dim=48, n_heads=6, n_layers=3, hidden_dim=96)
    elif a == "mobilebert":
        c = T.MobileBertConfig(vocab_size=SMOKE_VOCAB, hidden_size=64, embedding_size=32, true_hidden_size=32,
                               intra_bottleneck_size=32, num_hidden_layers=3, num_attention_heads=4,
                               intermediate_size=32, num_feedforward_networks=2)
    elif a == "gpt2":
        c = T.GPT2Config(vocab_size=SMOKE_VOCAB, n_embd=48, n_layer=3, n_head=4, n_positions=512,
                         bos_token_id=0, eos_token_id=0)
    elif a == "neox":
        c = T.GPTNeoXConfig(vocab_size=SMOKE_VOCAB, rotary_pct=0.25, max_position_embeddings=512,
                            bos_token_id=0, eos_token_id=0, **_smoke_dims(mid))
    elif a in ("llama", "mistral", "qwen2"):
        cls = {"llama": T.LlamaConfig, "mistral": T.MistralConfig, "qwen2": T.Qwen2Config}[a]
        c = cls(vocab_size=SMOKE_VOCAB, hidden_size=64, num_hidden_layers=3, num_attention_heads=8,
                num_key_value_heads=2 if gqa else 8, intermediate_size=128, max_position_embeddings=512,
                bos_token_id=1, eos_token_id=2)
    elif a == "phi":
        c = T.PhiConfig(vocab_size=SMOKE_VOCAB, hidden_size=64, num_hidden_layers=3, num_attention_heads=4,
                        intermediate_size=128, partial_rotary_factor=0.4, max_position_embeddings=512)
    elif a == "olmoe":
        c = T.OlmoeConfig(vocab_size=SMOKE_VOCAB, hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                          num_key_value_heads=4, intermediate_size=32, num_experts=4, num_experts_per_tok=2,
                          max_position_embeddings=512, pad_token_id=0, bos_token_id=1, eos_token_id=2)
    elif a == "qwen2_moe":
        c = T.Qwen2MoeConfig(vocab_size=SMOKE_VOCAB, hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                             num_key_value_heads=4, intermediate_size=64, moe_intermediate_size=16,
                             shared_expert_intermediate_size=32, num_experts=4, num_experts_per_tok=2,
                             max_position_embeddings=512)
    elif a == "olmo2":
        c = T.Olmo2Config(vocab_size=SMOKE_VOCAB, hidden_size=64, num_hidden_layers=3, num_attention_heads=4,
                          num_key_value_heads=4, intermediate_size=128, max_position_embeddings=512,
                          pad_token_id=0, bos_token_id=1, eos_token_id=2)
    elif a == "t5":
        c = T.T5Config(vocab_size=SMOKE_VOCAB, d_model=48, d_kv=12, d_ff=96, num_layers=2, num_heads=4,
                       decoder_start_token_id=0)
    elif a == "bart":
        c = T.BartConfig(vocab_size=SMOKE_VOCAB, d_model=48, encoder_layers=2, decoder_layers=2,
                         encoder_attention_heads=4, decoder_attention_heads=4, encoder_ffn_dim=96,
                         decoder_ffn_dim=96, max_position_embeddings=512)
    else:
        raise ValueError(a)
    c._attn_implementation = "eager"
    return c

_TASK_CLASSES = {
    "causal": "AutoModelForCausalLM", "mlm": "AutoModelForMaskedLM", "seq2seq": "AutoModelForSeq2SeqLM",
    "cls": "AutoModelForSequenceClassification", "base": "AutoModel"}

def default_task(mid):
    a = REG.get(mid, {}).get("arch", "bert")
    return {"bert": "mlm", "distilbert": "mlm", "mobilebert": "mlm", "t5": "seq2seq", "bart": "seq2seq"}.get(a, "causal")

def smoke_model(mid: str, revision=None, task=None, num_labels=2):
    """Deterministic random model; a child checkpoint is its parent plus a small perturbation,
    and a Pythia checkpoint at step s interpolates from random init (step 0) to the final weights."""
    task = task or default_task(mid)
    cfg = smoke_config(mid)
    if task == "cls":
        cfg.num_labels = num_labels
    cls = getattr(transformers, _TASK_CLASSES[task])
    torch.manual_seed(stable_hash("smoke", mid if REG.get(mid, {}).get("parent") not in REG else REG[mid]["parent"]))
    model = cls.from_config(cfg)
    if REG.get(mid, {}).get("parent") in REG:      # derived checkpoint: parent + 5% noise
        g = torch.Generator().manual_seed(stable_hash("child", mid))
        with torch.no_grad():
            for p in model.parameters():
                p.add_(0.05 * p.std().nan_to_num(0) * torch.randn(p.shape, generator=g))
    if revision is not None and revision.startswith("step"):
        s = int(revision[4:]); frac = 0.0 if s == 0 else min(1.0, math.log1p(s) / math.log1p(143000))
        g = torch.Generator().manual_seed(stable_hash("init", mid))
        with torch.no_grad():
            for p in model.parameters():
                if p.ndim == 2:
                    p.mul_(frac).add_((1 - frac) * 0.02 * torch.randn(p.shape, generator=g))
    return model.eval()

class WeightSource:
    """Tensor access to a checkpoint: in-memory state dict, smoke model, local files, or a Hub repository
    (streamed safetensors tensors, lazily downloaded shards, or downloaded .bin files)."""
    def __init__(self, model_id: str, revision: Optional[str] = None, state_dict=None, config: dict = None,
                 local_dir: Optional[str] = None):
        self.model_id, self.revision = model_id, revision
        self._sd, self._loc, self._remote = None, {}, {}
        self._key2file, self._open, self._downloaded, self.mode = {}, {}, [], "dict"
        self._pf = {}                                  # key -> Future (background prefetch, remote mode)
        if state_dict is not None:
            # references only (tensors may live on the GPU): get() copies one tensor at a time to the CPU,
            # so a model's full state dict is never duplicated in host memory
            self._sd = {k: v.detach() for k, v in state_dict.items()}
            self.config = config if isinstance(config, dict) else config.to_dict()
        elif SMOKE and local_dir is None:
            m = smoke_model(model_id, revision)
            self._sd = {k: v.detach().cpu() for k, v in m.state_dict().items()}
            self.config = m.config.to_dict()
        elif local_dir is not None:
            self._init_local(Path(local_dir))
        else:
            self._init_hub()
        if not self.config.get("model_type"):      # old configs (e.g. Google miniatures) lack model_type
            arch = REG.get(model_id, {}).get("arch", "bert")
            self.config["model_type"] = {"neox": "gpt_neox"}.get(arch, arch)
        self.model_type = self.config["model_type"]

    # ---------- initialisation
    def _init_local(self, d: Path):
        from safetensors import safe_open
        idx = list(d.glob("*.safetensors.index.json"))
        files = (sorted({d / f for f in json.loads(idx[0].read_text())["weight_map"].values()}) if idx else
                 [f for f in sorted(d.glob("*.safetensors")) if not f.name.startswith("consolidated")])
        if files:
            for f in files:
                h = safe_open(str(f), framework="pt")
                for k in h.keys():
                    self._loc[k] = h
            self.mode = "local"
        else:
            self._sd = {}
            for b in sorted(d.glob("pytorch_model*.bin")):
                self._sd.update(torch.load(b, map_location="cpu", weights_only=True))
        self.config = json.loads((d / "config.json").read_text())

    def _download(self, filename: str, size: int) -> str:
        ensure_space(size)
        p = hf_hub_download(self.model_id, filename, revision=self.revision, cache_dir=str(HF_CACHE))
        self._downloaded.append(p)
        return p

    def _init_hub(self):
        files = repo_files(self.model_id, self.revision)
        cfg_p = hf_hub_download(self.model_id, "config.json", revision=self.revision, cache_dir=str(HF_CACHE))
        self.config = json.loads(Path(cfg_p).read_text())
        top = {f: sz for f, sz in files.items() if "/" not in f}
        if "model.safetensors.index.json" in top:
            wm = json.loads(Path(hf_hub_download(self.model_id, "model.safetensors.index.json", revision=self.revision,
                                                 cache_dir=str(HF_CACHE))).read_text())["weight_map"]
            st_files = sorted(set(wm.values()))
        else:
            wm = None
            st_files = [f for f in top if f.endswith(".safetensors") and f.startswith("model")]
        self._sizes = top
        if st_files:
            try:                                  # stream tensors with HTTP range requests
                for f in st_files:
                    h = st_header(self.model_id, self.revision, f)
                    url = hf_hub_url(self.model_id, f, revision=self.revision)
                    for k, meta in h["header"].items():
                        if k != "__metadata__":
                            self._remote[k] = (url, h["n"], meta)
                self.mode = "remote"
                return
            except RangeUnsupported:
                print(f"[weights] range requests not supported for {self.model_id}; downloading shards lazily")
            if wm is not None:                    # lazily downloaded shards, at most two open at a time
                self._key2file = dict(wm)
                self.mode = "lazyfile"
            else:
                from safetensors import safe_open
                for f in st_files:
                    h = safe_open(self._download(f, top[f]), framework="pt")
                    for k in h.keys():
                        self._loc[k] = h
                self.mode = "local"
            return
        bins = sorted(f for f in top if re.fullmatch(r"pytorch_model.*\.bin", f))
        if not bins:
            raise FileNotFoundError(f"no PyTorch weights for {self.model_id}@{self.revision}")
        self._sd = {}
        for b in bins:
            p = self._download(b, top[b])
            try:
                sd = torch.load(p, map_location="cpu", weights_only=True, mmap=True)
            except Exception:
                sd = torch.load(p, map_location="cpu", weights_only=True)
            self._sd.update({k: v for k, v in sd.items()})
        self.mode = "bin"

    def _lazy_handle(self, f: str):
        from safetensors import safe_open
        if f in self._open:
            h = self._open.pop(f); self._open[f] = h          # mark as most recently used
            return h
        if len(self._open) >= 2:                             # close (and evict) the least recently used shard
            old, _ = next(iter(self._open.items()))
            del self._open[old]
            gc.collect()
            if evict_enabled():
                for p in [p for p in self._downloaded if p.endswith(old)]:
                    _evict_path(p); self._downloaded.remove(p)
        h = safe_open(self._download(f, self._sizes.get(f, 0)), framework="pt")
        self._open[f] = h
        return h

    def close(self):
        """Release file handles and delete downloaded weight files when eviction is enabled."""
        for f_ in self._pf.values():
            f_.cancel()
        self._pf.clear()
        self._open.clear(); self._loc.clear()
        if self.mode == "bin":
            self._sd = None
        gc.collect()
        if evict_enabled():
            for p in self._downloaded:
                _evict_path(p)
            self._downloaded = []

    # ---------- access
    def keys(self):
        if self._sd is not None:
            return list(self._sd.keys())
        if self.mode == "remote":
            return list(self._remote.keys())
        if self.mode == "lazyfile":
            return list(self._key2file.keys())
        return list(self._loc.keys())

    def has(self, k):
        if self._sd is not None:
            return k in self._sd
        if self.mode == "remote":
            return k in self._remote
        if self.mode == "lazyfile":
            return k in self._key2file
        return k in self._loc

    def shape(self, k):
        if self._sd is not None:
            return tuple(self._sd[k].shape)
        if self.mode == "remote":
            return tuple(self._remote[k][2]["shape"])
        if self.mode == "lazyfile":
            f = self._key2file[k]
            if f in self._open:
                return tuple(self._open[f].get_slice(k).get_shape())
            return tuple(self._lazy_handle(f).get_slice(k).get_shape())
        return tuple(self._loc[k].get_slice(k).get_shape())

    def get(self, k) -> torch.Tensor:
        if not self.has(k):
            alt = k.replace(".weight", ".gamma").replace(".bias", ".beta")
            if self.has(alt):
                k = alt
            else:
                raise KeyError(f"{k} not in {self.model_id}")
        if self._sd is not None:
            t = self._sd[k]
        elif self.mode == "remote":
            fut = self._pf.pop(k, None)
            try:
                t = fut.result() if fut is not None else self._fetch_remote(k)
            except (RangeUnsupported, PermissionError):
                raise
            except Exception:
                t = self._fetch_remote(k)       # a failed prefetch is retried in the foreground
            nk = next_layer_key(k)
            if nk is not None and nk in self._remote and nk not in self._pf and len(self._pf) < 16:
                self._pf[nk] = _PREFETCH_POOL.submit(self._fetch_remote, nk)
        elif self.mode == "lazyfile":
            t = self._lazy_handle(self._key2file[k]).get_tensor(k)
        else:
            t = self._loc[k].get_tensor(k)
        # always a private float64 CPU copy (state-dict sources reference live parameters)
        return t.detach().to(device="cpu", dtype=torch.float64, copy=self._sd is not None)

    def _fetch_remote(self, k) -> torch.Tensor:
        url, n, meta = self._remote[k]
        return st_tensor(lambda s_, e_: range_get(url, s_, e_), n, meta)

    def get_opt(self, k):
        try:
            return self.get(k)
        except KeyError:
            return None

    def prefix(self, pattern: str) -> str:
        rx = re.compile(pattern)
        for k in self.keys():
            m = rx.match(k)
            if m:
                return m.group(1)
        raise KeyError(f"pattern {pattern} not found in {self.model_id}")

    def param_counts(self):
        emb_rx = re.compile(r"(embed|wte|wpe|word_embeddings|position_embeddings|token_type_embeddings|"
                            r"shared\.weight|embed_in|embed_out|lm_head|relative_attention_bias)")
        head_rx = re.compile(r"^(cls\.|classifier|lm_head|qa_outputs|vocab_|embed_out|final_logits_bias|pre_classifier)")
        total = nonemb = 0
        if self.mode == "lazyfile":          # avoid downloading every shard: estimate from file sizes
            bpp = {"float32": 4, "float16": 2, "bfloat16": 2}.get(str(self.config.get("torch_dtype", "float32")), 4)
            return int(sum(self._sizes.get(f, 0) for f in set(self._key2file.values())) / bpp), np.nan
        for k in self.keys():
            n = int(np.prod(self.shape(k)))
            if head_rx.search(k):
                continue
            total += n
            if not emb_rx.search(k):
                nonemb += n
        return total, nonemb

def cfg_get(cfg, *names, default=None):
    for n in names:
        if n in cfg and cfg[n] is not None:
            return cfg[n]
    return default

GATED_SECTIONS = {"B3", "B5", "B6", "B7"}      # sections that analyse the gated Llama checkpoints

def check_hub_access():
    """Before any long computation, config.json of every registered checkpoint is requested from the Hub (in
    parallel, about 20 s). Stops at once if the Hub is unreachable, or if gated checkpoints used by the requested
    sections are not accessible with HF_TOKEN (unless ALLOW_MISSING_GATED). Absent repositories are listed."""
    if not _needs_network():
        return
    from huggingface_hub.errors import (GatedRepoError, RepositoryNotFoundError, RevisionNotFoundError,
                                        EntryNotFoundError)
    def probe(m):
        try:
            hub_retry(lambda: hf_hub_download(m, "config.json", cache_dir=str(HF_CACHE)), f"config of {m}",
                      waits=(10, 30))
            return m, "ok"
        except NetworkUnavailable:
            return m, "network"
        except GatedRepoError:
            return m, "gated"
        except (RepositoryNotFoundError, RevisionNotFoundError):
            return m, "absent"
        except EntryNotFoundError:
            return m, "ok"                      # no config.json at the top level: checked when loaded
        except Exception as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            return m, "gated" if code in (401, 403) else f"error {type(e).__name__}: {str(e)[:100]}"
    t0 = time.time()
    with ThreadPoolExecutor(8, thread_name_prefix="qkv-access") as ex:
        res = dict(ex.map(probe, sorted(REG)))
    by = collections.defaultdict(list)
    for m, st_ in res.items():
        by[st_ if st_ in ("ok", "network", "gated", "absent") else "error"].append(m)
    print(f"Hub access check: {len(by['ok'])}/{len(res)} checkpoints available ({time.time() - t0:.0f} s)")
    if by["network"]:
        raise RuntimeError(f"The Hugging Face Hub stopped answering during the access check ({len(by['network'])} "
                           "checkpoints). On Kaggle: Settings -> Internet -> On, then run again.")
    for k, label in (("absent", "not found on the Hub"), ("error", "not checked")):
        if by[k]:
            print(f"  {label} (skipped and listed as missing if used): " + ", ".join(by[k]))
    if by["gated"]:
        wanted = sorted(GATED_SECTIONS if "ALL" in SECTIONS else GATED_SECTIONS & set(SECTIONS))
        msg = ("No access to the gated checkpoints " + ", ".join(by["gated"]) + ". " +
               ("HF_TOKEN is not set: Add-ons -> Secrets, tick HF_TOKEN for this notebook"
                if not os.environ.get("HF_TOKEN") else
                "HF_TOKEN is set, but its account has no access: request access on each model page on "
                "huggingface.co with the same account and wait for the approval e-mail") +
               ", then run again. To run without them (they are then listed as missing), set "
               "ALLOW_MISSING_GATED = True in the first cell.")
        if wanted and os.environ.get("QKV_ALLOW_MISSING_GATED", "0") != "1":
            raise RuntimeError(msg + f" (Sections {', '.join(wanted)} use them.)")
        print("  [warning] " + msg)

check_hub_access()

### 1.3 Architecture adapters
Every adapter yields `AttnBlock`s holding per-head read matrices in the paper's convention
$W\in\mathbb{R}^{d_{\text{model}}\times d_{\text{head}}}$ (Q heads, K/V heads, the GQA map
$q\mapsto g(q)$), and the gain vector of the normalisation that
precedes each read (`gq`, `gk`, `gv`). Folding the gain replaces
$W$ by $\operatorname{diag}(\gamma)W$; it is the only parametrisation in which the column spaces are
invariants of the network function. `space_*` names the residual stream a matrix reads from
(encoder vs decoder stream for cross-attention). For MobileBERT, Q and K read a 128-d bottleneck
that is itself a linear image of the 512-d hidden stream: the Q–K pair is analysed in the
bottleneck (`pair_override`), Q–V and K–V in the hidden space after composing with the bottleneck map.
The write matrices $W_O$ are not used by any analysis; they are fetched only with `QKV_FETCH_WO=1`
(this saves about 40% of the transfer for decoder checkpoints).

In [ ]:
@dataclass
class AttnBlock:
    layer: int
    kind: str                      # self | enc-self | dec-self | cross
    Wq: torch.Tensor               # (n_q, d, dh)
    Wk: torch.Tensor               # (n_kv, d, dh)
    Wv: torch.Tensor               # (n_kv, d, dh)
    Wo: Optional[torch.Tensor]     # (n_q, dh, d) or None
    gq: Optional[torch.Tensor]
    gk: Optional[torch.Tensor]
    gv: Optional[torch.Tensor]
    kv_of_q: np.ndarray
    space_q: str = "res"
    space_k: str = "res"
    space_v: str = "res"
    rope: Optional[dict] = None    # {"inv_freq": tensor(rot/2), "rot_dim": int}
    pair_override: dict = field(default_factory=dict)   # pair -> (A (n_a,d',dh), B (n_b,d',dh), map)
    keys: dict = field(default_factory=dict)            # checkpoint keys of q/k/v (for surgery write-back)

    @property
    def n_q(self): return self.Wq.shape[0]
    @property
    def n_kv(self): return self.Wk.shape[0]
    @property
    def d(self): return self.Wq.shape[1]
    @property
    def dh(self): return self.Wq.shape[2]

    def folded(self, which: str, fold: bool = True):
        W, g = {"q": (self.Wq, self.gq), "k": (self.Wk, self.gk), "v": (self.Wv, self.gv)}[which]
        if fold and g is not None:
            return g.view(1, -1, 1) * W
        return W

FETCH_WO = os.environ.get("QKV_FETCH_WO", "0") == "1"

def _wo(fetch):
    """Per-head write matrices, fetched only on request (unused by the geometry)."""
    return fetch() if FETCH_WO else None

def lin_heads(W, n, dh):
    """nn.Linear weight (n*dh, d) -> per-head read matrices (n, d, dh)."""
    return W.reshape(n, dh, W.shape[1]).transpose(1, 2).contiguous()

def lin_out_heads(W, n, dh):
    """nn.Linear output weight (d, n*dh) -> per-head write matrices (n, dh, d)."""
    return W.reshape(W.shape[0], n, dh).permute(1, 2, 0).contiguous()

def rope_info(cfg: dict, dh: int, arch: str):
    """Rotary dimensions and inverse frequencies, taken from the HF rotary module when possible
    (handles llama3/yarn scaling), otherwise from the default formula."""
    rp = cfg.get("rope_parameters") or {}
    frac = cfg_get(cfg, "partial_rotary_factor", "rotary_pct", default=rp.get("partial_rotary_factor", 1.0))
    rot = int(dh * frac)
    rot -= rot % 2
    inv = None
    try:
        mt = cfg["model_type"]
        conf = AutoConfig.for_model(mt, **{k: v for k, v in cfg.items() if k not in ("model_type", "transformers_version")})
        mod = __import__(f"transformers.models.{mt}.modeling_{mt}", fromlist=["*"])
        cls = [getattr(mod, n) for n in dir(mod) if n.endswith("RotaryEmbedding") and not n.startswith("_")][0]
        inv = cls(conf).inv_freq.detach().to(torch.float64).cpu()
        if inv.numel() != rot // 2:
            inv = None
    except Exception as e:  # noqa
        inv = None
    if inv is None:
        base = cfg_get(cfg, "rope_theta", "rotary_emb_base", default=rp.get("rope_theta", 10000.0))
        inv = 1.0 / (base ** (torch.arange(0, rot, 2, dtype=torch.float64) / rot))
    return {"inv_freq": inv, "rot_dim": rot}

def _blocks_bert(ws):
    c = ws.config; d = c["hidden_size"]; n = c["num_attention_heads"]; dh = d // n; L = c["num_hidden_layers"]
    p = ws.prefix(r"^(.*?)encoder\.layer\.0\.attention\.self\.query\.(weight|kernel)$")
    for i in range(L):
        b = f"{p}encoder.layer.{i}."
        g = ws.get_opt(f"{p}embeddings.LayerNorm.weight" if i == 0 else f"{p}encoder.layer.{i-1}.output.LayerNorm.weight")
        kq, kk, kv = (b + f"attention.self.{x}.weight" for x in ("query", "key", "value"))
        yield AttnBlock(i, "self", lin_heads(ws.get(kq), n, dh), lin_heads(ws.get(kk), n, dh),
                        lin_heads(ws.get(kv), n, dh), _wo(lambda: lin_out_heads(ws.get(b + "attention.output.dense.weight"), n, dh)),
                        g, g, g, np.arange(n), keys=dict(q=kq, k=kk, v=kv, layout="linear"))

def _blocks_distilbert(ws):
    c = ws.config; d = c["dim"]; n = c["n_heads"]; dh = d // n; L = c["n_layers"]
    p = ws.prefix(r"^(.*?)transformer\.layer\.0\.attention\.q_lin\.weight$")
    for i in range(L):
        b = f"{p}transformer.layer.{i}.attention."
        g = ws.get_opt(f"{p}embeddings.LayerNorm.weight" if i == 0 else f"{p}transformer.layer.{i-1}.output_layer_norm.weight")
        yield AttnBlock(i, "self", lin_heads(ws.get(b + "q_lin.weight"), n, dh), lin_heads(ws.get(b + "k_lin.weight"), n, dh),
                        lin_heads(ws.get(b + "v_lin.weight"), n, dh), _wo(lambda: lin_out_heads(ws.get(b + "out_lin.weight"), n, dh)),
                        g, g, g, np.arange(n))

def _blocks_mobilebert(ws):
    c = ws.config; n = c["num_attention_heads"]; tdh = c["true_hidden_size"]; dh = tdh // n; L = c["num_hidden_layers"]
    p = ws.prefix(r"^(.*?)encoder\.layer\.0\.attention\.self\.query\.weight$")
    assert c.get("key_query_shared_bottleneck", True) and not c.get("use_bottleneck_attention", False)
    for i in range(L):
        b = f"{p}encoder.layer.{i}."
        g_prev = ws.get_opt(f"{p}embeddings.LayerNorm.weight" if i == 0 else f"{p}encoder.layer.{i-1}.output.bottleneck.LayerNorm.weight")
        Wb = ws.get(b + "bottleneck.attention.dense.weight")            # (tdh, hidden)
        gb = ws.get_opt(b + "bottleneck.attention.LayerNorm.weight")      # NoNorm gain (tdh,)
        Wq_b = lin_heads(ws.get(b + "attention.self.query.weight"), n, dh)  # (n, tdh, dh)
        Wk_b = lin_heads(ws.get(b + "attention.self.key.weight"), n, dh)
        Wv = lin_heads(ws.get(b + "attention.self.value.weight"), n, dh)    # (n, hidden, dh)
        gbv = gb.view(1, -1, 1) if gb is not None else 1.0
        # compose the bottleneck map so that Q, K, V all read the hidden stream (NoNorm is affine)
        Wq = torch.einsum("th,ntk->nhk", Wb, gbv * Wq_b)
        Wk = torch.einsum("th,ntk->nhk", Wb, gbv * Wk_b)
        blk = AttnBlock(i, "self", Wq, Wk, Wv, None, g_prev, g_prev, g_prev, np.arange(n))
        blk.pair_override["QK"] = (gbv * Wq_b, gbv * Wk_b)   # Q-K inside the 128-d bottleneck
        yield blk

def _blocks_gpt2(ws):
    c = ws.config; d = c["n_embd"]; n = c["n_head"]; dh = d // n; L = c["n_layer"]
    p = ws.prefix(r"^(.*?)h\.0\.attn\.c_attn\.weight$")
    for i in range(L):
        b = f"{p}h.{i}."
        W = ws.get(b + "attn.c_attn.weight")   # Conv1D (d, 3d)
        per = lambda M: M.reshape(d, n, dh).permute(1, 0, 2).contiguous()
        Wo = _wo(lambda: ws.get(b + "attn.c_proj.weight").reshape(n, dh, d))
        g = ws.get_opt(b + "ln_1.weight")
        yield AttnBlock(i, "self", per(W[:, :d]), per(W[:, d:2 * d]), per(W[:, 2 * d:]), Wo, g, g, g, np.arange(n),
                        keys=dict(qkv=b + "attn.c_attn.weight", layout="gpt2"))

def _blocks_neox(ws):
    c = ws.config; d = c["hidden_size"]; n = c["num_attention_heads"]; dh = d // n; L = c["num_hidden_layers"]
    p = ws.prefix(r"^(.*?)layers\.0\.attention\.query_key_value\.weight$")
    rp = rope_info(c, dh, "neox")
    for i in range(L):
        b = f"{p}layers.{i}."
        W = ws.get(b + "attention.query_key_value.weight").reshape(n, 3, dh, d)
        g = ws.get_opt(b + "input_layernorm.weight")
        yield AttnBlock(i, "self", W[:, 0].transpose(1, 2).contiguous(), W[:, 1].transpose(1, 2).contiguous(),
                        W[:, 2].transpose(1, 2).contiguous(), _wo(lambda: lin_out_heads(ws.get(b + "attention.dense.weight"), n, dh)),
                        g, g, g, np.arange(n), rope=rp, keys=dict(qkv=b + "attention.query_key_value.weight", layout="neox"))

def _blocks_llamalike(ws):
    c = ws.config; d = c["hidden_size"]; n = c["num_attention_heads"]; nkv = c.get("num_key_value_heads") or n
    dh = c.get("head_dim") or d // n; L = c["num_hidden_layers"]
    p = ws.prefix(r"^(.*?)layers\.0\.self_attn\.q_proj\.weight$")
    rp = rope_info(c, dh, ws.model_type)
    rep = n // nkv
    for i in range(L):
        b = f"{p}layers.{i}."
        g = ws.get_opt(b + "input_layernorm.weight")   # None for OLMo-2 (norm after attention)
        okey = b + ("self_attn.o_proj.weight" if ws.has(b + "self_attn.o_proj.weight") else "self_attn.dense.weight")
        kq, kk, kv = (b + f"self_attn.{x}_proj.weight" for x in "qkv")
        yield AttnBlock(i, "self", lin_heads(ws.get(kq), n, dh), lin_heads(ws.get(kk), nkv, dh),
                        lin_heads(ws.get(kv), nkv, dh), _wo(lambda: lin_out_heads(ws.get(okey), n, dh)),
                        g, g, g, np.arange(n) // rep, rope=rp, keys=dict(q=kq, k=kk, v=kv, layout="linear"))

def _blocks_t5(ws):
    c = ws.config; d = c["d_model"]; n = c["num_heads"]; dh = c["d_kv"]
    Le, Ld = c["num_layers"], c.get("num_decoder_layers") or c["num_layers"]
    genc = ws.get_opt("encoder.final_layer_norm.weight")
    def att(prefix, nm):
        return [lin_heads(ws.get(f"{prefix}{nm}.{x}.weight"), n, dh) for x in "qkv"] + \
               [_wo(lambda: lin_out_heads(ws.get(f"{prefix}{nm}.o.weight"), n, dh))]
    for i in range(Le):
        q, k, v, o = att(f"encoder.block.{i}.layer.0.", "SelfAttention")
        g = ws.get_opt(f"encoder.block.{i}.layer.0.layer_norm.weight")
        yield AttnBlock(i, "enc-self", q, k, v, o, g, g, g, np.arange(n), "enc", "enc", "enc")
    for i in range(Ld):
        q, k, v, o = att(f"decoder.block.{i}.layer.0.", "SelfAttention")
        g = ws.get_opt(f"decoder.block.{i}.layer.0.layer_norm.weight")
        yield AttnBlock(i, "dec-self", q, k, v, o, g, g, g, np.arange(n), "dec", "dec", "dec")
        q, k, v, o = att(f"decoder.block.{i}.layer.1.", "EncDecAttention")
        gq = ws.get_opt(f"decoder.block.{i}.layer.1.layer_norm.weight")
        yield AttnBlock(i, "cross", q, k, v, o, gq, genc, genc, np.arange(n), "dec", "enc", "enc")

def _blocks_bart(ws):
    c = ws.config; d = c["d_model"]
    p = ws.prefix(r"^(.*?)encoder\.layers\.0\.self_attn\.q_proj\.weight$")
    ne, nd = c["encoder_attention_heads"], c["decoder_attention_heads"]
    Le, Ld = c["encoder_layers"], c["decoder_layers"]
    def att(prefix, n):
        dh = d // n
        return [lin_heads(ws.get(f"{prefix}{x}_proj.weight"), n, dh) for x in "qkv"] + \
               [_wo(lambda: lin_out_heads(ws.get(f"{prefix}out_proj.weight"), n, dh))]
    g_enc_out = ws.get_opt(f"{p}encoder.layer_norm.weight")
    if g_enc_out is None:
        g_enc_out = ws.get_opt(f"{p}encoder.layers.{Le-1}.final_layer_norm.weight")
    for i in range(Le):
        q, k, v, o = att(f"{p}encoder.layers.{i}.self_attn.", ne)
        g = ws.get_opt(f"{p}encoder.layernorm_embedding.weight" if i == 0 else f"{p}encoder.layers.{i-1}.final_layer_norm.weight")
        yield AttnBlock(i, "enc-self", q, k, v, o, g, g, g, np.arange(ne), "enc", "enc", "enc")
    for i in range(Ld):
        q, k, v, o = att(f"{p}decoder.layers.{i}.self_attn.", nd)
        g = ws.get_opt(f"{p}decoder.layernorm_embedding.weight" if i == 0 else f"{p}decoder.layers.{i-1}.final_layer_norm.weight")
        yield AttnBlock(i, "dec-self", q, k, v, o, g, g, g, np.arange(nd), "dec", "dec", "dec")
        q, k, v, o = att(f"{p}decoder.layers.{i}.encoder_attn.", nd)
        gq = ws.get_opt(f"{p}decoder.layers.{i}.self_attn_layer_norm.weight")
        yield AttnBlock(i, "cross", q, k, v, o, gq, g_enc_out, g_enc_out, np.arange(nd), "dec", "enc", "enc")

def _blocks_tinygpt(ws):
    c = ws.config; d, n, L = c["d"], c["H"], c["L"]; dh = d // n
    nkv = c.get("n_kv") or n
    rp = None
    if c["pe"] == "rope":
        rp = {"inv_freq": 1.0 / (c.get("rope_base", 10000.0) ** (torch.arange(0, dh, 2, dtype=torch.float64) / dh)),
              "rot_dim": dh}
    for i in range(L):
        b = f"blocks.{i}."
        g = ws.get_opt(b + "ln1.weight")
        kq, kk, kv = (b + f"attn.{x}.weight" for x in "qkv")
        yield AttnBlock(i, "self", lin_heads(ws.get(kq), n, dh), lin_heads(ws.get(kk), nkv, dh), lin_heads(ws.get(kv), nkv, dh),
                        _wo(lambda: lin_out_heads(ws.get(b + "attn.o.weight"), n, dh)), g, g, g, np.arange(n) // (n // nkv), rope=rp,
                        keys=dict(q=kq, k=kk, v=kv, layout="linear"))

ADAPTERS = {"bert": _blocks_bert, "distilbert": _blocks_distilbert, "mobilebert": _blocks_mobilebert,
            "gpt2": _blocks_gpt2, "gpt_neox": _blocks_neox, "llama": _blocks_llamalike, "mistral": _blocks_llamalike,
            "qwen2": _blocks_llamalike, "qwen2_moe": _blocks_llamalike, "olmoe": _blocks_llamalike,
            "olmo2": _blocks_llamalike, "phi": _blocks_llamalike, "t5": _blocks_t5, "bart": _blocks_bart,
            "tinygpt": _blocks_tinygpt}

def iter_blocks(ws: WeightSource):
    mt = ws.config.get("model_type")
    if mt not in ADAPTERS:
        raise ValueError(f"no adapter for model_type={mt} ({ws.model_id})")
    yield from ADAPTERS[mt](ws)

### 1.4 Principal angles, null calibration and per-head geometry
For orthonormal bases $U_A\in\mathbb{R}^{n\times k}$, $U_B\in\mathbb{R}^{n\times k'}$ ($m=\min(k,k')$),
$\cos\theta_i=\sigma_i(U_A^\top U_B)$. Reported quantities:
overlap $=\frac1m\sum\cos^2\theta_i$; geodesic $\|\theta\|_2$; chordal $\|\sin\theta\|_2$;
projection-F $\|P_A-P_B\|_F=\sqrt{k+k'-2\sum\cos^2\theta_i}$; analytic null
$\mu_0=\max(k,k')/n$; normalised excess $\rho=(\text{overlap}-\mu_0)/(1-\mu_0)$;
$z=(\sum\cos^2-kk'/n)/\sqrt{2kk'(n-k)(n-k')/(n^2(n-1)(n+2))}$.
The *structured* null replaces the matched partner by the other heads of the same layer
(mismatched pairs), which controls for anisotropy shared by all heads. The spectral
self-alignment $s=\operatorname{tr}(M)/\|M\|_*$ of $M=W_QW_K^\top$ is a gauge-invariant,
spectrum-weighted complement.

**Device.** When a GPU is available the per-block computation runs on it, still in float64
(`QKV_GEOM_DEVICE=cpu` forces the CPU). The large operations ($d\times d_h$ factorisations, the
all-pairs Gram products) run on the GPU; the small $d_h\times d_h$ SVDs run on the CPU. Orthonormal
bases are obtained as $W=QR$, $R=U_R\Sigma V^\top$, $U=QU_R$, which is an SVD of $W$; every
reported quantity depends only on the column spaces and singular values, hence not on this choice.
A self-check at start-up compares each GPU with the CPU on a random block and drops any GPU that
differs by more than $10^{-9}$. With several GPUs, successive blocks of a checkpoint are analysed on
different GPUs in parallel while the next blocks are being fetched.

In [ ]:
GEOM_DTYPE = torch.float64
GEOM_VERSION = "v3"
GEOM_DEVICE = os.environ.get("QKV_GEOM_DEVICE", "auto")
if GEOM_DEVICE == "auto":
    GEOM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_GEOM_QR = os.environ.get("QKV_GEOM_QR", "0") == "1"      # QR route on the CPU as well (tests only)

def orth(W: torch.Tensor, rtol: float = 1e-7):
    """Orthonormal basis of the column space. Returns U (on W's device), S and numerical ranks (CPU).
    CPU tensors: batched SVD. GPU tensors: QR on the GPU, then the SVD of the small R factor on the CPU."""
    W = W.to(GEOM_DTYPE)
    if (W.is_cuda or _GEOM_QR) and W.shape[-2] >= W.shape[-1]:
        Q, Rf = torch.linalg.qr(W)
        Ur, S, _ = torch.linalg.svd(Rf.cpu(), full_matrices=False)
        U = Q @ Ur.to(W.device)
    elif W.is_cuda:
        U, S, _ = torch.linalg.svd(W.cpu(), full_matrices=False)
        U = U.to(W.device)
    else:
        U, S, _ = torch.linalg.svd(W, full_matrices=False)
    ranks = (S > rtol * S[..., :1]).sum(-1)
    return U, S, ranks

def cosines(Ua: torch.Tensor, Ub: torch.Tensor) -> torch.Tensor:
    """Principal cosines (CPU tensor); the product is formed on the inputs' device."""
    X = Ua.mT @ Ub
    return torch.linalg.svdvals(X.cpu() if X.is_cuda else X).clamp(0.0, 1.0)

def null_mean(k1, k2, n):
    return max(k1, k2) / n

def null_var_sumcos2(k1, k2, n):
    return 2.0 * k1 * k2 * (n - k1) * (n - k2) / (n ** 2 * (n - 1) * (n + 2))

def pair_metrics(c: torch.Tensor, k1: int, k2: int, n: int) -> dict:
    """c: (..., m) principal cosines -> dict of numpy arrays."""
    c = c.to(GEOM_DTYPE)
    m = min(k1, k2)
    s2 = (c ** 2).sum(-1)
    ov = s2 / m
    mu0 = null_mean(k1, k2, n)
    th = torch.arccos(c)
    out = dict(overlap=ov, sumcos2=s2, null=torch.full_like(ov, mu0),
               rho=(ov - mu0) / (1 - mu0),
               z=(s2 - k1 * k2 / n) / math.sqrt(null_var_sumcos2(k1, k2, n)),
               ratio=ov / mu0,
               geodesic=th.norm(dim=-1), chordal=torch.sqrt((m - s2).clamp(min=0)),
               projF=torch.sqrt((k1 + k2 - 2 * s2).clamp(min=0)),
               mean_angle=torch.rad2deg(th).mean(-1), min_angle=torch.rad2deg(th[..., 0]))
    return {k: v.cpu().numpy() for k, v in out.items()}

def overlap_matrix(Ua: torch.Tensor, Ub: torch.Tensor) -> torch.Tensor:
    """Ua (na,n,k), Ub (nb,n,k') -> (na, nb) overlaps (mean cos^2 over min(k,k') angles).
    Uses sum_i cos^2(theta_i) = ||Ua_i^T Ub_j||_F^2 (orthonormal bases), i.e. one GEMM for all pairs
    instead of na*nb small SVDs (identical values up to rounding)."""
    na, n, k1 = Ua.shape
    nb, _, k2 = Ub.shape
    A = Ua.to(GEOM_DTYPE).permute(1, 0, 2).reshape(n, na * k1)
    B = Ub.to(GEOM_DTYPE).permute(1, 0, 2).reshape(n, nb * k2)
    G = (A.mT @ B).reshape(na, k1, nb, k2)
    return ((G ** 2).sum((1, 3)) / min(k1, k2)).cpu()

def stable_rank(S):
    return ((S ** 2).sum(-1) / (S[..., 0] ** 2)).cpu().numpy()

def block_geometry(blk: AttnBlock, fold: bool = True, device: Optional[str] = None) -> List[dict]:
    """Per-head rows for one attention block (matched pairs, structured null, self-alignment)."""
    rows = []
    dev = torch.device(device or GEOM_DEVICE)
    Wq, Wk, Wv = (blk.folded(x, fold).to(dev, GEOM_DTYPE) for x in "qkv")
    Uq, Sq, rq = orth(Wq); Uk, Sk, rk = orth(Wk); Uv, Sv, rv = orth(Wv)
    if min(rq.min(), rk.min(), rv.min()) < blk.dh:
        warnings.warn(f"rank-deficient head in layer {blk.layer} ({blk.kind})")
    n, dh, g = blk.d, blk.dh, blk.kv_of_q
    srq, srk, srv = stable_rank(Sq), stable_rank(Sk), stable_rank(Sv)
    # spectral self-alignment of M = Wq Wk^T (exact, through k x k cores)
    Tq = Uq.mT @ Wq; Tk = Uk.mT @ Wk
    C = Tq @ Tk[g].mT                                   # (n_q, k, k)
    UkUq = Uk[g].mT @ Uq
    C = C.cpu()
    s_align = (torch.einsum("hij,hji->h", C, UkUq.cpu()) / torch.linalg.svdvals(C).sum(-1)).numpy()

    def add(pair, A, B, ia, ib, mis_mat, nn_, cross_space):
        k1, k2 = A.shape[-1], B.shape[-1]
        met = pair_metrics(cosines(A[ia], B[ib]), k1, k2, nn_)
        for j in range(len(ia)):
            row = dict(layer=blk.layer, kind=blk.kind, pair=pair, head=int(ia[j]), partner=int(ib[j]),
                       n=nn_, k1=k1, k2=k2, cross_space=cross_space, **{key: float(val[j]) for key, val in met.items()})
            if mis_mat is not None and mis_mat.shape[1] > 1:
                mask = torch.ones(mis_mat.shape[1], dtype=torch.bool); mask[ib[j]] = False
                mo = float(mis_mat[ia[j], mask].mean()); mu0 = null_mean(k1, k2, nn_)
                row.update(mis_overlap=mo, mis_rho=(mo - mu0) / (1 - mu0))
                row["coupling"] = row["rho"] - row["mis_rho"]
            if pair == "QK":
                row.update(s_align=float(s_align[ia[j]]), srank_a=float(srq[ia[j]]), srank_b=float(srk[ib[j]]))
            rows.append(row)

    hq = np.arange(blk.n_q); hk = np.arange(blk.n_kv)
    # Q-K
    if "QK" in blk.pair_override:
        A, B = blk.pair_override["QK"]
        UA, _, _ = orth(A.to(dev, GEOM_DTYPE)); UB, _, _ = orth(B.to(dev, GEOM_DTYPE))
        add("QK", UA, UB, hq, g, overlap_matrix(UA, UB), UA.shape[1], False)
    else:
        add("QK", Uq, Uk, hq, g, overlap_matrix(Uq, Uk), n, blk.space_q != blk.space_k)
    add("KV", Uk, Uv, hk, hk, overlap_matrix(Uk, Uv), n, blk.space_k != blk.space_v)
    add("QV", Uq, Uv, hq, g, overlap_matrix(Uq, Uv), n, blk.space_q != blk.space_v)
    # GQA: span of each group's query heads vs the shared key head
    if blk.n_kv < blk.n_q and "QK" not in blk.pair_override:
        for gi in range(blk.n_kv):
            members = np.where(g == gi)[0]
            UG, _, _ = orth(torch.cat([Wq[h] for h in members], dim=-1).unsqueeze(0))
            met = pair_metrics(cosines(UG, Uk[gi:gi + 1]), UG.shape[-1], dh, n)
            rows.append(dict(layer=blk.layer, kind=blk.kind, pair="QKgroup", head=int(gi), partner=int(gi), n=n,
                             k1=UG.shape[-1], k2=dh, cross_space=False, **{k_: float(v[0]) for k_, v in met.items()}))
    return rows

def _geom_selfcheck(device: str) -> float:
    """Largest absolute difference between block_geometry on `device` and on the CPU (random GQA block
    with normalisation gains)."""
    g_ = torch.Generator().manual_seed(0)
    n_, nkv_, d_, dh_ = 8, 2, 256, 32
    rnd = lambda *sh: torch.randn(*sh, generator=g_, dtype=GEOM_DTYPE)
    gain = 1 + 0.2 * rnd(d_)
    blk = AttnBlock(0, "self", rnd(n_, d_, dh_), rnd(nkv_, d_, dh_), rnd(nkv_, d_, dh_), None, gain, gain, gain,
                    np.arange(n_) // (n_ // nkv_))
    a = pd.DataFrame(block_geometry(blk, device="cpu")); b = pd.DataFrame(block_geometry(blk, device=device))
    num = a.select_dtypes("number").columns
    return float(np.nanmax(np.abs(a[num].to_numpy(float) - b[num].to_numpy(float))))

GEOM_DEVICES = [GEOM_DEVICE]          # devices that process successive blocks of a checkpoint in parallel
if GEOM_DEVICE != "cpu":
    _cand = [g for g in GPUS if g.startswith("cuda")] if GEOM_DEVICE == "cuda" else [GEOM_DEVICE]
    _ok = []
    for _dv in (_cand or ["cuda:0"]):
        try:
            _err = _geom_selfcheck(_dv)
            if not _err < 1e-9:
                raise RuntimeError(f"GPU and CPU differ by {_err:.1e}")
            _ok.append(_dv)
            print(f"geometry on {_dv} in float64 ({torch.cuda.get_device_name(_dv)}); "
                  f"self-check vs CPU: max difference {_err:.1e}")
        except Exception as e:
            print(f"[geometry] {_dv} not used ({repr(e)[:150]})")
    GEOM_DEVICES = _ok or ["cpu"]
    GEOM_DEVICE = GEOM_DEVICES[0]
    if not _ok:
        print("[geometry] no usable GPU; geometry runs on the CPU")
else:
    print("geometry on cpu (float64)")
if os.environ.get("QKV_TEST_DEVICES"):                 # tests only: exercise the multi-device pipeline on the CPU
    GEOM_DEVICES = os.environ["QKV_TEST_DEVICES"].split(",")

def model_info(ws: WeightSource, first_block: AttnBlock, n_layers: int) -> dict:
    tot, nonemb = ws.param_counts()
    meta = REG.get(ws.model_id, {})
    return dict(model=ws.model_id, revision=ws.revision, model_type=ws.model_type, family=meta.get("family"),
                pe=meta.get("pe"), attn=meta.get("attn"), procedure=meta.get("procedure"),
                d_model=first_block.d, d_head=first_block.dh, n_q=first_block.n_q, n_kv=first_block.n_kv,
                n_layers=n_layers, params=tot, params_nonemb=nonemb,
                rot_frac=(first_block.rope["rot_dim"] / first_block.dh) if first_block.rope else 0.0)

MISSING: Dict[str, str] = {}

def record_missing(key: str, e: Exception, strict: bool = False):
    """A checkpoint that cannot be used (absent from the Hub, gated without access, unsupported format) is
    recorded and skipped. Failures of the run itself are not missing checkpoints and stop the cell, whose results
    then stay pending, so that no statistic is computed on a set of models reduced by them: a network failure
    (NetworkUnavailable), GPU memory or disk exhaustion, and, with strict=True (training jobs), any error that
    is not an unavailable checkpoint."""
    if is_network_error(e):
        NET_DOWN.set()
        raise NetworkUnavailable(f"{key}: the Hugging Face Hub cannot be reached ({type(e).__name__}: {str(e)[:160]})") from e
    if is_resource_error(e) or (strict and not is_unavailable_checkpoint(e)):
        raise e
    MISSING[key] = repr(e)[:300]

def open_weights(model_id: str, revision: Optional[str] = None) -> WeightSource:
    """WeightSource with retries on network errors (see hub_retry)."""
    return hub_retry(lambda: WeightSource(model_id, revision), f"weights of {model_id}" + (f"@{revision}" if revision else ""))

def _geom_cache_path(model_id, revision, fold):
    key = hashlib.sha1(f"{model_id}|{revision}|{fold}|{GEOM_VERSION}".encode()).hexdigest()[:16]
    return CACHE / "geometry" / f"{key}.pkl"

def _n_layers_cfg(cfg: dict):
    n = cfg_get(cfg, "num_hidden_layers", "n_layer", "num_layers", "n_layers", "L")
    if cfg.get("model_type") in ("t5", "bart"):
        return None                     # encoder + decoder (+ cross) blocks: no single count
    return n

def _finish_geometry(rows, model_id, revision):
    df = pd.DataFrame(rows)
    df["n_layers_kind"] = df.groupby("kind")["layer"].transform("max") + 1
    df["depth"] = df["layer"] / (df["n_layers_kind"] - 1).clip(lower=1)
    df.insert(0, "model", model_id)
    df.insert(1, "revision", revision)
    return df

def model_geometry(model_id: str, revision: Optional[str] = None, fold: bool = True, use_cache: bool = True,
                   ws: Optional[WeightSource] = None, both_folds: bool = False, verbose: bool = True,
                   device: Optional[str] = None):
    """Per-head geometry of every attention block of a checkpoint (cached on disk).
    With both_folds=True the gain-folded and the raw parametrisations are computed in the same pass, so the
    weights are fetched once. Returns (DataFrame, info) for `fold`, or (None, None) if unavailable."""
    cpath = _geom_cache_path(model_id, revision, fold)
    cacheable = use_cache and ws is None
    targets = [fold] + ([not fold] if both_folds and cacheable else [])
    todo = [f for f in targets if not (cacheable and _geom_cache_path(model_id, revision, f).exists())]
    if not todo:
        obj = pd.read_pickle(cpath)
        return obj["df"], obj["info"]
    ws_given = ws is not None
    if not ws_given:
        check_budget(f"(geometry of {model_id})")      # finished checkpoints are cached; the next session continues
    t0, b0 = time.time(), NET["bytes"]
    try:
        ws = ws or open_weights(model_id, revision)
    except Exception as e:
        record_missing(f"{model_id}@{revision}", e)        # raises NetworkUnavailable on a network failure
        print(f"[skip] {model_id}@{revision}: {repr(e)[:200]}")
        return None, None
    L_cfg = _n_layers_cfg(ws.config)
    tag = model_id + (f"@{revision}" if revision else "")
    verbose = verbose and ws.mode != "dict"
    if verbose and not ws_given:
        print(f"[geom] {tag}: {ws.mode} weights, {L_cfg or '?'} layers, {device or '+'.join(GEOM_DEVICES)}, "
              f"{'folded + raw' if len(todo) == 2 else ('folded' if todo[0] else 'raw')}", flush=True)
    rows = {f: [] for f in todo}
    devs = [str(device)] if device is not None else list(GEOM_DEVICES)
    # GPU(s): blocks are analysed in one worker thread per device (block i on device i mod n) while the
    # main thread fetches the next blocks; results are collected in block order.
    pipelined = len(devs) > 1 or devs[0] != "cpu"

    def compute(blk, dv):
        if dv.startswith("cuda"):
            torch.cuda.set_device(dv)
        r0 = block_geometry(blk, todo[0], dv)
        r1 = None
        if len(todo) == 2:
            has_gain = any(x is not None for x in (blk.gq, blk.gk, blk.gv))
            r1 = block_geometry(blk, todo[1], dv) if has_gain else [dict(r) for r in r0]
        return r0, r1

    def collect(res):
        rows[todo[0]].extend(res[0])
        if res[1] is not None:
            rows[todo[1]].extend(res[1])

    pools = [ThreadPoolExecutor(1, thread_name_prefix="qkv-geom") for _ in devs] if pipelined else []
    pending = collections.deque()
    first, L, last_print = None, 0, time.time()
    try:
        for i, blk in enumerate(iter_blocks(ws)):
            first = first or blk
            L = max(L, blk.layer + 1)
            if pipelined:
                pending.append(pools[i % len(devs)].submit(compute, blk, devs[i % len(devs)]))
                while len(pending) > 2 * len(devs):          # bounded look-ahead (memory)
                    collect(pending.popleft().result())
            else:
                collect(compute(blk, devs[0]))
            if verbose and not ws_given and time.time() - last_print > 30:
                last_print = time.time()
                print(f"    {blk.kind} layer {blk.layer + 1}/{L_cfg or '?'}  {last_print - t0:.0f} s  "
                      f"{(NET['bytes'] - b0) / 1e6:.0f} MB fetched", flush=True)
        while pending:
            collect(pending.popleft().result())
    finally:
        for p_ in pools:
            p_.shutdown(wait=True, cancel_futures=True)
    info = model_info(ws, first, L)
    if not ws_given:
        ws.close()
    out = {f: _finish_geometry(rows[f], model_id, revision) for f in todo}
    if cacheable:
        for f, df_f in out.items():
            pd.to_pickle({"df": df_f, "info": info}, _geom_cache_path(model_id, revision, f))
    if verbose and not ws_given:
        print(f"[geom] {tag} done in {time.time() - t0:.0f} s ({(NET['bytes'] - b0) / 1e6:.0f} MB fetched)", flush=True)
    if fold in out:
        return out[fold], info
    obj = pd.read_pickle(cpath)
    return obj["df"], obj["info"]

def geometry_many(model_ids, fold=True, revision=None):
    dfs, infos = [], []
    for m in model_ids:
        d_, i_ = model_geometry(m, revision, fold)
        if d_ is not None:
            dfs.append(d_); infos.append(i_)
    return (pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()), pd.DataFrame(infos)

### 1.5 Statistics
* hierarchical bootstrap of a model-level mean (resample layers, then heads within layers);
* layer-clustered sign-flip permutation test for paired head-level differences;
* Holm step-down correction;
* seed noise floor: SD across seeds and the 95th percentile of pairwise |differences|;
* linear vs. segmented (one-breakpoint) regression with AIC/BIC and a wild-bootstrap
  sup-likelihood-ratio test of "no breakpoint" (Davies-type problem: the breakpoint is not
  identified under the null, so the LR statistic is calibrated by bootstrap);
* mixed-effects models (statsmodels MixedLM) and Spearman correlations with bootstrap CIs.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

def hier_boot(df: pd.DataFrame, value: str, n_boot: int = None, seed: int = 0, cluster: str = "layer",
              stat=np.mean):
    """Point estimate and 95% percentile CI of stat(value), resampling clusters then rows."""
    n_boot = n_boot or CFG["n_boot"]
    df = df.dropna(subset=[value])
    if len(df) == 0:
        return np.nan, np.nan, np.nan
    groups = [g[value].to_numpy() for _, g in df.groupby(cluster)]
    est = stat(df[value].to_numpy())
    rng = np.random.default_rng(seed)
    G = len(groups)
    boots = np.empty(n_boot)
    if stat is np.mean:                  # vectorised: clusters padded to a common length and masked
        sizes = np.array([len(g_) for g_ in groups]); nmax = int(sizes.max())
        V = np.zeros((G, nmax))
        for i, g_ in enumerate(groups):
            V[i, :len(g_)] = g_
        chunk = max(1, int(4e6 // (G * nmax)))
        for s0 in range(0, n_boot, chunk):
            nb_ = min(chunk, n_boot - s0)
            pick = rng.integers(0, G, (nb_, G))
            sz = sizes[pick][..., None]                                   # (b, G, 1)
            idx = (rng.random((nb_, G, nmax)) * sz).astype(np.int64)     # within-cluster draws with replacement
            mask = np.arange(nmax) < sz
            vals = V[pick[..., None], idx]
            boots[s0:s0 + nb_] = (vals * mask).sum((1, 2)) / mask.sum((1, 2))
    else:
        for b in range(n_boot):
            pick = rng.integers(0, G, G)
            vals = np.concatenate([groups[i][rng.integers(0, len(groups[i]), len(groups[i]))] for i in pick])
            boots[b] = stat(vals)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(est), float(lo), float(hi)

def signflip_test(df: pd.DataFrame, value: str, cluster: str = "layer", n_perm: int = None, seed: int = 0):
    """Two-sided test of mean(value)=0 with sign flips at the cluster level (exact if feasible)."""
    n_perm = n_perm or CFG["n_perm"]
    m = df.dropna(subset=[value]).groupby(cluster)[value].mean().to_numpy()
    G = len(m)
    obs = abs(m.mean())
    if G == 0:
        return np.nan
    if 2 ** G <= n_perm:                 # exact: the observed sign pattern is one of the 2^G patterns
        signs = np.array(list(itertools.product([-1, 1], repeat=G)))
        null = np.abs((signs * m).mean(1))
        return float((null >= obs - 1e-15).mean())
    signs = np.random.default_rng(seed).choice([-1, 1], size=(n_perm, G))
    null = np.abs((signs * m).mean(1))
    return float((1 + (null >= obs - 1e-15).sum()) / (1 + len(null)))

def cluster_ttest(df: pd.DataFrame, value: str, cluster: str = "layer"):
    """Two-sided one-sample t-test of mean(value)=0 on cluster (layer) means, df = n_clusters - 1.
    Unlike the sign-flip test it has no discreteness floor, which matters for shallow models
    (an exact sign-flip test on 6 layers cannot go below p = 2/64)."""
    m = df.dropna(subset=[value]).groupby(cluster)[value].mean().to_numpy()
    if len(m) < 2 or np.allclose(m.std(ddof=1), 0):
        return np.nan
    return float(st.ttest_1samp(m, 0.0).pvalue)

def perm_test_two_groups(a, b, n_perm=None, seed=0):
    n_perm = n_perm or CFG["n_perm"]
    a, b = np.asarray(a), np.asarray(b)
    obs = abs(a.mean() - b.mean())
    pool = np.concatenate([a, b]); rng = np.random.default_rng(seed)
    cnt = 0
    for _ in range(n_perm):
        rng.shuffle(pool)
        cnt += abs(pool[:len(a)].mean() - pool[len(a):].mean()) >= obs - 1e-15
    return float((1 + cnt) / (1 + n_perm))

def exact_perm_test(xa, xb, max_enum: int = 200000, seed: int = 0):
    """Two-sided permutation test of a difference in means between two small groups of runs.
    Exact enumeration when feasible. Returns (p, smallest attainable p)."""
    from math import comb
    xa, xb = np.asarray(xa, float), np.asarray(xb, float)
    allv = np.r_[xa, xb]; n, k = len(allv), len(xa)
    obs = abs(xa.mean() - xb.mean())
    if comb(n, k) <= max_enum:
        stats = []
        for c in itertools.combinations(range(n), k):
            m = np.zeros(n, bool); m[list(c)] = True
            stats.append(abs(allv[m].mean() - allv[~m].mean()))
        stats = np.array(stats)
        p = float((stats >= obs - 1e-12).mean())
    else:
        rng = np.random.default_rng(seed); cnt = 0
        for _ in range(CFG["n_perm"]):
            perm = rng.permutation(allv); cnt += abs(perm[:k].mean() - perm[k:].mean()) >= obs - 1e-12
        p = float((1 + cnt) / (1 + CFG["n_perm"]))
    pmin = (2.0 if k == n - k else 1.0) / comb(n, k)
    return p, pmin

def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.any():
        out[ok] = multipletests(p[ok], method="holm")[1]
    return out

def seed_floor(values):
    v = np.asarray([x for x in values if np.isfinite(x)])
    if len(v) < 2:
        return dict(n=len(v), sd=np.nan, q95=np.nan, mean=np.nan)
    diffs = np.abs(v[:, None] - v[None, :])[np.triu_indices(len(v), 1)]
    return dict(n=len(v), mean=float(v.mean()), sd=float(v.std(ddof=1)), q95=float(np.percentile(diffs, 95)),
                range=float(v.max() - v.min()))

def _ols(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    return beta, float(res @ res)

def _seg_design(x, psi):
    return np.column_stack([np.ones_like(x), x, np.maximum(x - psi, 0.0)])

def fit_linear_segmented(x, y, qlo=0.15, qhi=0.85, n_grid=200):
    """OLS line vs. continuous one-breakpoint model; returns fits with AIC/BIC (Gaussian likelihood)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    Xl = np.column_stack([np.ones_like(x), x])
    bl, rss_l = _ols(Xl, y)
    lo, hi = np.quantile(x, [qlo, qhi])
    grid = np.linspace(lo, hi, n_grid)
    best = (np.inf, None, None)
    for psi in grid:
        b, rss = _ols(_seg_design(x, psi), y)
        if rss < best[0]:
            best = (rss, psi, b)
    rss_s, psi, bs = best
    rss_s = max(rss_s, 1e-300); rss_l = max(rss_l, 1e-300)
    ll = lambda rss: -0.5 * n * (np.log(2 * np.pi * rss / n) + 1)
    k_l, k_s = 3, 5        # (intercept, slope, sigma) vs (+ slope change, breakpoint)
    Xq = np.column_stack([np.ones_like(x), x, x ** 2])
    bq, rss_q = _ols(Xq, y)
    k_q = 4
    res = dict(n=n, psi=float(psi), beta_lin=bl, beta_seg=bs, beta_quad=bq,
               rss_lin=rss_l, rss_seg=rss_s, rss_quad=rss_q,
               aic_lin=2 * k_l - 2 * ll(rss_l), aic_seg=2 * k_s - 2 * ll(rss_s), aic_quad=2 * k_q - 2 * ll(rss_q),
               bic_lin=k_l * np.log(n) - 2 * ll(rss_l), bic_seg=k_s * np.log(n) - 2 * ll(rss_s),
               bic_quad=k_q * np.log(n) - 2 * ll(rss_q),
               lr=n * np.log(rss_l / rss_s))
    res["dbic"] = res["bic_lin"] - res["bic_seg"]      # >0 favours the breakpoint
    res["daic"] = res["aic_lin"] - res["aic_seg"]
    return res

def breakpoint_test(x, y, n_boot=None, seed=0):
    """Wild (Rademacher) bootstrap p-value of the sup-LR statistic under the linear null."""
    n_boot = n_boot or max(200, CFG["n_boot"] // 2)
    x, y = np.asarray(x, float), np.asarray(y, float)
    fit = fit_linear_segmented(x, y)
    Xl = np.column_stack([np.ones_like(x), x])
    yhat = Xl @ fit["beta_lin"]; res = y - yhat
    rng = np.random.default_rng(seed)
    lr_null = np.empty(n_boot)
    for b in range(n_boot):
        ystar = yhat + res * rng.choice([-1.0, 1.0], size=len(y))
        lr_null[b] = fit_linear_segmented(x, ystar)["lr"]
    fit["p_break"] = float((1 + (lr_null >= fit["lr"]).sum()) / (1 + n_boot))
    return fit

THRESHOLD_RULE = dict(alpha=0.05, min_dbic=6.0)   # pre-registered: breakpoint claimed only if both hold

def spearman_ci(x, y, n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    if len(x) < 4:
        return np.nan, np.nan, np.nan, np.nan
    r, p = st.spearmanr(x, y)
    rng = np.random.default_rng(seed); bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        if np.unique(x[i]).size > 1 and np.unique(y[i]).size > 1:
            bs.append(st.spearmanr(x[i], y[i])[0])
    lo, hi = np.percentile(bs, [2.5, 97.5]) if bs else (np.nan, np.nan)
    return float(r), float(lo), float(hi), float(p)

def partial_spearman(x, y, z, n_boot=None, seed=0):
    """Spearman correlation of x and y after removing the rank-linear effect of z."""
    rx, ry, rz = (st.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return spearman_ci(ex, ey, n_boot, seed)

def mixed_model(formula, df, group):
    try:
        m = smf.mixedlm(formula, df, groups=df[group]).fit(reml=True, method="lbfgs")
        return m
    except Exception as e:
        print("[mixedlm failed]", repr(e)[:200])
        return None

def effective_rank(S):
    """Entropy effective rank (Roy & Vetterli, 2007) of singular values S."""
    p = S / S.sum()
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))

### 1.6 RoPE utilities and alternative similarity measures
With RoPE (rotate-half convention), $q_m=R(m\omega)W_Q^\top x$ and the score between a query at
position $m$ and a key at position $m-\delta$ is $x^\top B_\delta y$ with
$B_\delta=W_Q\,R(-\delta\omega)\,W_K^\top$. Writing $W_Q=U_QT_Q$, $W_K=U_KT_K$, every quantity is
computed through the $k\times k$ core $C_\delta=T_QR(-\delta\omega)T_K^\top$ (exact, no
$d_{\text{model}}^2$ matrix is formed).

In [ ]:
def rope_matrix(inv_freq: torch.Tensor, rot_dim: int, dh: int, delta: float) -> torch.Tensor:
    """R(delta*omega) acting on head coordinates (rotate-half planes (j, j+rot/2); identity elsewhere)."""
    R = torch.eye(dh, dtype=GEOM_DTYPE)
    half = rot_dim // 2
    ang = delta * inv_freq.to(GEOM_DTYPE)
    c, s = torch.cos(ang), torch.sin(ang)
    j = torch.arange(half)
    R[j, j] = c; R[j + half, j + half] = c
    R[j, j + half] = -s; R[j + half, j] = s
    return R

def rope_planes(rot_dim: int):
    half = rot_dim // 2
    return [(j, j + half) for j in range(half)]

def rope_block_analysis(blk: AttnBlock, offsets, topr: int, fold: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Per head and offset: full-rank angles (offset-independent under RoPE), self-alignment s(delta), dominant-subspace overlap;
    per head and frequency plane: 2-D subspace overlaps (RoPE-gauge invariants)."""
    Wq, Wk = blk.folded("q", fold), blk.folded("k", fold)
    Uq, _, _ = orth(Wq); Uk, _, _ = orth(Wk)
    Tq, Tk = Uq.mT @ Wq, Uk.mT @ Wk
    g, n, dh = blk.kv_of_q, blk.d, blk.dh
    inv, rot = blk.rope["inv_freq"], blk.rope["rot_dim"]
    static = (cosines(Uq, Uk[g]) ** 2).mean(-1)
    QtK = Uq.mT @ Uk[g]
    rows = []
    for dlt in offsets:
        Rm = rope_matrix(inv, rot, dh, -dlt)
        C = Tq @ Rm @ Tk[g].mT
        Y, S, Zh = torch.linalg.svd(C)
        Z = Zh.mT
        full = (torch.linalg.svdvals(Y.mT @ QtK @ Z).clamp(0, 1) ** 2).mean(-1)   # Prop. 2: equals `static`
        s_al = torch.einsum("hij,hji->h", C, QtK.mT) / S.sum(-1)
        cr = torch.linalg.svdvals(Y[..., :topr].mT @ QtK @ Z[..., :topr]).clamp(0, 1)
        ov_r = (cr ** 2).mean(-1)
        for h in range(blk.n_q):
            rows.append(dict(layer=blk.layer, head=h, delta=dlt, static_overlap=float(static[h]),
                             full_rank_overlap=float(full[h]), prop2_dev=float(abs(full[h] - static[h])),
                             s_align=float(s_al[h]), top_r_overlap=float(ov_r[h]), top_r_null=topr / n,
                             top_r_rho=float((ov_r[h] - topr / n) / (1 - topr / n)),
                             sv_top_frac=float(S[h, :topr].sum() / S[h].sum())))
    prow = []
    Wkg = Wk[g]
    for j, (a, b) in enumerate(rope_planes(rot)):
        UA, _, _ = orth(Wq[:, :, [a, b]]); UB, _, _ = orth(Wkg[:, :, [a, b]])
        ov = (cosines(UA, UB) ** 2).mean(-1)
        for h in range(blk.n_q):
            prow.append(dict(layer=blk.layer, head=h, plane=j, omega=float(inv[j]), overlap=float(ov[h]),
                             rho=float((ov[h] - 2 / n) / (1 - 2 / n))))
    if rot < dh:   # pass-through block of partial RoPE (GL-gauge invariant as a whole)
        idx = list(range(rot, dh))
        UA, _, _ = orth(Wq[:, :, idx]); UB, _, _ = orth(Wkg[:, :, idx])
        ov = (cosines(UA, UB) ** 2).mean(-1); k_ = len(idx)
        for h in range(blk.n_q):
            prow.append(dict(layer=blk.layer, head=h, plane=-1, omega=0.0, overlap=float(ov[h]),
                             rho=float((ov[h] - k_ / n) / (1 - k_ / n))))
    return pd.DataFrame(rows), pd.DataFrame(prow)

# ---------- alternative similarity measures between two read matrices X (n x k), Y (n x k') ----------
def _unit(X):
    return X / X.norm()

def sim_overlap(X, Y):
    UX, _, _ = orth(X); UY, _, _ = orth(Y)
    return float((cosines(UX, UY) ** 2).sum() / min(X.shape[1], Y.shape[1]))

def sim_geodesic(X, Y):
    UX, _, _ = orth(X); UY, _, _ = orth(Y)
    return float(torch.arccos(cosines(UX, UY)).norm())

def sim_chordal(X, Y):
    UX, _, _ = orth(X); UY, _, _ = orth(Y)
    c = cosines(UX, UY); return float(torch.sqrt((1 - c ** 2).clamp(min=0).sum()))

def sim_projF(X, Y):
    UX, _, _ = orth(X); UY, _, _ = orth(Y)
    return float((UX @ UX.mT - UY @ UY.mT).norm())

def sim_cka(X, Y):
    """Linear CKA, uncentred (rows index residual coordinates, not samples)."""
    return float((X.mT @ Y).norm() ** 2 / ((X.mT @ X).norm() * (Y.mT @ Y).norm()))

def _truncate(X, var=0.99):
    U, S, _ = torch.linalg.svd(X, full_matrices=False)
    e = torch.cumsum(S ** 2, 0) / (S ** 2).sum()
    r = int((e < var).sum()) + 1
    return U[:, :r], S[:r]

def sim_svcca(X, Y, var=0.99):
    UX, _ = _truncate(X, var); UY, _ = _truncate(Y, var)
    return float(cosines(UX, UY).mean())

def sim_pwcca(X, Y):
    """Projection-weighted CCA (Morcos et al., 2018), weights computed on X's columns."""
    UX, _, _ = orth(X); UY, _, _ = orth(Y)
    A, c, _ = torch.linalg.svd(UX.mT @ UY, full_matrices=False)
    m = c.shape[0]
    H = UX @ A[:, :m]                        # canonical variates of X in R^n
    alpha = (H.mT @ X).abs().sum(1)
    alpha = alpha / alpha.sum()
    return float((alpha * c.clamp(0, 1)).sum())

def sim_procrustes(X, Y):
    """Similarity 1 - d^2/2 with d the orthogonal Procrustes distance of Frobenius-normalised X, Y."""
    return float(torch.linalg.svdvals(_unit(X).mT @ _unit(Y)).sum())

def sim_rawcos(X, Y):
    if X.shape != Y.shape:
        return np.nan
    return float((X * Y).sum() / (X.norm() * Y.norm()))

def sim_selfalign(X, Y):
    """s = tr(X Y^T) / ||X Y^T||_* (functional invariant of M = X Y^T)."""
    M = X @ Y.mT
    return float(torch.trace(M) / torch.linalg.svdvals(M).sum())

METRICS = {"overlap": sim_overlap, "geodesic": sim_geodesic, "chordal": sim_chordal, "projF": sim_projF,
           "CKA": sim_cka, "SVCCA": sim_svcca, "PWCCA": sim_pwcca, "Procrustes": sim_procrustes,
           "raw cosine": sim_rawcos, "self-alignment s": sim_selfalign}

### 1.7 Functional evaluation: corpora, perplexity, MLM accuracy, attention-based head scores,
activation capture, Grassmann surgery and fine-tuning helpers

In [ ]:
from transformers import AutoTokenizer

class SmokeTok:
    """Stand-in tokenizer for the smoke test (random token ids)."""
    def __init__(self, vocab=SMOKE_VOCAB):
        self.vocab_size, self.bos_token_id, self.eos_token_id = vocab, 1, 2
        self.pad_token_id, self.cls_token_id, self.sep_token_id, self.mask_token_id = 0, 1, 2, 3
        self.name_or_path = "smoke"

def load_model(model_id: str, task: Optional[str] = None, dtype=None, num_labels: int = 2, device=None):
    """Model and tokenizer from the Hub, with retries on network errors (see hub_retry)."""
    task = task or default_task(model_id)
    device = device if device is not None else DEVICE
    if SMOKE:
        return smoke_model(model_id, task=task, num_labels=num_labels).to(device), SmokeTok()
    return hub_retry(lambda: _load_model_once(model_id, task, dtype, num_labels, device), f"model {model_id}")

def _load_model_once(model_id: str, task: str, dtype, num_labels: int, device):
    import transformers as T
    cls = getattr(T, _TASK_CLASSES[task])
    if dtype is None:
        dtype = torch.bfloat16 if (GPU_BF16 and "large" in REG.get(model_id, {}).get("tags", set())) else torch.float32
    kw = dict(dtype=dtype, attn_implementation="eager", cache_dir=str(HF_CACHE))
    if task == "cls":
        kw["num_labels"] = num_labels
    try:
        files = repo_files(model_id)
        w = {f: s for f, s in files.items() if "/" not in f and (f.endswith(".safetensors") or f.endswith(".bin"))}
        st_ = {f: s for f, s in w.items() if f.endswith(".safetensors")}
        ensure_space(sum((st_ or w).values()))
    except OSError:
        raise
    except Exception as e:
        if is_network_error(e):
            raise
        print("[disk] could not size", model_id, repr(e)[:100])
    ACTIVE_REPOS.add(model_id)
    try:
        model = cls.from_pretrained(model_id, **kw).to(device).eval()
        tok = AutoTokenizer.from_pretrained(model_id, cache_dir=str(HF_CACHE))
    finally:
        ACTIVE_REPOS.discard(model_id)
    if evict_enabled():
        evict_repo(model_id)        # weights now live in memory; free the disk
    return model, tok

def _tok_name(tok):
    return re.sub(r"[^A-Za-z0-9]+", "_", getattr(tok, "name_or_path", "tok"))[-60:]

def corpus_tokens(tok, corpus: str, n_tokens: int, split: str = "validation") -> torch.Tensor:
    """First n_tokens tokens of a corpus (WikiText-103 raw or TinyStories), cached as .npy."""
    path = CACHE / "data" / f"{corpus}_{split}_{_tok_name(tok)}_{n_tokens}.npy"
    if path.exists():
        return torch.from_numpy(np.load(path).astype(np.int64))
    if SMOKE:
        arr = np.random.default_rng(stable_hash(corpus, split)).integers(4, tok.vocab_size, n_tokens)
    else:
        from datasets import load_dataset
        def stream(ds_, keep=lambda t: True):   # batches read from the memory-mapped dataset, never the whole column
            for b_ in ds_.iter(batch_size=1000):
                for t in b_["text"]:
                    if keep(t):
                        yield t
        if corpus == "wikitext103":
            ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split=split)
            texts = stream(ds, lambda t: bool(t.strip()))
            sep = None
        elif corpus == "tinystories":
            ds = load_dataset("roneneldan/TinyStories", split=split)
            texts = stream(ds); sep = tok.eos_token_id
        else:
            raise ValueError(corpus)
        print(f"[data] tokenising {corpus} ({split}): {n_tokens:,} tokens", flush=True)
        t_tok = time.time()
        out, n = [], 0
        batch = []
        def flush(batch):
            nonlocal n
            try:
                enc = tok(batch, add_special_tokens=False, verbose=False)["input_ids"]
            except TypeError:
                enc = tok(batch, add_special_tokens=False)["input_ids"]
            for ids in enc:
                out.append(np.asarray(ids + ([sep] if sep is not None else []), dtype=np.int64)); n += len(ids)
        for t in texts:
            batch.append(t)
            if len(batch) == 1000:
                flush(batch); batch = []
                if n >= n_tokens:
                    break
        if batch and n < n_tokens:
            flush(batch)
        arr = np.concatenate(out)[:n_tokens]
        print(f"[data] {corpus} ({split}) tokenised in {time.time() - t_tok:.0f} s", flush=True)
    try:
        np.save(path, arr.astype(np.uint16 if arr.max() < 65535 else np.int32))
    except OSError as e:
        print(f"[disk] token cache not written ({e}); continuing in memory")
        if path.exists():
            path.unlink()
    return torch.from_numpy(arr.astype(np.int64))

def _bos(tok):
    return getattr(tok, "bos_token_id", None)

@torch.no_grad()
def lm_eval(model, tok, ids: torch.Tensor, ctx: int = None, batch: int = 8) -> dict:
    """Mean next-token NLL, perplexity and top-1 accuracy over non-overlapping windows."""
    ctx = ctx or CFG["ctx_eval"]
    bos = _bos(tok)
    w = ctx - (1 if bos is not None else 0)
    X = ids[: (len(ids) // w) * w].view(-1, w)
    if bos is not None:
        X = torch.cat([torch.full((X.shape[0], 1), bos), X], 1)
    tot, cnt, cor = 0.0, 0, 0
    dev = model_device(model)
    for i in range(0, X.shape[0], batch):
        x = X[i:i + batch].to(dev)
        logits = model(input_ids=x).logits.float()
        lg, tg = logits[:, :-1], x[:, 1:]
        tot += F.cross_entropy(lg.reshape(-1, lg.shape[-1]), tg.reshape(-1), reduction="sum").item()
        cor += (lg.argmax(-1) == tg).sum().item(); cnt += tg.numel()
    nll = tot / cnt
    return dict(nll=nll, ppl=math.exp(nll), acc=cor / cnt, n_tokens=cnt)

@torch.no_grad()
def mlm_eval(model, tok, ids: torch.Tensor, ctx: int = 128, p: float = 0.15, seed: int = 0, batch: int = 16) -> dict:
    """Masked-token cross-entropy and accuracy with a fixed random mask (all masked tokens -> [MASK])."""
    w = ctx - 2
    X = ids[: (len(ids) // w) * w].view(-1, w)
    X = torch.cat([torch.full((X.shape[0], 1), tok.cls_token_id), X, torch.full((X.shape[0], 1), tok.sep_token_id)], 1)
    g = torch.Generator().manual_seed(seed)
    M = torch.rand(X.shape, generator=g) < p
    M[:, 0] = M[:, -1] = False
    tot, cnt, cor = 0.0, 0, 0
    dev = model_device(model)
    for i in range(0, X.shape[0], batch):
        x, m = X[i:i + batch], M[i:i + batch]
        xin = x.clone(); xin[m] = tok.mask_token_id
        logits = model(input_ids=xin.to(dev)).logits.float().cpu()
        lg, tg = logits[m], x[m]
        tot += F.cross_entropy(lg, tg, reduction="sum").item(); cor += (lg.argmax(-1) == tg).sum().item(); cnt += len(tg)
    return dict(nll=tot / max(cnt, 1), acc=cor / max(cnt, 1), n_masked=cnt)

# ---------------- head-function scores (Section B5) ----------------
@torch.no_grad()
def _attn(model, x):
    out = model(input_ids=x.to(DEVICE), output_attentions=True)
    return [a.float().cpu() for a in out.attentions]

@torch.no_grad()
def head_scores(model, tok, causal: bool = True, seed: int = 0) -> pd.DataFrame:
    """Induction, previous-token, positional (R^2 of the offset-averaged pattern) and sink scores."""
    T, nseq = CFG["head_seq_len"], CFG["head_n_seq"]
    bos = _bos(tok) if causal else tok.cls_token_id
    ids = corpus_tokens(tok, "wikitext103", T * nseq)
    X = ids[: T * nseq].view(nseq, T)
    if bos is not None:
        X = torch.cat([torch.full((nseq, 1), bos), X[:, :-1]], 1)
    acc = None
    for i in range(0, nseq, 4):
        att = _attn(model, X[i:i + 4])
        if acc is None:
            acc = [[] for _ in att]
        for l, a in enumerate(att):
            acc[l].append(a)
    rows = []
    Tn = X.shape[1]
    t_idx = torch.arange(Tn)
    for l in range(len(acc)):
        A = torch.cat(acc[l]); acc[l] = None    # (B, H, T, T); the layer's maps are released after use
        prev = A[:, :, t_idx[2:], t_idx[2:] - 1].mean((0, 2))
        sink = A[:, :, 1:, 0].mean((0, 2))
        off = (t_idx.view(-1, 1) - t_idx.view(1, -1))          # t - s
        valid = (off >= 0) if causal else torch.ones_like(off, dtype=torch.bool)
        valid &= (t_idx.view(-1, 1) >= 8)
        offs = off[valid]; vals = A[:, :, valid]                # (B, H, P)
        uo, inv = torch.unique(offs, return_inverse=True)
        H_ = A.shape[1]
        flat = vals.permute(1, 0, 2).reshape(H_, -1)
        inv_full = inv.repeat(A.shape[0])
        sums = torch.zeros(H_, len(uo)).index_add_(1, inv_full, flat)
        cnts = torch.zeros(len(uo)).index_add_(0, inv_full, torch.ones_like(inv_full, dtype=torch.float))
        means = sums / cnts
        resid = flat - means[:, inv_full]
        r2 = 1 - (resid ** 2).sum(1) / ((flat - flat.mean(1, keepdim=True)) ** 2).sum(1).clamp(min=1e-12)
        for h in range(H_):
            rows.append(dict(layer=l, head=h, prev=float(prev[h]), sink=float(sink[h]), positional=float(r2[h])))
    # induction on repeated random tokens
    Ti = CFG["induction_T"]
    g = torch.Generator().manual_seed(seed)
    lo = 4 if SMOKE else 1000
    r = torch.randint(lo, min(tok.vocab_size, 20000) if not SMOKE else tok.vocab_size, (nseq, Ti), generator=g)
    Xi = torch.cat([r, r], 1)
    off0 = 0
    if bos is not None:
        Xi = torch.cat([torch.full((nseq, 1), bos), Xi], 1); off0 = 1
    ind_acc = None
    for i in range(0, nseq, 4):
        att = _attn(model, Xi[i:i + 4])
        q_pos = torch.arange(Ti - 1) + off0 + Ti
        k_pos = torch.arange(Ti - 1) + off0 + 1
        s = torch.stack([a[:, :, q_pos, k_pos].mean(-1).sum(0) for a in att])   # (L, H)
        ind_acc = s if ind_acc is None else ind_acc + s
    ind = ind_acc / nseq
    for row in rows:
        row["induction"] = float(ind[row["layer"], row["head"]])
    return pd.DataFrame(rows)

HEAD_TYPE_RULE = [("induction", "induction", 0.4), ("previous-token", "prev", 0.4),
                  ("positional", "positional", 0.6), ("sink", "sink", 0.5)]

def head_type(row):
    for name, col, thr in HEAD_TYPE_RULE:
        if row[col] >= thr:
            return name
    return "other"

# ---------------- activation capture (Section B3) ----------------
_PROJ_SUFFIX = {"gpt2": ("h.{i}.attn.c_attn", "fused_gpt2"), "gpt_neox": ("layers.{i}.attention.query_key_value", "fused_neox"),
                "bert": ("encoder.layer.{i}.attention.self.", "bert"), "tinygpt": ("blocks.{i}.attn.", "tinygpt")}

def _find_module(model, suffix):
    for name, mod in model.named_modules():
        if name == suffix or name.endswith("." + suffix):
            return mod
    raise KeyError(suffix)

@torch.no_grad()
def capture_qk(model, model_type: str, n_layers: int, n_q: int, n_kv: int, dh: int, X: torch.Tensor):
    """Per layer: q (N, n_q, dh), k (N, n_kv, dh) pre-rotation (bias included) and positions (N,)."""
    store = {}
    hooks = []
    def mk(i, which):
        def f(mod, inp, out):
            store.setdefault((i, which), []).append(out.detach().float().cpu())
        return f
    for i in range(n_layers):
        if model_type == "gpt2":
            hooks.append(_find_module(model, f"h.{i}.attn.c_attn").register_forward_hook(mk(i, "qkv")))
        elif model_type == "gpt_neox":
            hooks.append(_find_module(model, f"layers.{i}.attention.query_key_value").register_forward_hook(mk(i, "qkv")))
        elif model_type == "bert":
            hooks.append(_find_module(model, f"encoder.layer.{i}.attention.self.query").register_forward_hook(mk(i, "q")))
            hooks.append(_find_module(model, f"encoder.layer.{i}.attention.self.key").register_forward_hook(mk(i, "k")))
        elif model_type == "tinygpt":
            hooks.append(_find_module(model, f"blocks.{i}.attn.q").register_forward_hook(mk(i, "q")))
            hooks.append(_find_module(model, f"blocks.{i}.attn.k").register_forward_hook(mk(i, "k")))
        else:
            hooks.append(_find_module(model, f"layers.{i}.self_attn.q_proj").register_forward_hook(mk(i, "q")))
            hooks.append(_find_module(model, f"layers.{i}.self_attn.k_proj").register_forward_hook(mk(i, "k")))
    try:
        for b in range(0, X.shape[0], 4):
            model(input_ids=X[b:b + 4].to(DEVICE))
    finally:
        for h in hooks:
            h.remove()
    pos = torch.arange(X.shape[1]).repeat(X.shape[0])
    out = []
    for i in range(n_layers):          # store entries are released as soon as they are concatenated
        if (i, "qkv") in store:
            parts = store.pop((i, "qkv"))
            Y = torch.cat(parts).reshape(-1, parts[0].shape[-1]); del parts
            if model_type == "gpt2":
                d = Y.shape[-1] // 3
                q, k = Y[:, :d].reshape(-1, n_q, dh), Y[:, d:2 * d].reshape(-1, n_kv, dh)
            else:
                Y = Y.reshape(-1, n_q, 3, dh); q, k = Y[:, :, 0], Y[:, :, 1]
        else:
            q = torch.cat(store.pop((i, "q"))).reshape(-1, n_q, dh)
            k = torch.cat(store.pop((i, "k"))).reshape(-1, n_kv, dh)
        out.append((q, k))          # float32; converted per head downstream to limit memory
    return out, pos

def apply_rope_acts(x: torch.Tensor, pos: torch.Tensor, inv_freq: torch.Tensor, rot: int) -> torch.Tensor:
    """x (N, h, dh) -> R(pos * omega) x, rotate-half convention."""
    half = rot // 2
    ang = pos.double().view(-1, 1) * inv_freq.double().view(1, -1)      # (N, half)
    c, s = torch.cos(ang).unsqueeze(1), torch.sin(ang).unsqueeze(1)
    a, b = x[..., :half], x[..., half:rot]
    return torch.cat([a * c - b * s, b * c + a * s, x[..., rot:]], -1)

def cca_overlap(A: torch.Tensor, B: torch.Tensor, eps: float = 1e-6) -> float:
    """Mean squared canonical correlation between two centred activation sets (N x k)."""
    A = A - A.mean(0); B = B - B.mean(0)
    def wh(X):
        C = X.mT @ X / len(X)
        C = C + eps * torch.trace(C) / C.shape[0] * torch.eye(C.shape[0], dtype=C.dtype)
        e, V = torch.linalg.eigh(C)
        return X @ V @ torch.diag(e.clamp(min=1e-12) ** -0.5) @ V.mT
    a, b = wh(A), wh(B)
    c = torch.linalg.svdvals(a.mT @ b / len(A)).clamp(0, 1)
    return float((c ** 2).mean())

def pca_basis(X: torch.Tensor, var: float = 0.9):
    X = X - X.mean(0)
    _, S, Vh = torch.linalg.svd(X, full_matrices=False)
    e = torch.cumsum(S ** 2, 0) / (S ** 2).sum()
    r = int((e < var).sum()) + 1
    return Vh[:r].mT

def pca_overlap(A, B, var=0.9):
    UA, UB = pca_basis(A, var), pca_basis(B, var)
    k1, k2 = UA.shape[1], UB.shape[1]
    return float((cosines(UA, UB) ** 2).sum() / min(k1, k2)), k1, k2

# ---------------- Grassmann surgery (Section B4) ----------------
def geodesic_basis(U0: torch.Tensor, U1: torch.Tensor, t: float, E: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Point at fraction t of the Grassmann geodesic from span(U0) to span(U1), expressed in a basis
    that coincides with U0 at t=0. If E is given (orthonormal, orthogonal to U0), the same principal
    angles are applied along the random directions E instead (amplitude-matched control)."""
    A, c, Bh = torch.linalg.svd(U0.mT @ U1)
    c = c.clamp(-1, 1); th = torch.arccos(c)
    P0, P1 = U0 @ A, U1 @ Bh.mT
    if E is None:
        s = torch.sin(th)
        E = (P1 - P0 * c) / torch.where(s > 1e-12, s, torch.ones_like(s))
        E = torch.where(s > 1e-12, E, torch.zeros_like(E))
    Y = P0 * torch.cos(t * th) + E * torch.sin(t * th)
    return Y @ A.mT

def random_complement(U0: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    n, k = U0.shape
    G = torch.randn(n, k, generator=gen, dtype=U0.dtype)
    G = G - U0 @ (U0.mT @ G)
    Q, _ = torch.linalg.qr(G)
    return Q

@torch.no_grad()
def write_k(model, blk: AttnBlock, Wk_new: torch.Tensor):
    """Write per-head K read matrices (n_kv, d, dh) back into the model parameters."""
    params = dict(model.named_parameters())
    lay = blk.keys["layout"]
    if lay == "linear":
        P = params[blk.keys["k"]]
        P.copy_(Wk_new.transpose(1, 2).reshape(P.shape).to(P.dtype))
    elif lay == "gpt2":
        P = params[blk.keys["qkv"]]; d = P.shape[0]
        P[:, d:2 * d] = Wk_new.permute(1, 0, 2).reshape(d, d).to(P.dtype)
    elif lay == "neox":
        P = params[blk.keys["qkv"]]; n, d_, dh = Wk_new.shape
        V = P.view(n, 3, dh, d_); V[:, 1] = Wk_new.transpose(1, 2).to(P.dtype)
    else:
        raise ValueError(lay)

def surgery_targets(blk: AttnBlock, fold=True):
    """Orthonormal K bases U0 (n_kv,d,dh) and their targets U1 (matched Q head, or the dominant
    d_head-dimensional subspace of the group's Q heads under GQA)."""
    Wq, Wk = blk.folded("q", fold), blk.folded("k", fold)
    U0, _, _ = orth(Wk)
    U1 = []
    for gi in range(blk.n_kv):
        members = np.where(blk.kv_of_q == gi)[0]
        Wg = torch.cat([Wq[h] for h in members], -1)
        Ug, _, _ = orth(Wg)
        U1.append(Ug[:, :blk.dh])
    return U0, torch.stack(U1)

def ws_from_model(model, model_id):
    cfg = model.config.to_dict() if hasattr(model.config, "to_dict") else dict(model.config)
    return WeightSource(model_id, state_dict=model.state_dict(), config=cfg)

### 1.8 Small from-scratch Transformers (B3/B4), differentiable overlap, fine-tuning and distillation
`TinyGPT` is a pre-LN decoder whose only varying components are the positional encoding
(`abs` learned absolute, `rope` full rotary, `nope` none) and, optionally, the number of key-value
heads (`n_kv < H` gives grouped-query attention); everything else (width 384, 8 layers, 6 query
heads, GPT-2 tokenizer, tied embeddings, ≈34M parameters) is identical across conditions.

In [ ]:
from types import SimpleNamespace

class TinyAttn(nn.Module):
    def __init__(self, d, H, pe, ctx, rope_base=10000.0, n_kv=None):
        super().__init__()
        self.H, self.dh, self.pe = H, d // H, pe
        self.nkv = n_kv or H
        assert H % self.nkv == 0
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, self.nkv * self.dh, bias=False)
        self.v = nn.Linear(d, self.nkv * self.dh, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        if pe == "rope":
            inv = 1.0 / (rope_base ** (torch.arange(0, self.dh, 2).float() / self.dh))
            ang = torch.arange(ctx).float().view(-1, 1) * inv.view(1, -1)
            self.register_buffer("cos", torch.cos(ang), persistent=False)
            self.register_buffer("sin", torch.sin(ang), persistent=False)

    def _rope(self, x):  # (B, H, T, dh), rotate-half
        T, half = x.shape[2], self.dh // 2
        c, s = self.cos[:T].to(x.dtype), self.sin[:T].to(x.dtype)
        a, b = x[..., :half], x[..., half:]
        return torch.cat([a * c - b * s, b * c + a * s], -1)

    def forward(self, x):
        B, T, d = x.shape
        sh = lambda t, n: t.view(B, T, n, self.dh).transpose(1, 2)
        q, k, v = sh(self.q(x), self.H), sh(self.k(x), self.nkv), sh(self.v(x), self.nkv)
        if self.pe == "rope":
            q, k = self._rope(q), self._rope(k)
        if self.nkv < self.H:      # key-value head j serves query heads j*r ... (j+1)*r-1
            r = self.H // self.nkv
            k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, d))

class TinyBlock(nn.Module):
    def __init__(self, d, H, pe, ctx, n_kv=None):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = TinyAttn(d, H, pe, ctx, n_kv=n_kv)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d, L, H, ctx, pe, n_kv=None):
        super().__init__()
        self.cfg = dict(model_type="tinygpt", vocab=vocab, d=d, L=L, H=H, ctx=ctx, pe=pe, n_kv=n_kv or H)
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ctx, d) if pe == "abs" else None
        self.blocks = nn.ModuleList([TinyBlock(d, H, pe, ctx, n_kv) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        self.apply(self._init)
        for n_, p in self.named_parameters():
            if n_.endswith("attn.o.weight") or n_.endswith("mlp.2.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * L))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)
    def forward(self, input_ids, **kw):
        x = self.tok(input_ids)
        if self.pos is not None:
            x = x + self.pos(torch.arange(input_ids.shape[1], device=input_ids.device))
        for b in self.blocks:
            x = b(x)
        logits = self.lnf(x) @ self.tok.weight.T
        # a plain namespace: an object of a class created on the fly sits in a reference cycle (the class refers to
        # itself), so it and its logits were freed only by the cyclic garbage collector, and successive batches of
        # 50k-way logits accumulated on the GPU until it ran out of memory
        return SimpleNamespace(logits=logits)

def tinygpt_ws(model, name):
    sd = {k: v.detach().float().cpu() for k, v in model.state_dict().items()}
    return WeightSource(name, state_dict=sd, config=dict(model.cfg))

def diff_overlap(Wq_lin: torch.Tensor, Wk_lin: torch.Tensor, gain: Optional[torch.Tensor], H: int,
                 kv_of_q=None) -> torch.Tensor:
    """Differentiable per-head Q-K overlap (mean cos^2) from nn.Linear weights (H*dh, d); gain folded."""
    d = Wq_lin.shape[1]; dh = Wq_lin.shape[0] // H; nkv = Wk_lin.shape[0] // dh
    Wq = Wq_lin.float().view(H, dh, d).transpose(1, 2)
    Wk = Wk_lin.float().view(nkv, dh, d).transpose(1, 2)
    if gain is not None:
        Wq = gain.float().view(1, -1, 1) * Wq; Wk = gain.float().view(1, -1, 1) * Wk
    Uq, _ = torch.linalg.qr(Wq); Uk, _ = torch.linalg.qr(Wk)
    if kv_of_q is None and nkv < H:
        kv_of_q = np.arange(H) // (H // nkv)
    if kv_of_q is not None:
        Uk = Uk[torch.as_tensor(kv_of_q)]
    return ((Uq.mT @ Uk) ** 2).sum((-1, -2)) / dh          # (H,)

def tinystories_tokens():
    """GPT-2-tokenised TinyStories (train / validation), cached."""
    sc = CFG["scratch"]
    tok = SmokeTok() if SMOKE else AutoTokenizer.from_pretrained("openai-community/gpt2", cache_dir=str(HF_CACHE))
    tr = corpus_tokens(tok, "tinystories", sc["train_tokens"], split="train")
    va = corpus_tokens(tok, "tinystories", sc["val_tokens"], split="validation")
    return tok, tr, va

@torch.no_grad()
def eval_loss(model, val, ctx, batch=32, max_batches=50):
    model.eval()
    X = val[: (len(val) // (ctx + 1)) * (ctx + 1)].view(-1, ctx + 1)
    tot, n = 0.0, 0
    dev = model_device(model)
    for i in range(0, min(len(X), batch * max_batches), batch):
        x = X[i:i + batch].to(dev)
        with amp_ctx():
            lg = model(x[:, :-1]).logits
        tot += F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1), reduction="sum").item()
        n += x[:, 1:].numel()
    model.train()
    return tot / n

def mean_layer_overlap(model):
    vals = []
    for b in model.blocks:
        vals.append(diff_overlap(b.attn.q.weight, b.attn.k.weight, b.ln1.weight, b.attn.H))
    return torch.stack(vals)          # (L, H)

def scratch_cond(pe: str, n_kv: Optional[int] = None) -> str:
    """Condition label: 'abs', 'rope', 'nope' (MHA) or e.g. 'rope-gqa' (n_kv < H)."""
    return pe if not n_kv or n_kv >= CFG["scratch"]["H"] else f"{pe}-gqa"

def train_tinygpt(pe: str, seed: int, lam: float = 0.0, tag: str = "", tokens=None, n_kv: Optional[int] = None,
                  device=None) -> dict:
    """Train one TinyGPT; lam>0 penalises and lam<0 rewards Q-K overlap (mean over heads);
    n_kv < H gives grouped-query attention with otherwise identical architecture. `device` is the GPU
    used for training (run_parallel); the returned model is on the CPU (trained models are kept for later
    cells and would otherwise accumulate in GPU memory)."""
    sc = CFG["scratch"]
    gqa = f"-gqa{n_kv}" if n_kv and n_kv < sc["H"] else ""
    name = f"tinygpt-{pe}{gqa}-s{seed}-lam{lam:+.3f}{tag}"
    ck = CACHE / "scratch" / f"{name}.pt"
    tok, tr, va = tokens if tokens is not None else tinystories_tokens()
    vocab = tok.vocab_size if SMOKE else len(tok)
    dev = torch.device(device) if device is not None else MAIN_DEVICE
    with _INIT_LOCK:                     # initialisation identical to a sequential run with the same seed
        seed_local(SEED + seed, dev)
        model = TinyGPT(vocab, sc["d"], sc["L"], sc["H"], sc["ctx"], pe, n_kv=n_kv)
    model = model.to(dev)
    log = []
    if ck.exists():
        obj = torch.load(ck, map_location=dev, weights_only=False)
        model.load_state_dict(obj["state"]); log = obj["log"]
        return dict(model=model.cpu(), log=pd.DataFrame(log), name=name, val_loss=obj["val_loss"])
    t_start = time.time()
    opt = torch.optim.AdamW(model.parameters(), lr=sc["lr"], betas=(0.9, 0.95), weight_decay=0.1)
    sched = lambda s: min(1.0, (s + 1) / sc["warmup"]) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / sc["steps"]))))
    g = torch.Generator().manual_seed(SEED + 1000 + seed)
    scaler = make_scaler()
    nwin = len(tr) - sc["ctx"] - 1
    # intermediate checkpoint (model, optimiser, loss scaler, data-sampler state): a run cut by the end of a
    # session resumes where it stopped and continues exactly as an uninterrupted run would
    part = ck.with_name(ck.stem + ".part.pt")
    start = 0
    if time_left() < 0 and not part.exists():          # no time left in this session: do not start the run
        check_budget(f"({name} not started)")
    if part.exists():
        # loaded on the CPU: the data-sampler state must stay a CPU ByteTensor (Generator.set_state); model and
        # optimiser states are copied to the training device by load_state_dict
        obj = torch.load(part, map_location="cpu", weights_only=False)
        model.load_state_dict(obj["state"]); opt.load_state_dict(obj["opt"]); scaler.load_state_dict(obj["scaler"])
        g.set_state(obj["g"]); log = obj["log"]; start = obj["step"]
        print(f"[train] {name} on {dev}: resuming at step {start}/{sc['steps']}", flush=True)
    else:
        print(f"[train] {name} on {dev}", flush=True)
    prog = Progress(name, sc["steps"], every=600, start=start)
    model.train()
    for step in range(start, sc["steps"] + 1):
        if time_left() < 0 and step < sc["steps"]:      # end of the session: save the state, continue next session
            part.parent.mkdir(exist_ok=True, parents=True)
            torch.save(dict(state=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), g=g.get_state(),
                            log=log, step=step), str(part) + ".tmp")
            os.replace(str(part) + ".tmp", part)
            print(f"[train] {name}: time budget reached at step {step}; state saved", flush=True)
            check_budget(f"({name} saved at step {step})")
        if step % sc["log_every"] == 0 or step == sc["steps"]:
            with torch.no_grad():
                ov = mean_layer_overlap(model).cpu().numpy()
            n_ = model.cfg["d"]; dh = n_ // model.cfg["H"]; mu0 = dh / n_
            log.append(dict(step=step, val_loss=eval_loss(model, va, sc["ctx"]), overlap=float(ov.mean()),
                            rho=float((ov.mean() - mu0) / (1 - mu0))))
        if step == sc["steps"]:
            break
        if STOP_EVENT.is_set():
            raise KeyboardInterrupt(f"{name} stopped")
        for gr in opt.param_groups:
            gr["lr"] = sc["lr"] * sched(step)
        idx = torch.randint(0, nwin, (sc["batch"],), generator=g)
        x = torch.stack([tr[i:i + sc["ctx"] + 1] for i in idx.tolist()]).to(dev)
        # one optimiser step on the whole batch; its gradient is accumulated over equal micro-batches (mean loss of
        # each micro-batch / number of micro-batches = mean loss of the batch), which bounds the memory taken by
        # the 50k-way output layer; the overlap term depends on the weights only and is added once
        mb = min(sc.get("micro", sc["batch"]), sc["batch"]); n_mb = sc["batch"] // mb
        assert n_mb * mb == sc["batch"]
        opt.zero_grad(set_to_none=True)
        for k in range(n_mb):
            xb = x[k * mb:(k + 1) * mb]
            with amp_ctx():
                lg = model(xb[:, :-1]).logits
            loss = F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), xb[:, 1:].reshape(-1)) / n_mb
            if lam != 0.0 and k == 0:
                loss = loss + lam * mean_layer_overlap(model).mean()
            scaler.scale(loss).backward()
            del lg, loss
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        prog(step + 1, f"(last val. loss {log[-1]['val_loss']:.3f})")
        if (step + 1) % sc.get("ckpt_every", 1000) == 0 and step + 1 < sc["steps"]:
            part.parent.mkdir(exist_ok=True, parents=True)
            torch.save(dict(state=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), g=g.get_state(),
                            log=log, step=step + 1), str(part) + ".tmp")
            os.replace(str(part) + ".tmp", part)          # atomic: an interruption never leaves a corrupt file
    ck.parent.mkdir(exist_ok=True, parents=True)
    torch.save(dict(state=model.state_dict(), log=log, val_loss=log[-1]["val_loss"]), ck)
    if part.exists():
        part.unlink()
    print(f"[train] {name} done in {time.time() - t_start:.0f} s (val. loss {log[-1]['val_loss']:.3f})", flush=True)
    del opt, scaler
    return dict(model=model.cpu(), log=pd.DataFrame(log), name=name, val_loss=log[-1]["val_loss"])

# ---------------- SST-2 fine-tuning (B7 controlled drift, B8 diagnostic) ----------------
def sst2_data(tok, max_len=64):
    if SMOKE:
        g = torch.Generator().manual_seed(0)
        mk = lambda n: (torch.randint(4, tok.vocab_size, (n, 16), generator=g), torch.randint(0, 2, (n,), generator=g))
        return mk(CFG["glue_max_train"]), mk(CFG["glue_max_eval"])
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", "sst2")
    def enc(split, cap):
        d = ds[split] if cap is None else ds[split].select(range(min(cap, len(ds[split]))))
        texts = [str(t) for t in d["sentence"]]          # datasets >= 4 returns a lazy Column, not a list
        e = tok(texts, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        return (e["input_ids"], e["attention_mask"]), torch.tensor([int(y) for y in d["label"]])
    return enc("train", CFG["glue_max_train"]), enc("validation", CFG["glue_max_eval"])

def finetune_sst2(model_id: str, seed: int = 0, lr: Optional[float] = None, bs: int = 32, device=None):
    """Fine-tune a BERT-style encoder on SST-2; returns (model on `device`, dev accuracy)."""
    dev = torch.device(device) if device is not None else MAIN_DEVICE
    with _INIT_LOCK:                     # seeding + classifier initialisation, one job at a time
        seed_local(SEED + seed, dev)
        model, tok = load_model(model_id, task="cls", dtype=torch.float32, device="cpu")
    model = model.to(dev)
    (Xtr, ytr), (Xva, yva) = sst2_data(tok)
    if isinstance(Xtr, tuple):
        (itr, mtr), (iva, mva) = Xtr, Xva
    else:
        itr, mtr, iva, mva = Xtr, torch.ones_like(Xtr), Xva, torch.ones_like(Xva)
    H = model.config.hidden_size
    lr = lr or (1e-4 if H <= 256 else 5e-5)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = CFG["glue_epochs"] * math.ceil(len(ytr) / bs)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, steps // 10)) * max(0.0, 1 - s / steps))
    g = torch.Generator().manual_seed(seed)
    scaler = make_scaler()
    model.train()
    prog = Progress(f"SST-2 {model_id}", steps, every=300)
    done = 0
    for ep in range(CFG["glue_epochs"]):
        perm = torch.randperm(len(ytr), generator=g)
        for i in range(0, len(ytr), bs):
            if STOP_EVENT.is_set():
                raise KeyboardInterrupt(f"{model_id} fine-tuning stopped")
            check_budget(f"(fine-tuning {model_id})")
            j = perm[i:i + bs]
            with amp_ctx():
                out = model(input_ids=itr[j].to(dev), attention_mask=mtr[j].to(dev), labels=ytr[j].to(dev))
            amp_step(out.loss.float(), opt, model.parameters(), scaler)
            sch.step()
            done += 1; prog(done)
    model.eval(); cor = 0
    with torch.no_grad():
        for i in range(0, len(yva), 128):
            lg = model(input_ids=iva[i:i + 128].to(dev), attention_mask=mva[i:i + 128].to(dev)).logits
            cor += (lg.argmax(-1).cpu() == yva[i:i + 128]).sum().item()
    return model, cor / len(yva)

### 1.9 Model-level summaries and shared plotting helpers

In [ ]:
REGIME_ORDER = ["abs-MHA", "relbias-MHA", "partial-RoPE", "RoPE-MHA", "RoPE-GQA"]

def regime_of(info_or_mid) -> str:
    meta = REG.get(info_or_mid if isinstance(info_or_mid, str) else info_or_mid["model"], {})
    pe, attn = meta.get("pe"), meta.get("attn", "MHA")
    if pe == "abs":
        return "abs-MHA"
    if pe == "relbias":
        return "relbias-MHA"
    if pe == "prope":
        return "partial-RoPE"
    if pe == "rope":
        return "RoPE-GQA" if attn == "GQA" else "RoPE-MHA"
    return "other"

REGIME_LABEL = {"abs-MHA": "absolute-PE MHA", "relbias-MHA": "relative-bias", "partial-RoPE": "partial-RoPE",
                "RoPE-MHA": "RoPE MHA", "RoPE-GQA": "RoPE GQA", "other": "other"}
_PRETTY = {"gpt2": "GPT-2", "bert-tiny": "BERT-tiny", "phi-2": "Phi-2", "t5-base": "T5-base", "bart-base": "BART-base",
           "mobilebert": "MobileBERT", "OLMoE-1B-7B-0924": "OLMoE-1B-7B", "Mistral-7B-v0.1": "Mistral-7B",
           "Mistral-7B-Instruct-v0.1": "Mistral-7B-Instruct", "TinyBERT_General_4L_312D": "TinyBERT-4L-312",
           "TinyBERT_General_6L_768D": "TinyBERT-6L-768", "BiomedNLP-BiomedBERT-base-abstract": "BiomedBERT",
           "biobert-base-cased-v1.2": "BioBERT", "mistral-7b-sft-beta": "Mistral-7B-SFT-beta", "zephyr-7b-beta": "Zephyr-7B-beta"}

FAMILY_GROUP = {"MultiBERTs": "BERT", "BERT-miniature": "BERT", "BERT": "BERT"}

def family_of(mid: str) -> str:
    """Independent pre-training lineage used as the statistical unit for family-level inference
    (MultiBERTs and the BERT miniatures share BERT's data and recipe and are grouped with it)."""
    fam = REG.get(mid, {}).get("family", mid)
    return FAMILY_GROUP.get(fam, fam)

def short(mid: str) -> str:
    s = mid.split("/")[-1]
    s = (s.replace("-intermediate-step-1431k-3T", "").replace("bert_uncased_", "BERT ").replace("-uncased", "")
          .replace("distilbert-base", "DistilBERT").replace("bert-base", "BERT-base"))
    if s in _PRETTY:
        return _PRETTY[s]
    m = re.match(r"pythia-([\d.]+)([mb])(.*)", s)
    if m:
        return f"Pythia-{m.group(1)}{m.group(2).upper()}{m.group(3)}"
    return s

def summarize(df: pd.DataFrame, value: str = "rho", kinds=None, pairs=PAIRS, seed: int = 0) -> pd.DataFrame:
    """Model x kind x pair means with hierarchical-bootstrap 95% CIs (clusters = layers)."""
    rows = []
    for (m, rev, kind), g in df.groupby(["model", "revision", "kind"], dropna=False):
        if kinds is not None and kind not in kinds:
            continue
        for p in pairs:
            gp = g[g.pair == p]
            if gp.empty:
                continue
            est, lo, hi = hier_boot(gp, value, seed=seed + stable_hash(m, p) % 1000)
            row = dict(model=m, revision=rev, kind=kind, pair=p, mean=est, lo=lo, hi=hi, n_heads=len(gp),
                       overlap=gp.overlap.mean(), null=gp.null.mean(), cross_space=bool(gp.cross_space.any()))
            if "coupling" in gp and gp["coupling"].notna().any():
                c, clo, chi = hier_boot(gp, "coupling", seed=seed + 7)
                row.update(coupling=c, coupling_lo=clo, coupling_hi=chi, mis_rho=gp.mis_rho.mean())
            if p == "QK" and "s_align" in gp:
                row["s_align"] = gp.s_align.mean()
            rows.append(row)
    return pd.DataFrame(rows)

def hierarchy_tests(df: pd.DataFrame, kind="self"):
    """Per model: paired (per query head) differences rho_QK - rho_KV and rho_KV - rho_QV, tested with a
    t-test on layer means (layers are the independent units; sign-flip p-values are kept as a check).
    Returns one row per model with raw p-values; Holm is applied across models and the two contrasts."""
    out = []
    for (m, rev), g in df[df.kind == kind].groupby(["model", "revision"], dropna=False):
        qk = g[g.pair == "QK"].set_index(["layer", "head"])["rho"]
        qv = g[g.pair == "QV"].set_index(["layer", "head"])["rho"]
        kvt = g[g.pair == "KV"].set_index(["layer", "head"])["rho"]
        part = g[g.pair == "QK"].set_index(["layer", "head"])["partner"]
        kv = pd.Series([kvt.get((l, int(p)), np.nan) for (l, h), p in part.items()], index=part.index)
        d = pd.DataFrame({"d1": qk - kv, "d2": kv - qv}).reset_index()
        out.append(dict(model=m, revision=rev, d1=d.d1.mean(), d2=d.d2.mean(), n_layers=d.layer.nunique(),
                        p1=cluster_ttest(d, "d1"), p2=cluster_ttest(d, "d2"),
                        p1_signflip=signflip_test(d, "d1", seed=stable_hash(m, 1) % 997),
                        p2_signflip=signflip_test(d, "d2", seed=stable_hash(m, 2) % 997)))
    t = pd.DataFrame(out)
    if len(t):
        adj = holm(np.r_[t.p1.values, t.p2.values])
        t["p1_holm"], t["p2_holm"] = adj[:len(t)], adj[len(t):]
        def lab(r):
            s1 = "QK>KV" if (r.d1 > 0 and r.p1_holm < 0.05) else ("QK<KV" if (r.d1 < 0 and r.p1_holm < 0.05) else "QK~KV")
            s2 = "KV>QV" if (r.d2 > 0 and r.p2_holm < 0.05) else ("KV<QV" if (r.d2 < 0 and r.p2_holm < 0.05) else "KV~QV")
            return f"{s1}, {s2}"
        t["ordering"] = t.apply(lab, axis=1)
        t["full_hierarchy"] = t.ordering == "QK>KV, KV>QV"
        t["violated"] = t.ordering.str.contains("<")
    return t

def bar_ci(ax, summ: pd.DataFrame, models, pairs=PAIRS, value="mean", title="", ylabel=r"normalised excess $\rho$"):
    w = 0.8 / len(pairs)
    for j, p in enumerate(pairs):
        s = summ[summ.pair == p].set_index("model").reindex(models)
        x = np.arange(len(models)) + (j - (len(pairs) - 1) / 2) * w
        ax.bar(x, s[value], w, color=PAIR_COLORS.get(p, "0.5"), label=p.replace("QK", "Q–K").replace("KV", "K–V").replace("QV", "Q–V"))
        ax.errorbar(x, s[value], yerr=[s[value] - s["lo"], s["hi"] - s[value]], fmt="none", ecolor="k", lw=0.7, capsize=1.5)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xticks(np.arange(len(models))); ax.set_xticklabels([short(m) for m in models], rotation=35, ha="right", fontsize=7)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(frameon=False, ncol=3, fontsize=7)

def label_column(ax, xs, ys, labels, x_text, y_lo, y_hi, fontsize=6, merge_tol=(2e-3, 4e-3)):
    """Labels in one column at x = x_text, in the vertical order of their points and evenly spaced between
    y_lo and y_hi, joined to the points by thin leader lines; points that coincide share one label."""
    pts = sorted(zip(xs, ys, labels), key=lambda t: t[1])
    groups = []
    for x, y, l in pts:
        if groups and abs(groups[-1][0] - x) < merge_tol[0] and abs(groups[-1][1] - y) < merge_tol[1]:
            groups[-1][2].append(l)
        else:
            groups.append([x, y, [l]])
    def joined(ls):
        ls = sorted(ls, key=len)
        if len(ls) == 2 and ls[1].startswith(ls[0] + "-"):          # "X" and "X-Instruct" -> "X (+Instruct)"
            return f"{ls[0]} (+{ls[1][len(ls[0]) + 1:]})"
        return ", ".join(ls)
    ty = np.linspace(y_lo, y_hi, len(groups)) if len(groups) > 1 else [y_lo]
    for t, (x, y, ls) in zip(ty, groups):
        ax.annotate(joined(ls), (x, y), xytext=(x_text, t), textcoords="data", fontsize=fontsize, va="center",
                    arrowprops=dict(arrowstyle="-", lw=0.3, color="0.6", shrinkA=1, shrinkB=2, relpos=(0, 0.5)))

def plot_rq1_orig(S0, S0raw, models):
    """Figure fig_rq1_orig: (a) calibrated excess with CIs; (b) raw Q-K overlap against the chance level."""
    from matplotlib.lines import Line2D
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.0), gridspec_kw=dict(width_ratios=[1.45, 1]), layout="constrained")
    bar_ci(ax[0], S0, models, title="(a) null-calibrated excess (95% CI)")
    sr = S0raw[S0raw.pair == "QK"].set_index("model").reindex(models)["mean"]
    nulls = S0[S0.pair == "QK"].set_index("model").reindex(models)["null"]
    regs = [regime_of(m) for m in models]
    cmap = plt.get_cmap("tab10")
    ax[1].scatter(nulls, sr, c=[cmap(REGIME_ORDER.index(r_)) for r_ in regs], s=16, zorder=3)
    xx = np.linspace(0, 0.55, 10)
    ax[1].plot(xx, xx, color="0.4", lw=0.8, ls="--")
    lo_m = [m for m in models if nulls[m] < 0.3]; hi_m = [m for m in models if nulls[m] >= 0.3]
    top = max(0.78, float(sr.max()) + 0.05)
    label_column(ax[1], nulls[lo_m].values, sr[lo_m].values, [short(m) for m in lo_m], x_text=0.10,
                 y_lo=0.34 * top / 0.78, y_hi=0.72 * top / 0.78, fontsize=5.5)
    for m in hi_m:
        ax[1].annotate(short(m), (nulls[m], sr[m]), xytext=(-6, 0), textcoords="offset points", ha="right", va="center", fontsize=5.5)
    ax[1].set_xlim(0, 0.56); ax[1].set_ylim(0, top)
    handles = [Line2D([], [], ls="--", color="0.4", lw=0.8, label="chance ($\\mu_0$)")] + \
              [Line2D([], [], ls="", marker="o", ms=4, color=cmap(REGIME_ORDER.index(r_)), label=REGIME_LABEL[r_])
               for r_ in REGIME_ORDER if r_ in regs]
    ax[1].legend(handles=handles, frameon=False, fontsize=5.5, loc="lower right")
    ax[1].set_xlabel(r"analytic null $\mu_0=d_h/d$"); ax[1].set_ylabel("raw Q–K overlap")
    ax[1].set_title("(b) raw overlap vs. its chance level")
    return fig

def plot_seed_floor(allseeds, unstable=None):
    """Figure fig_seed_floor: model-level statistics of every pre-training seed, by family. `unstable` maps a
    family name to model ids drawn with open markers (runs documented as unstable)."""
    unstable = unstable or {}
    fig, axs = plt.subplots(1, 4, figsize=(7.4, 2.5), layout="constrained")
    fams = list(dict.fromkeys(allseeds.family))
    cmap = plt.get_cmap("tab10")
    for a_, k in zip(axs, ["rho_QK", "rho_KV", "rho_QV", "coupling_QK"]):
        for i, f in enumerate(fams):
            g = allseeds[allseeds.family == f]
            x = np.full(len(g), i) + np.random.default_rng(i).uniform(-0.15, 0.15, len(g))
            bad = g.model.isin(unstable.get(f, [])).to_numpy()
            a_.scatter(x[~bad], g[k].to_numpy()[~bad], s=6, alpha=0.75, color=cmap(i))
            if bad.any():
                a_.scatter(x[bad], g[k].to_numpy()[bad], s=14, facecolors="none", edgecolors=cmap(i), lw=0.8)
        a_.set_xticks(range(len(fams))); a_.set_xticklabels(fams, rotation=45, ha="right", fontsize=6)
        a_.tick_params(axis="y", labelsize=6.5)
        a_.set_title({"rho_QK": r"$\rho_{QK}$", "rho_KV": r"$\rho_{KV}$", "rho_QV": r"$\rho_{QV}$", "coupling_QK": "Q–K coupling"}[k])
    if any(unstable.values()):
        axs[0].scatter([], [], s=14, facecolors="none", edgecolors="0.3", lw=0.8, label="documented unstable run")
        axs[0].legend(frameon=False, fontsize=5.5, loc="lower left")
    return fig

def head_heatmap(ax, df_model: pd.DataFrame, pair="QK", value="rho", kind="self", title="", vmin=None, vmax=None, cmap="viridis"):
    g = df_model[(df_model.pair == pair) & (df_model.kind == kind)]
    M = g.pivot(index="layer", columns="head", values=value)
    im = ax.imshow(M.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True)); ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("head"); ax.set_ylabel("layer"); ax.set_title(title, fontsize=8)
    return im

In [ ]:
GEO = {}     # cache of per-model DataFrames used across sections
INFO = {}

def geo(mid, revision=None, fold=True, both_folds=False):
    """Geometry of one checkpoint (memory cache over the disk cache of model_geometry)."""
    key = (mid, revision, fold)
    if key not in GEO:
        d_, i_ = model_geometry(mid, revision, fold, both_folds=both_folds)
        GEO[key], INFO[key] = d_, i_
    return GEO[key], INFO[key]

def geo_many(mids, fold=True, both_folds=False):
    dfs, infos = [], []
    for i, m in enumerate(mids):
        if len(mids) > 3 and (m, None, fold) not in GEO:
            print(f"[{i + 1}/{len(mids)}] {m}", flush=True)
        d_, i_ = geo(m, None, fold, both_folds)
        if d_ is not None:
            dfs.append(d_); infos.append(i_)
    return (pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()), pd.DataFrame(infos)


def hier_sentence(H):
    parts = []
    for reg_ in REGIME_ORDER:
        h = H[H.regime == reg_]
        if len(h) == 0:
            continue
        parts.append(f"{int(h.full_hierarchy.sum())} of {len(h)} {REGIME_LABEL[reg_]} model{'s' if len(h) > 1 else ''}")
    viol = H[H.violated]
    s = "the complete ordering $\\rho_{QK}>\\rho_{KV}>\\rho_{QV}$ is significant (Holm-adjusted $p<0.05$) in " + "; ".join(parts)
    if len(viol):
        s += ", and at least one inequality is significantly reversed in " + ", ".join(short(m) for m in viol.model)
    else:
        s += ", and no inequality is significantly reversed in any model"
    return s

FLOOR = {}

def model_stats(df):
    s = summarize(df[df.kind.isin(["self", "enc-self"])], pairs=PAIRS)
    out = {f"rho_{p}": float(s[s.pair == p]["mean"].mean()) for p in PAIRS}
    c = s[s.pair == "QK"]
    out["coupling_QK"] = float(c["coupling"].mean()) if "coupling" in c else np.nan
    return out

def seed_family(mids, label):
    rows = []
    for m in mids:
        d_, i_ = geo(m)
        if d_ is None:
            continue
        rows.append(dict(model=m, family=label, **model_stats(d_)))
    t = pd.DataFrame(rows)
    fl = {k: seed_floor(t[k]) for k in ["rho_QK", "rho_KV", "rho_QV", "coupling_QK"]} if len(t) else {}
    return t, fl


def floor_for(mid: str, stat: str = "rho_QK", variant: str = "primary"):
    """Seed floor used for a checkpoint: MultiBERTs for BERT-architecture encoders, otherwise the
    PolyPythias size closest in parameter count (an explicit, stated approximation).
    variant="primary": for Pythia-410M, the floor of the stable runs (B1.3; seeds 3 and 4 are documented
    unstable runs); variant="all": floors over all seeds, as pre-specified (sensitivity analysis)."""
    if not FLOOR:
        p = OUT / "floor.json"
        if p.exists():
            FLOOR.update(json.loads(p.read_text()))
    meta = REG.get(mid, {})
    if meta.get("arch") in ("bert", "distilbert", "mobilebert") and "BERT" in FLOOR:
        return FLOOR["BERT"][stat]["q95"], "MultiBERTs"
    sizes = {"70m": 7.0e7, "160m": 1.6e8, "410m": 4.1e8}
    _, i_ = geo(mid)
    P = i_["params"] if i_ is not None else 1e9
    best = min((k for k in sizes if f"Pythia-{k}" in FLOOR), key=lambda k: abs(np.log(P / sizes[k])), default=None)
    if best is None:
        return np.nan, "none"
    key = f"Pythia-{best}"
    if variant == "primary" and f"{key}-stable" in FLOOR:
        return FLOOR[f"{key}-stable"][stat]["q95"], f"PolyPythias-{best} (stable runs)"
    return FLOOR[key][stat]["q95"], f"PolyPythias-{best}"

def model_level(mids, revision=None):
    rows = []
    for m in mids:
        d_, i_ = geo(m, revision)
        if d_ is None:
            continue
        s = summarize(d_[d_.kind == "self"])
        r = dict(model=m, revision=revision, params=i_["params"], params_nonemb=i_["params_nonemb"], d_model=i_["d_model"],
                 d_head=i_["d_head"], n_layers=i_["n_layers"], L=REG.get(m, {}).get("L"), H=REG.get(m, {}).get("H"))
        for p in PAIRS:
            q = s[s.pair == p]
            r[f"rho_{p}"], r[f"lo_{p}"], r[f"hi_{p}"] = q["mean"].iloc[0], q["lo"].iloc[0], q["hi"].iloc[0]
            r[f"raw_{p}"] = q["overlap"].iloc[0]
        r["null"] = s.null.iloc[0]
        rows.append(r)
    return pd.DataFrame(rows)

## A. Model-free theory checks (Propositions 1–3, Table 2, Boxes 1–4)
These cells use only random matrices: they verify the propositions numerically and draw the toy
figures of the intuitive boxes. They do not depend on any checkpoint.

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.1")
        rng = np.random.default_rng(SEED)
        def rand_weight(n, k, cond=30.0, rng=rng):
            U, _ = np.linalg.qr(rng.standard_normal((n, k))); V, _ = np.linalg.qr(rng.standard_normal((k, k)))
            s = np.exp(np.linspace(0, -np.log(cond), k))
            return torch.tensor(U @ np.diag(s) @ V.T)
        def rand_gl(k, cond=100.0, rng=rng):
            U, _ = np.linalg.qr(rng.standard_normal((k, k))); V, _ = np.linalg.qr(rng.standard_normal((k, k)))
            return torch.tensor(U @ np.diag(np.exp(np.linspace(0, -np.log(cond), k))) @ V.T)
        def rand_o(n, rng=rng):
            Q, _ = np.linalg.qr(rng.standard_normal((n, n))); return torch.tensor(Q)

        n, k = 64, 8
        dev_ov, dev_M, ch_raw, ch_cka, ch_diag = [], [], [], [], []
        for _ in range(200):
            Wq, Wk = rand_weight(n, k), rand_weight(n, k)
            A = rand_gl(k)
            Wq2, Wk2 = Wq @ A, Wk @ torch.linalg.inv(A).mT
            dev_M.append(float((Wq @ Wk.mT - Wq2 @ Wk2.mT).abs().max() / (Wq @ Wk.mT).abs().max()))
            dev_ov.append(abs(sim_overlap(Wq, Wk) - sim_overlap(Wq2, Wk2)))
            ch_raw.append(abs(sim_rawcos(Wq, Wk) - sim_rawcos(Wq2, Wk2)))
            ch_cka.append(abs(sim_cka(Wq, Wk) - sim_cka(Wq2, Wk2)))
            D = torch.diag(torch.tensor(np.exp(rng.normal(0, 0.5, n))))
            ch_diag.append(abs(sim_overlap(Wq, Wk) - sim_overlap(D @ Wq, D @ Wk)))
        R.set("a1.maxdev-m", fmt_sci(max(dev_M)))
        R.set("a1.maxdev-overlap", fmt_sci(max(dev_ov)))
        R.set("a1.rawcos-change", float(np.median(ch_raw)), "{:.3f}")
        R.set("a1.cka-change", float(np.median(ch_cka)), "{:.3f}")
        R.set("a1.diag-change", float(np.median(ch_diag)), "{:.3f}")
        R.set("a1.n-draws", 200, "{:d}")
        print(dict(dev_M=max(dev_M), dev_ov=max(dev_ov), raw=np.median(ch_raw), cka=np.median(ch_cka), diag=np.median(ch_diag)))

### A.2 Invariance table (Table 2): every entry is checked numerically
Transformations: T1 independent right-orthogonal maps; T2 independent right-$GL(k)$ maps;
T3 joint Q–K gauge $(W_QA,W_KA^{-\top})$; T4 common left-orthogonal map (residual rotation);
T5 common positive diagonal map (normalisation gain); T6 positive isotropic scaling.
An entry is "invariant" if the largest change over 50 random draws is below $10^{-8}$.

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.2")
        TRANSFORMS = ["T1 right O(k)", "T2 right GL(k)", "T3 Q-K gauge", "T4 left O(n)", "T5 left diag", "T6 scale"]
        def apply_T(t, X, Y):
            k1, k2 = X.shape[1], Y.shape[1]; nn_ = X.shape[0]
            if t == 0: return X @ rand_o(k1), Y @ rand_o(k2)
            if t == 1: return X @ rand_gl(k1), Y @ rand_gl(k2)
            if t == 2:
                A = rand_gl(k1); return X @ A, Y @ torch.linalg.inv(A).mT
            if t == 3:
                U = rand_o(nn_); return U @ X, U @ Y
            if t == 4:
                D = torch.diag(torch.tensor(np.exp(rng.normal(0, 0.7, nn_)))); return D @ X, D @ Y
            if t == 5:
                return X * float(np.exp(rng.normal())), Y * float(np.exp(rng.normal()))
        inv_rows = []
        for mname, f in METRICS.items():
            row = {"Metric": mname}
            for t, tname in enumerate(TRANSFORMS):
                dev = 0.0
                for _ in range(50):
                    X, Y = rand_weight(48, 6, cond=1e3), rand_weight(48, 6, cond=1e3)
                    X2, Y2 = apply_T(t, X, Y)
                    a, b = f(X, Y), f(X2, Y2)
                    dev = max(dev, abs(a - b) / max(1.0, abs(a)))
                row[tname] = dev
            inv_rows.append(row)
        inv = pd.DataFrame(inv_rows)
        EXPECTED = {  # analytical derivation reported in the paper (1 = invariant)
            "overlap": [1, 1, 1, 1, 0, 1], "geodesic": [1, 1, 1, 1, 0, 1], "chordal": [1, 1, 1, 1, 0, 1],
            "projF": [1, 1, 1, 1, 0, 1], "CKA": [1, 0, 0, 1, 0, 1], "SVCCA": [1, 0, 0, 1, 0, 1],
            "PWCCA": [0, 0, 0, 1, 0, 1], "Procrustes": [1, 0, 0, 1, 0, 1], "raw cosine": [0, 0, 0, 1, 0, 1],
            "self-alignment s": [0, 0, 1, 1, 0, 1]}
        measured = {r["Metric"]: [int(r[t] < 1e-8) for t in TRANSFORMS] for r in inv_rows}
        mism = [(m, TRANSFORMS[j]) for m in EXPECTED for j in range(6) if EXPECTED[m][j] != measured[m][j]]
        tab = inv.copy()
        for t in TRANSFORMS:
            tab[t] = [r"\cmark" if v < 1e-8 else r"\xmark" for v in inv[t]]
        tab.columns = ["Measure"] + [rf"\textsc{{{t.split()[0]}}}" for t in TRANSFORMS]
        save_table(tab, "tab_invariance", col_format="l" + "c" * 6)
        inv.to_csv(TAB / "tab_invariance_raw.csv", index=False)
        n_ent = len(EXPECTED) * 6
        R.set("a2.n-entries", n_ent, "{:d}")
        R.set("a2.verdict", f"all {n_ent} entries agree with the analytical derivation" if not mism else
              "the numerical check disagrees with the derivation for " + ", ".join(f"{m}/{t}" for m, t in mism))
        print(inv.round(10)); print("mismatches:", mism)

### A.3 Analytical null (Proposition 2): Monte Carlo check and Box 3 figure

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.3")
        def haar_sumcos2(n, k1, k2, n_draw, seed):
            g = torch.Generator().manual_seed(seed)
            V, _ = torch.linalg.qr(torch.randn(n, k2, generator=g, dtype=torch.float64))
            out = np.empty(n_draw); bs = 200
            for i in range(0, n_draw, bs):
                m = min(bs, n_draw - i)
                Q, _ = torch.linalg.qr(torch.randn(m, n, k1, generator=g, dtype=torch.float64))
                out[i:i + m] = ((Q.mT @ V) ** 2).sum((-1, -2)).numpy()
            return out
        grid = [(32, 8, 2), (64, 8, 8), (128, 64, 64), (768, 64, 64), (2048, 64, 256), (4096, 128, 128)]
        rows = []
        for (nn_, k1, k2) in grid:
            nd = CFG["n_null_mc"] if nn_ <= 768 else max(100, CFG["n_null_mc"] // 10)
            s = haar_sumcos2(nn_, k1, k2, nd, stable_hash("null", nn_, k1, k2))
            mu, var = k1 * k2 / nn_, null_var_sumcos2(k1, k2, nn_)
            rows.append(dict(n=nn_, k=k1, kp=k2, draws=nd, mc_mean=s.mean(), th_mean=mu, mc_var=s.var(ddof=1), th_var=var,
                             rel_err_mean=abs(s.mean() - mu) / mu, rel_err_var=abs(s.var(ddof=1) - var) / var,
                             se_rel_mean=s.std(ddof=1) / np.sqrt(nd) / mu))
        nullchk = pd.DataFrame(rows)
        save_table(nullchk[["n", "k", "kp", "draws", "mc_mean", "th_mean", "mc_var", "th_var"]].rename(
            columns={"kp": "k'", "mc_mean": "MC mean", "th_mean": "theory mean", "mc_var": "MC var", "th_var": "theory var"}),
            "tab_null_check", float_fmt="{:.4g}")
        R.set("a3.max-relerr-mean", float(nullchk.rel_err_mean.max() * 100), "{:.2f}\\%")
        R.set("a3.max-relerr-var", float(nullchk.rel_err_var.max() * 100), "{:.1f}\\%")
        R.set("a3.n-configs", len(grid), "{:d}")
        nullchk["z_mean"] = (nullchk.mc_mean - nullchk.th_mean) / (np.sqrt(nullchk.th_var / nullchk.draws))
        nullchk["z_var"] = (nullchk.mc_var - nullchk.th_var) / (nullchk.th_var * np.sqrt(2.0 / (nullchk.draws - 1)))
        R.set("a3.max-z", float(max(nullchk.z_mean.abs().max(), nullchk.z_var.abs().max())), "{:.1f}")
        print(nullchk)

        # Box 3 figure: null distributions and the null level as a function of d_head / d_model
        fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
        for (nn_, kk), col in zip([(128, 64), (768, 64), (4096, 128)], ["#C44E52", "#4C72B0", "#55A868"]):
            s = haar_sumcos2(nn_, kk, kk, 2000 if not SMOKE else 100, stable_hash("box3", nn_)) / kk
            ax[0].hist(s - kk / nn_, bins=40, density=True, alpha=0.55, color=col,
                       label=rf"$n={nn_},\ k={kk}$: $\mu_0={kk/nn_:.3f}$")
        ax[0].axvline(0, color="0.3", lw=1, ls="--")
        ax[0].set_xlabel(r"overlap $-\ \mu_0$ for two random subspaces"); ax[0].set_ylabel("density")
        ax[0].legend(frameon=False, fontsize=6.5); ax[0].set_title(r"(a) random overlaps concentrate at $\mu_0=k/n$")
        arch = {"BERT-tiny": (64, 128, (4, -8)), "BERT-mini": (64, 256, (4, -8)), "BERT-small": (64, 512, (4, -8)),
                "BERT-base, GPT-2": (64, 768, (4, -8)), "Qwen2.5-0.5B": (64, 896, (-58, 2)),
                "Llama-3.1-8B, Mistral-7B": (128, 4096, (4, -8))}
        xs = np.logspace(np.log10(0.02), np.log10(0.6), 100)
        ax[1].plot(xs, xs, color="0.3", lw=1)
        for name, (kk, nn_, off) in arch.items():
            ax[1].scatter(kk / nn_, kk / nn_, s=14, zorder=3)
            ax[1].annotate(name, (kk / nn_, kk / nn_), fontsize=6.5, xytext=off, textcoords="offset points")
        ax[1].set_xscale("log"); ax[1].set_yscale("log")
        from matplotlib.ticker import NullFormatter, FixedLocator, FixedFormatter
        for axis in (ax[1].xaxis, ax[1].yaxis):
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_locator(FixedLocator([0.03, 0.1, 0.3])); axis.set_major_formatter(FixedFormatter(["0.03", "0.1", "0.3"]))
        ax[1].set_xlabel(r"$d_{\mathrm{head}}/d_{\mathrm{model}}$"); ax[1].set_ylabel(r"null overlap $\mu_0$")
        ax[1].set_title("(b) architecture constants fix the null")
        savefig(fig, "fig_box3_null")

### A.4 Toy 3-D figures for Boxes 1, 2 and 4

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.4")
        from mpl_toolkits.mplot3d.art3d import Poly3DCollection
        def plane_patch(ax, u, v, color, alpha=0.18, s=1.2):
            pts = [s * (a * u + b * v) for a, b in [(-1, -1), (1, -1), (1, 1), (-1, 1)]]
            ax.add_collection3d(Poly3DCollection([pts], color=color, alpha=alpha, linewidths=0))
        def arrow(ax, v, color, lw=1.6, label=None):
            ax.quiver(0, 0, 0, *v, color=color, lw=lw, arrow_length_ratio=0.12)
            if label: ax.text(*(1.08 * np.asarray(v)), label, color=color, fontsize=8)
        def style(ax, title):
            ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([]); ax.set_title(title, fontsize=8.5)
            ax.view_init(elev=18, azim=-60)

        # Box 1: two planes sharing a line -> theta_1 = 0, theta_2 = dihedral angle
        phi = np.deg2rad(55)
        e1 = np.array([1.0, 0, 0]); a2 = np.array([0, 1.0, 0]); b2 = np.array([0, np.cos(phi), np.sin(phi)])
        fig = plt.figure(figsize=(7.2, 2.6))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        plane_patch(ax, e1, a2, "#4C72B0"); plane_patch(ax, e1, b2, "#DD8452")
        arrow(ax, e1, "k", label=r"$u_1=v_1$"); arrow(ax, a2, "#4C72B0", label=r"$u_2$"); arrow(ax, b2, "#DD8452", label=r"$v_2$")
        style(ax, r"$\mathcal{Q}$ (blue) and $\mathcal{K}$ (orange): $\theta_1=0^\circ,\ \theta_2=55^\circ$")
        ax2 = fig.add_subplot(1, 2, 2)
        ths = np.linspace(0, 90, 91)
        ax2.plot(ths, (1 + np.cos(np.deg2rad(ths)) ** 2) / 2, color="k")
        ax2.axvline(55, color="#DD8452", ls="--", lw=1)
        ax2.axhline(2 / 3, color="0.5", ls=":", lw=1); ax2.text(2, 2 / 3 + 0.015, r"chance for $k=2$ in $\mathbb{R}^3$: $2/3$", fontsize=7)
        ax2.set_xlabel(r"dihedral angle $\theta_2$ (deg)"); ax2.set_ylabel(r"overlap $=\frac{1}{2}(\cos^2\theta_1+\cos^2\theta_2)$")
        savefig(fig, "fig_box1_angles")

        # Box 2: gauge -- two bases of the same plane; overlap is constant, raw cosine is not
        fig = plt.figure(figsize=(7.2, 2.6))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        u, v = np.array([1.0, 0.2, 0.1]), np.array([0.1, 1.0, 0.4]); u /= np.linalg.norm(u); v /= np.linalg.norm(v)
        plane_patch(ax, u, v - (u @ v) * u / 1.0, "#4C72B0")
        Aex = np.array([[1.3, 0.8], [-0.4, 0.9]])
        W = np.stack([u, v], 1); W2 = W @ Aex
        arrow(ax, W[:, 0], "#4C72B0", label=r"$w_1$"); arrow(ax, W[:, 1], "#4C72B0", label=r"$w_2$")
        arrow(ax, W2[:, 0] / np.linalg.norm(W2[:, 0]), "#8172B3", label=r"$(WA)_1$")
        arrow(ax, W2[:, 1] / np.linalg.norm(W2[:, 1]), "#8172B3", label=r"$(WA)_2$")
        style(ax, r"$W_Q$ and $W_QA$ span the same plane")
        ax2 = fig.add_subplot(1, 2, 2)
        Wq0, Wk0 = rand_weight(32, 4), rand_weight(32, 4)
        ovs, raws, ckas = [], [], []
        for _ in range(60):
            A = rand_gl(4, cond=float(np.exp(rng.uniform(0, 4))))
            Wq1, Wk1 = Wq0 @ A, Wk0 @ torch.linalg.inv(A).mT
            ovs.append(sim_overlap(Wq1, Wk1)); raws.append(sim_rawcos(Wq1, Wk1)); ckas.append(sim_cka(Wq1, Wk1))
        xs_ = np.arange(60)
        ax2.plot(xs_, ovs, "o", ms=3, color="k", label="overlap (principal angles)")
        ax2.plot(xs_, ckas, "s", ms=3, color="#DD8452", label="linear CKA")
        ax2.plot(xs_, raws, "^", ms=3, color="#C44E52", label="raw cosine")
        ax2.set_xlabel(r"random gauge $A$ (same $W_QW_K^\top$)"); ax2.set_ylabel("similarity of $W_Q$, $W_K$")
        ax2.set_ylim(top=max(max(ovs), max(ckas), max(raws)) * 1.45)
        ax2.legend(frameon=False, fontsize=6.5, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0))
        savefig(fig, "fig_box2_gauge")

        # Box 4: RoPE rotates head coordinates, not the residual-space plane
        fig = plt.figure(figsize=(7.2, 2.6))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        plane_patch(ax, u, (v - (u @ v) * u) / np.linalg.norm(v - (u @ v) * u), "#55A868")
        for t_, al in zip(np.linspace(0, 2.4, 7), np.linspace(0.3, 1.0, 7)):
            Rt = np.array([[np.cos(t_), -np.sin(t_)], [np.sin(t_), np.cos(t_)]])
            col = (W @ Rt)[:, 0]
            ax.quiver(0, 0, 0, *(col / np.linalg.norm(col)), color=(0.2, 0.4, 0.2, al), lw=1.2, arrow_length_ratio=0.1)
        style(ax, r"columns of $W_QR_\delta$ for several $\delta$ stay in $\mathrm{Im}(W_Q)$")
        ax2 = fig.add_subplot(1, 2, 2)
        om = 1.0 / (10000 ** (np.arange(0, 64, 2) / 64))
        dl = np.unique(np.round(np.logspace(0, np.log10(256), 60)).astype(int))
        im = ax2.imshow(np.cos(np.outer(om, dl)), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1, origin="lower",
                        extent=[0, len(dl), 0, len(om)])
        tk = [1, 4, 16, 64, 256]
        ax2.set_xticks([int(np.searchsorted(dl, t)) + 0.5 for t in tk]); ax2.set_xticklabels(tk)
        ax2.set_xlabel(r"query-key distance $\delta$"); ax2.set_ylabel(r"plane $j$ (high $\to$ low frequency)")
        ax2.set_title(r"$\cos(\delta\omega_j)$: weight of plane $j$'s aligned part in $B_\delta$", fontsize=8.5)
        fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
        savefig(fig, "fig_box4_rope")

### A.5 Proposition 3 (RoPE): numerical check
(i) the principal angles between the left and right spaces of $B_\delta$ do not depend on $\delta$;
(ii) the gauge maps that commute with every $R_\delta$ (per-plane scaled rotations, any $GL$ on
the pass-through block) leave every $B_\delta$ unchanged and leave per-plane overlaps unchanged;
(iii) a generic $GL$ gauge changes $B_\delta$ for $\delta\neq0$.

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.5")
        n, dh, rot = 64, 16, 12
        inv = 1.0 / (10000 ** (torch.arange(0, rot, 2, dtype=torch.float64) / rot))
        Wq, Wk = rand_weight(n, dh), rand_weight(n, dh)
        blk = AttnBlock(0, "self", Wq[None], Wk[None], Wk[None], None, None, None, None, np.zeros(1, dtype=int),
                        rope={"inv_freq": inv, "rot_dim": rot})
        rd, pdf = rope_block_analysis(blk, CFG["rope_offsets"], topr=4, fold=False)
        dev_prop2 = rd.prop2_dev.max()
        # explicit B_delta vs core-based self-alignment
        dev_core = 0.0
        for dlt in CFG["rope_offsets"]:
            B = Wq @ rope_matrix(inv, rot, dh, -dlt) @ Wk.mT
            s_exp = float(torch.trace(B) / torch.linalg.svdvals(B).sum())
            dev_core = max(dev_core, abs(s_exp - rd[rd.delta == dlt].s_align.iloc[0]))
        # commutant gauge
        Acom = torch.zeros(dh, dh, dtype=torch.float64)
        for j, (a, b) in enumerate(rope_planes(rot)):
            r_, ph = float(np.exp(rng.normal())), float(rng.uniform(0, 2 * np.pi))
            Acom[a, a] = Acom[b, b] = r_ * np.cos(ph); Acom[a, b] = -r_ * np.sin(ph); Acom[b, a] = r_ * np.sin(ph)
        Acom[rot:, rot:] = rand_gl(dh - rot)
        Agen = rand_gl(dh)
        dev_com, dev_gen = 0.0, 0.0
        for dlt in CFG["rope_offsets"]:
            Rm = rope_matrix(inv, rot, dh, -dlt)
            B0 = Wq @ Rm @ Wk.mT
            B1 = (Wq @ Acom) @ Rm @ (Wk @ torch.linalg.inv(Acom).mT).mT
            B2 = (Wq @ Agen) @ Rm @ (Wk @ torch.linalg.inv(Agen).mT).mT
            dev_com = max(dev_com, float((B1 - B0).abs().max() / B0.abs().max()))
            dev_gen = max(dev_gen, float((B2 - B0).abs().max() / B0.abs().max()))
        blk_c = AttnBlock(0, "self", (Wq @ Acom)[None], (Wk @ torch.linalg.inv(Acom).mT)[None], Wk[None], None, None, None, None,
                          np.zeros(1, dtype=int), rope={"inv_freq": inv, "rot_dim": rot})
        blk_g = AttnBlock(0, "self", (Wq @ Agen)[None], (Wk @ torch.linalg.inv(Agen).mT)[None], Wk[None], None, None, None, None,
                          np.zeros(1, dtype=int), rope={"inv_freq": inv, "rot_dim": rot})
        _, pc = rope_block_analysis(blk_c, [0], topr=4, fold=False)
        _, pg = rope_block_analysis(blk_g, [0], topr=4, fold=False)
        dev_plane_com = float((pc.overlap.values - pdf.overlap.values).__abs__().max())
        dev_plane_gen = float((pg.overlap.values - pdf.overlap.values).__abs__().max())
        R.set("a5.prop2-dev", fmt_sci(dev_prop2))
        R.set("a5.core-dev", fmt_sci(dev_core))
        R.set("a5.commutant-dev", fmt_sci(dev_com))
        R.set("a5.generic-dev", float(dev_gen), "{:.2f}")
        R.set("a5.plane-commutant-dev", fmt_sci(dev_plane_com))
        R.set("a5.plane-generic-dev", float(dev_plane_gen), "{:.2f}")
        print(dict(prop2=dev_prop2, core=dev_core, com=dev_com, gen=dev_gen, plane_com=dev_plane_com, plane_gen=dev_plane_gen))

### A.6 Validation of the weight extraction
For every supported architecture, a small randomly initialised model is built from its
configuration; the attention probabilities returned by the reference implementation are
reconstructed from the extracted per-head matrices, the captured attention inputs, the stored
biases and our own rotary implementation. This checks head layouts, GQA maps, normalisation
positions and RoPE conventions without downloading any checkpoint.

In [ ]:
if section_enabled("A"):
    with guard():
        R.cell("A.6")
        def _recon_err(mid, qmod, kind="self", layer=1, causal=True, bias=None, T=9):
            m_ = smoke_model(mid, task=default_task(mid)).double()
            ws_ = ws_from_model(m_, mid)
            blk = [b for b in iter_blocks(ws_) if b.layer == layer and b.kind == kind][0]
            sd_ = m_.state_dict(); store = {}
            hk = _find_module(m_, qmod.format(i=layer)).register_forward_hook(lambda mod, i, o: store.__setitem__("x", i[0].detach()))
            ids_ = torch.randint(4, SMOKE_VOCAB, (1, T), generator=torch.Generator().manual_seed(1))
            kw = {"decoder_input_ids": ids_} if default_task(mid) == "seq2seq" else {}
            out = m_(input_ids=ids_, output_attentions=True, **kw); hk.remove()
            att = out.attentions if getattr(out, "attentions", None) is not None else out.encoder_attentions
            A_ = att[layer][0].double(); x = store["x"][0].double()
            bq, bk = bias(sd_, blk) if bias else (torch.zeros(blk.n_q * blk.dh, dtype=GEOM_DTYPE), torch.zeros(blk.n_kv * blk.dh, dtype=GEOM_DTYPE))
            q = torch.einsum("td,hdk->thk", x, blk.Wq) + bq.view(1, blk.n_q, blk.dh)
            k = torch.einsum("td,hdk->thk", x, blk.Wk) + bk.view(1, blk.n_kv, blk.dh)
            pos = torch.arange(T)
            if blk.rope is not None:
                q = apply_rope_acts(q, pos, blk.rope["inv_freq"], blk.rope["rot_dim"]); k = apply_rope_acts(k, pos, blk.rope["inv_freq"], blk.rope["rot_dim"])
            S_ = torch.einsum("thk,shk->hts", q, k[:, blk.kv_of_q]) / math.sqrt(blk.dh)
            if causal:
                S_ = S_.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), -1e30)
            return float((torch.softmax(S_, -1) - A_).abs().max())
        checks = {
            "BERT": ("google-bert/bert-base-uncased", "encoder.layer.{i}.attention.self.query", "self", False,
                     lambda sd_, b: (sd_["bert.encoder.layer.1.attention.self.query.bias"], sd_["bert.encoder.layer.1.attention.self.key.bias"])),
            "DistilBERT": ("distilbert/distilbert-base-uncased", "transformer.layer.{i}.attention.q_lin", "self", False,
                           lambda sd_, b: (sd_["distilbert.transformer.layer.1.attention.q_lin.bias"], sd_["distilbert.transformer.layer.1.attention.k_lin.bias"])),
            "GPT-2": ("openai-community/gpt2", "h.{i}.attn.c_attn", "self", True,
                      lambda sd_, b: (sd_["transformer.h.1.attn.c_attn.bias"][:b.d], sd_["transformer.h.1.attn.c_attn.bias"][b.d:2 * b.d])),
            "GPT-NeoX (partial RoPE)": ("EleutherAI/pythia-160m", "layers.{i}.attention.query_key_value", "self", True,
                                        lambda sd_, b: tuple(sd_["gpt_neox.layers.1.attention.query_key_value.bias"].view(b.n_q, 3, b.dh)[:, j].reshape(-1) for j in (0, 1))),
            "Llama (GQA, RoPE)": ("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", "layers.{i}.self_attn.q_proj", "self", True, None),
            "Qwen2 (GQA, RoPE, biases)": ("Qwen/Qwen2.5-0.5B", "layers.{i}.self_attn.q_proj", "self", True,
                                           lambda sd_, b: (sd_["model.layers.1.self_attn.q_proj.bias"], sd_["model.layers.1.self_attn.k_proj.bias"])),
            "Mistral (GQA, RoPE)": ("mistralai/Mistral-7B-v0.1", "layers.{i}.self_attn.q_proj", "self", True, None),
            "Phi (partial RoPE)": ("microsoft/phi-2", "layers.{i}.self_attn.q_proj", "self", True,
                                   lambda sd_, b: (sd_["model.layers.1.self_attn.q_proj.bias"], sd_["model.layers.1.self_attn.k_proj.bias"])),
            "BART encoder": ("facebook/bart-base", "encoder.layers.{i}.self_attn.q_proj", "enc-self", False,
                             lambda sd_, b: (sd_["model.encoder.layers.1.self_attn.q_proj.bias"], sd_["model.encoder.layers.1.self_attn.k_proj.bias"])),
        }
        errs = {}
        for name, (mid, qmod, kind, causal, bias) in checks.items():
            errs[name] = _recon_err(mid, qmod, kind, 1, causal, bias)
        print(pd.Series(errs))
        R.set("a6.recon-maxerr", fmt_sci(max(errs.values())))
        R.set("a6.n-arch", len(errs), "{:d}")

## B0. Re-analysis of the twelve reference checkpoints (those of the preliminary analysis)
Raw overlaps (unfolded weights, as in the preliminary analysis) are recomputed for continuity;
all inferences use the gain-folded, null-calibrated quantities.

In [ ]:
if section_enabled("B0"):
    with guard():
        R.cell("B0.1")
        # folded and raw geometry in one pass per checkpoint (weights fetched once; both cached on disk,
        # so an interrupted run resumes at the first unfinished checkpoint)
        df0, info0 = geo_many(ORIG12, fold=True, both_folds=True)
        df0raw, _ = geo_many(ORIG12, fold=False)
        S0 = summarize(df0)
        S0raw = summarize(df0raw, value="overlap")
        orig_models = [m for m in ORIG12 if m in set(df0.model)]
        R.set("b0.n-models", len(orig_models), "{:d}")
        def gv(S, m, p, col="mean"):
            r = S[(S.model == m) & (S.pair == p)]
            return float(r[col].iloc[0]) if len(r) else np.nan
        BB, DB, BT = "google-bert/bert-base-uncased", "distilbert/distilbert-base-uncased", "prajjwal1/bert-tiny"
        for tag, m in [("bert-base", BB), ("distil", DB), ("bert-tiny", BT)]:
            R.set(f"b0.{tag}-qk-raw", gv(S0raw, m, "QK", "mean"), "{:.3f}")
            R.set(f"b0.{tag}-kv-raw", gv(S0raw, m, "KV", "mean"), "{:.3f}")
            R.set(f"b0.{tag}-qv-raw", gv(S0raw, m, "QV", "mean"), "{:.3f}")
            R.set(f"b0.{tag}-null", gv(S0, m, "QK", "null"), "{:.3f}")
            r = S0[(S0.model == m) & (S0.pair == "QK")]
            if len(r):
                R.set(f"b0.{tag}-qk-rho", fmt_ci(r["mean"].iloc[0], r["lo"].iloc[0], r["hi"].iloc[0]))
        # sensitivity to normalisation-gain folding
        fold_diff = (S0raw.set_index(["model", "pair"])["mean"] - S0.set_index(["model", "pair"])["overlap"]).abs()
        R.set("b0.fold-maxdiff", float(fold_diff.max()), "{:.3f}")
        R.set("b0.fold-maxdiff-model", short(fold_diff.idxmax()[0]) + ", " + {"QK": "Q–K", "KV": "K–V", "QV": "Q–V"}[fold_diff.idxmax()[1]])
        # hierarchy tests
        H0 = hierarchy_tests(df0)
        H0["regime"] = H0.model.map(regime_of)
        R.set("b0.hierarchy-verdict", hier_sentence(H0))
        # matched vs mismatched coupling
        cpl = S0[S0.pair == "QK"]
        R.set("b0.coupling-pos", int((cpl.coupling_lo > 0).sum()), "{:d}")
        R.set("b0.coupling-n", int(cpl.coupling.notna().sum()), "{:d}")
        # table
        rows_tab = []
        for m in orig_models:
            i_ = info0[info0.model == m].iloc[0]
            row = {"Model": short(m), "Regime": regime_of(m), r"$d_h/d$": f"{i_.d_head}/{i_.d_model}",
                   r"$\mu_0$": i_.d_head / i_.d_model, "raw Q–K": gv(S0raw, m, "QK")}
            for p in PAIRS:
                r = S0[(S0.model == m) & (S0.pair == p)].iloc[0]
                row[rf"$\rho_{{{p}}}$"] = fmt_ci(r["mean"], r["lo"], r["hi"], 3)
            r = S0[(S0.model == m) & (S0.pair == "QK")].iloc[0]
            row["coupling"] = fmt_ci(r.get("coupling", np.nan), r.get("coupling_lo", np.nan), r.get("coupling_hi", np.nan), 3) \
                if np.isfinite(r.get("coupling", np.nan)) else "--"
            row["ordering"] = H0.set_index("model").loc[m, "ordering"].replace(">", "$>$").replace("<", "$<$").replace("~", r"$\approx$")
            rows_tab.append(row)
        save_table(pd.DataFrame(rows_tab), "tab_rq1_orig", col_format="llcccccccl")
        H0.to_csv(TAB / "hierarchy_orig.csv", index=False)
        print(S0.round(3).to_string()); print(H0)

In [ ]:
if section_enabled("B0"):
    with guard():
        R.cell("B0.2")
        fig = plot_rq1_orig(S0, S0raw, orig_models)
        savefig(fig, "fig_rq1_orig")

        fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.4), sharey=True)
        for a_, p in zip(axs, PAIRS):
            for m in orig_models:
                g = df0[(df0.model == m) & (df0.pair == p) & (df0.kind == "self")].groupby("depth")["rho"]
                mu, sd = g.mean(), g.std().fillna(0)
                a_.plot(mu.index, mu.values, lw=1, label=short(m))
                a_.fill_between(mu.index, mu - sd, mu + sd, alpha=0.08)
            a_.axhline(0, color="k", lw=0.6); a_.set_xlabel("relative depth"); a_.set_title(p.replace("QK", "Q–K").replace("KV", "K–V").replace("QV", "Q–V"))
        axs[0].set_ylabel(r"$\rho$ (mean $\pm$ s.d. over heads)")
        axs[-1].legend(frameon=False, fontsize=5.5, ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left")
        savefig(fig, "fig_rq1_depth")

        fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.6), gridspec_kw=dict(width_ratios=[1, 1, 0.5]))
        for a_, m in zip(axs, [BB, DB, BT]):
            d_, _ = geo(m)
            if d_ is not None:
                im = head_heatmap(a_, d_, "QK", "rho", title=short(m), vmin=-0.1, vmax=0.8)
        fig.colorbar(im, ax=axs, fraction=0.025, label=r"$\rho_{QK}$")
        savefig(fig, "fig_heatmaps_orig")

### B0.3 Empirical comparison of similarity measures (invariance table on real weights)
Per head (Q–K pair, gain folded): all measures of the invariance table (A.2); Spearman correlations across heads;
sensitivity to a random Q–K gauge $A$ (condition number 10) applied to real weights, which leaves
the network function unchanged.

In [ ]:
if section_enabled("B0"):
    with guard():
        R.cell("B0.3")
        MET_MODELS = ["google-bert/bert-base-uncased", "openai-community/gpt2", "EleutherAI/pythia-160m", "Qwen/Qwen2.5-0.5B"]
        rows = []
        rng_m = np.random.default_rng(SEED + 3)
        for m in MET_MODELS:
            try:
                ws = open_weights(m)
            except Exception as e:
                record_missing(m, e); continue
            for blk in iter_blocks(ws):
                Wq, Wk = blk.folded("q"), blk.folded("k")
                for h in range(blk.n_q):
                    X, Y = Wq[h], Wk[blk.kv_of_q[h]]
                    r = {"model": m, "layer": blk.layer, "head": h}
                    for name, f in METRICS.items():
                        r[name] = f(X, Y)
                    # anisotropy of the two read matrices
                    sx, sy = torch.linalg.svdvals(X), torch.linalg.svdvals(Y)
                    r["aniso"] = float(np.log10((sx[0] / sx[-1]) * (sy[0] / sy[-1])))
                    # functional-preserving random gauge
                    U_, _ = np.linalg.qr(rng_m.standard_normal((blk.dh, blk.dh))); V_, _ = np.linalg.qr(rng_m.standard_normal((blk.dh, blk.dh)))
                    A = torch.tensor(U_ @ np.diag(np.exp(np.linspace(0, -np.log(10), blk.dh))) @ V_.T)
                    X2, Y2 = X @ A, Y @ torch.linalg.inv(A).mT
                    for name, f in METRICS.items():
                        a, b = r[name], f(X2, Y2)
                        r["gauge_" + name] = abs(a - b) / max(abs(a), 1e-12) if np.isfinite(a) else np.nan
                    rows.append(r)
            ws.close()
        MET = pd.DataFrame(rows)
        names = list(METRICS)
        signed = MET[names].copy()
        for dname in ["geodesic", "chordal", "projF"]:
            signed[dname] = -signed[dname]          # distances -> similarities for display
        # per-model Spearman matrices, averaged (pooling across models with different d_head would mix scales)
        CORR = sum(signed[MET.model == m].corr(method="spearman") for m in MET.model.unique()) / MET.model.nunique()
        gch = MET[["gauge_" + n for n in names]].median()
        R.set("b0m.n-heads", len(MET), "{:d}")
        R.set("b0m.overlap-chordal", float(CORR.loc["overlap", "chordal"]), "{:.3f}")
        R.set("b0m.overlap-geodesic", float(CORR.loc["overlap", "geodesic"]), "{:.3f}")
        R.set("b0m.overlap-cka", float(CORR.loc["overlap", "CKA"]), "{:.2f}")
        R.set("b0m.overlap-svcca", float(CORR.loc["overlap", "SVCCA"]), "{:.2f}")
        R.set("b0m.overlap-pwcca", float(CORR.loc["overlap", "PWCCA"]), "{:.2f}")
        R.set("b0m.overlap-procrustes", float(CORR.loc["overlap", "Procrustes"]), "{:.2f}")
        R.set("b0m.overlap-rawcos", float(CORR.loc["overlap", "raw cosine"]), "{:.2f}")
        R.set("b0m.overlap-salign", float(CORR.loc["overlap", "self-alignment s"]), "{:.2f}")
        R.set("b0m.gauge-overlap", fmt_sci(float(gch["gauge_overlap"]) if gch["gauge_overlap"] > 0 else 0.0))
        R.set("b0m.gauge-cka", float(gch["gauge_CKA"] * 100), "{:.0f}\\%")
        R.set("b0m.gauge-procrustes", float(gch["gauge_Procrustes"] * 100), "{:.0f}\\%")
        R.set("b0m.gauge-rawcos", float(gch["gauge_raw cosine"] * 100), "{:.0f}\\%")
        rk = lambda v: st.rankdata(v) / len(v)
        MET["discord"] = np.abs(rk(MET["overlap"]) - rk(MET["CKA"]))
        r_, lo_, hi_, _ = spearman_ci(MET["aniso"], MET["discord"])
        R.set("b0m.discord-aniso", fmt_ci(r_, lo_, hi_, 2))
        fig, ax = plt.subplots(1, 2, figsize=(7.6, 3.2), gridspec_kw=dict(width_ratios=[1.25, 1]), layout="constrained")
        im = ax[0].imshow(CORR.values, cmap="RdBu_r", vmin=-1, vmax=1)
        ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels(names, rotation=60, ha="right", fontsize=6.5)
        ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels(names, fontsize=6.5)
        for i in range(len(names)):
            for j in range(len(names)):
                ax[0].text(j, i, f"{CORR.values[i, j]:.2f}", ha="center", va="center", fontsize=5)
        ax[0].set_title("(a) Spearman across heads, mean over models\n(distances sign-flipped)")
        fig.colorbar(im, ax=ax[0], fraction=0.046, pad=0.02)
        vals = np.maximum(gch.values, 1e-17)
        ax[1].barh(range(len(names)), vals, color=["#4C72B0"] * 4 + ["#DD8452"] * 5 + ["#55A868"])
        ax[1].set_xscale("log"); ax[1].set_yticks(range(len(names))); ax[1].set_yticklabels(names, fontsize=6.5)
        ax[1].set_xlabel("median relative change under a\nfunction-preserving Q–K gauge"); ax[1].set_title("(b) gauge sensitivity (real weights)")
        savefig(fig, "fig_metrics")
        MET.to_csv(TAB / "metrics_per_head.csv", index=False)

## B1. Seed noise floor of the MultiBERTs
The noise floor of a model-level statistic $S$ (mean over heads of $\rho_{QK}$, $\rho_{KV}$,
$\rho_{QV}$ or of the Q–K coupling) is the 95th percentile of $|S_i-S_j|$ over pairs of
independently seeded pre-training runs with identical data and recipe. It serves as a reference
scale for differences between single BERT-sized checkpoints.

In [ ]:
if section_enabled("B1"):
    with guard():
        R.cell("B1.1")
        MB, FLOOR["BERT"] = seed_family(MULTIBERTS, "MultiBERTs")
        R.set("b1.mb-n", len(MB), "{:d}")
        for k, tag in [("rho_QK", "qk"), ("rho_KV", "kv"), ("rho_QV", "qv"), ("coupling_QK", "cpl")]:
            f = FLOOR["BERT"][k]
            R.set(f"b1.mb-{tag}-mean", f["mean"], "{:.3f}")
            R.set(f"b1.mb-{tag}-sd", f["sd"], "{:.4f}")
            R.set(f"b1.mb-{tag}-q95", f["q95"], "{:.4f}")
        Hmb = hierarchy_tests(pd.concat([geo(m)[0] for m in MB.model]))
        R.set("b1.mb-hier-count", f"{int(Hmb.full_hierarchy.sum())} of {len(Hmb)}")
        json.dump(FLOOR, open(OUT / "floor.json", "w"), indent=1, default=float)   # for later sections and sessions
        # position of the original Google BERT-base and of DistilBERT relative to the seed distribution
        bb = model_stats(geo("google-bert/bert-base-uncased")[0]) if geo("google-bert/bert-base-uncased")[0] is not None else None
        if bb is not None and len(MB):
            lo, hi = MB.rho_QK.min(), MB.rho_QK.max()
            R.set("b1.bertbase-within", "inside" if lo <= bb["rho_QK"] <= hi else "outside")
            db = geo("distilbert/distilbert-base-uncased")[0]
            if db is not None:
                dd = model_stats(db)["rho_QK"] - bb["rho_QK"]
                R.set("b1.distil-vs-base", dd, "{:+.3f}")
                R.set("b1.distil-vs-floor", "below" if abs(dd) < FLOOR["BERT"]["rho_QK"]["q95"] else "above")

## B2. Capacity versus distillation procedure
* 24 BERT miniatures (same data and recipe; depth $L$ and width $H$ vary) isolate capacity;
* distilled models are compared with the capacity-only fit at equal parameter count;
* Pythia 14M–2.8B (same data and order) and Pythia checkpoints follow the emergence of geometry;
* linear vs. one-breakpoint regression of $\rho$ on $\log_{10}$(parameters): a "threshold" is
  claimed only if the bootstrap sup-LR test rejects linearity after Holm correction **and**
  $\Delta\mathrm{BIC}\ge 6$ (pre-registered rule `THRESHOLD_RULE`).

In [ ]:
if section_enabled("B2"):
    with guard():
        R.cell("B2.1")
        MINI = model_level(MINIS)
        MINI["logP"] = np.log10(MINI.params)
        R.set("b2.minis-n", len(MINI), "{:d}")
        R.set("b2.minis-rho-min", MINI.rho_QK.min(), "{:.3f}"); R.set("b2.minis-rho-max", MINI.rho_QK.max(), "{:.3f}")
        r_, lo_, hi_, p_ = spearman_ci(MINI.logP, MINI.rho_QK)
        R.set("b2.minis-spearman", fmt_ci(r_, lo_, hi_, 2))
        r_, lo_, hi_, p_ = spearman_ci(MINI.null, MINI.raw_QK)
        R.set("b2.minis-raw-null-spearman", fmt_ci(r_, lo_, hi_, 2))
        if MINI.L.notna().all():
            ols = smf.ols("rho_QK ~ np.log2(L) + np.log2(H)", MINI).fit()
            ci = ols.conf_int()
            R.set("b2.minis-coef-l", fmt_ci(ols.params["np.log2(L)"], ci.loc["np.log2(L)", 0], ci.loc["np.log2(L)", 1], 3))
            R.set("b2.minis-coef-h", fmt_ci(ols.params["np.log2(H)"], ci.loc["np.log2(H)", 0], ci.loc["np.log2(H)", 1], 3))
            R.set("b2.minis-r2", ols.rsquared, "{:.2f}")

In [ ]:
if section_enabled("B2"):
    with guard():
        R.cell("B2.2")
        PYS = model_level([f"EleutherAI/pythia-{s}" for s in PYTHIA_SIZES])
        PYS["logP"] = np.log10(PYS.params)
        r_, lo_, hi_, _ = spearman_ci(PYS.logP, PYS.rho_QK)
        R.set("b2.pythia-n", len(PYS), "{:d}")
        R.set("b2.pythia-spearman", fmt_ci(r_, lo_, hi_, 2))
        fits = []
        for fam, D in [("BERT miniatures", MINI), ("Pythia sizes", PYS)]:
            for p in PAIRS:
                if len(D) < 6:
                    continue
                f = breakpoint_test(D.logP.values, D[f"rho_{p}"].values, seed=stable_hash(fam, p) % 10000)
                fits.append(dict(family=fam, pair=p, n=f["n"], dAIC=f["daic"], dBIC=f["dbic"],
                                 dBIC_quad=f["bic_lin"] - f["bic_quad"], psi=10 ** f["psi"], p_break=f["p_break"], fit=f))
        FITS = pd.DataFrame(fits)
        if len(FITS):
            FITS["p_holm"] = holm(FITS.p_break.values)
            FITS["threshold"] = (FITS.p_holm < THRESHOLD_RULE["alpha"]) & (FITS.dBIC >= THRESHOLD_RULE["min_dbic"])
            tab = FITS[["family", "pair", "n", "dAIC", "dBIC", "dBIC_quad", "psi", "p_break", "p_holm", "threshold"]].copy()
            tab["psi"] = tab.psi.map(lambda v: f"{v / 1e6:.1f}M")
            tab["threshold"] = tab.threshold.map({True: "yes", False: "no"})
            tab["pair"] = tab.pair.map({"QK": "Q–K", "KV": "K–V", "QV": "Q–V"})
            save_table(tab.rename(columns={"dAIC": r"$\Delta$AIC", "dBIC": r"$\Delta$BIC", "dBIC_quad": r"$\Delta$BIC$_{\mathrm{quad}}$",
                                           "psi": r"$\hat\psi$", "p_break": "$p$", "p_holm": r"$p_{\mathrm{Holm}}$"}),
                       "tab_rq2_fits", float_fmt="{:.2f}")
            mq = FITS[(FITS.family == "BERT miniatures") & (FITS.pair == "QK")]
            if len(mq):
                R.set("b2.minis-qk-dbic", float(mq.dBIC.iloc[0]), "{:.1f}")
                R.set("b2.minis-qk-p", fmt_p(float(mq.p_holm.iloc[0])))
            yes, no = FITS[FITS.threshold], FITS[~FITS.threshold]
            pn = lambda q: q.replace('QK', 'Q–K').replace('KV', 'K–V').replace('QV', 'Q–V')
            if len(yes):
                R.set("b2.threshold-verdict", "a breakpoint meets the pre-registered criterion for " + "; ".join(
                    f"the {pn(r.pair)} excess of the {r.family} ($\\hat\\psi\\approx{r.psi / 1e6:.0f}$M parameters, "
                    f"$\\Delta$BIC $={r.dBIC:.1f}$)" for r in yes.itertuples())
                    + (", but not for " + "; ".join(f"the {pn(r.pair)} excess of the {r.family} ($\\Delta$BIC $={r.dBIC:.1f}$)"
                                                    for r in no.itertuples()) if len(no) else ""))
            else:
                R.set("b2.threshold-verdict", f"none of the {len(FITS)} family $\\times$ pair fits meets the pre-registered "
                      "breakpoint criterion, so the data are described as a continuous dependence on log-parameters rather "
                      "than as a capacity threshold")
            R.set("b2.n-fits", len(FITS), "{:d}")

In [ ]:
if section_enabled("B2"):
    with guard():
        R.cell("B2.3")
        DIS = model_level(DISTILLED)
        DIS["logP"] = np.log10(DIS.params)
        # capacity-only prediction from the miniature fit, with 95% prediction intervals
        ols1 = smf.ols("rho_QK ~ logP", MINI).fit()
        pr = ols1.get_prediction(DIS[["logP"]]).summary_frame(alpha=0.05)
        DIS["pred"], DIS["pi_lo"], DIS["pi_hi"] = pr["mean"].values, pr["obs_ci_lower"].values, pr["obs_ci_upper"].values
        DIS["outside"] = (DIS.rho_QK < DIS.pi_lo) | (DIS.rho_QK > DIS.pi_hi)
        R.set("b2.distill-outside", f"{int(DIS.outside.sum())} of {len(DIS)}")
        tab = pd.DataFrame({"Model": DIS.model.map(short), "params (M)": DIS.params / 1e6, r"$d_h/d$": [f"{a}/{b}" for a, b in zip(DIS.d_head, DIS.d_model)],
                            r"$\rho_{QK}$ [95\% CI]": [fmt_ci(a, b, c) for a, b, c in zip(DIS.rho_QK, DIS.lo_QK, DIS.hi_QK)],
                            r"capacity-only prediction [95\% PI]": [fmt_ci(a, b, c) for a, b, c in zip(DIS.pred, DIS.pi_lo, DIS.pi_hi)],
                            "outside PI": DIS.outside.map({True: "yes", False: "no"})})
        save_table(tab, "tab_rq2_distilled", float_fmt="{:.1f}")
        # same-size, different-procedure contrast: DistilBERT vs the 6-layer, 768-wide miniature
        a = DIS[DIS.model == "distilbert/distilbert-base-uncased"]; b = MINI[MINI.model == "google/bert_uncased_L-6_H-768_A-12"]
        if len(a) and len(b):
            dd = float(a.rho_QK.iloc[0] - b.rho_QK.iloc[0])
            R.set("b2.distil-vs-mini", dd, "{:+.3f}")
            fl = FLOOR.get("BERT", {}).get("rho_QK", {}).get("q95", np.nan) if FLOOR else floor_for("google-bert/bert-base-uncased")[0]
            R.set("b2.distil-vs-mini-floor", "exceeds" if abs(dd) > fl else "does not exceed")

In [ ]:
if section_enabled("B2"):
    with guard():
        R.cell("B2.4")
        CK = []
        for sz in PYTHIA_CHECKPOINT_SIZES:
            for s in PYTHIA_STEPS:
                t = model_level([f"EleutherAI/pythia-{sz}"], revision=f"step{s}")
                if len(t):
                    t["size"], t["step"] = sz, s
                    CK.append(t)
        CK = pd.concat(CK, ignore_index=True) if CK else pd.DataFrame()
        if len(CK):
            s0 = CK[CK.step == 0]
            R.set("b2.step0-maxabs", fmt_sci(float(s0[[f"rho_{p}" for p in PAIRS]].abs().max().max())))
            em = []
            _sz = lambda z: float(z[:-1]) * (1e9 if z.endswith("b") else 1e6)
            for sz, g in sorted(CK.groupby("size"), key=lambda kv: _sz(kv[0])):
                g = g.sort_values("step"); fin = g.rho_QK.iloc[-1]
                hit = g[(g.rho_QK >= 0.5 * fin) & (g.step > 0)]
                em.append(f"{sz}: step {int(hit.step.iloc[0]) if len(hit) else 'n/a'}")
            R.set("b2.emergence", "; ".join(em))
            CK.to_csv(TAB / "pythia_checkpoints.csv", index=False)

        fig = plt.figure(figsize=(7.4, 5.0))
        gs = fig.add_gridspec(2, 3)
        ax = fig.add_subplot(gs[0, 0])
        if MINI.L.notna().all():
            Mh = MINI.pivot(index="L", columns="H", values="rho_QK")
            im = ax.imshow(Mh.values, origin="lower", cmap="viridis", aspect="auto")
            ax.set_xticks(range(Mh.shape[1])); ax.set_xticklabels(Mh.columns); ax.set_yticks(range(Mh.shape[0])); ax.set_yticklabels(Mh.index)
            for i in range(Mh.shape[0]):
                for j in range(Mh.shape[1]):
                    ax.text(j, i, f"{Mh.values[i, j]:.2f}", ha="center", va="center", fontsize=5.5, color="w")
            ax.set_xlabel("width $H$"); ax.set_ylabel("depth $L$"); ax.set_title(r"(a) miniatures: $\rho_{QK}$")
        ax = fig.add_subplot(gs[0, 1:])
        sc_ = ax.scatter(MINI.params, MINI.rho_QK, c=np.log2(MINI.H.astype(float)) if MINI.H.notna().all() else None, cmap="plasma", s=14, label="miniatures")
        ax.errorbar(MINI.params, MINI.rho_QK, yerr=[MINI.rho_QK - MINI.lo_QK, MINI.hi_QK - MINI.rho_QK], fmt="none", ecolor="0.6", lw=0.5)
        ax.scatter(DIS.params, DIS.rho_QK, marker="*", s=60, color="k", label="distilled")
        for r in DIS.itertuples():
            ax.annotate(short(r.model), (r.params, r.rho_QK), fontsize=5.5, xytext=(3, 3), textcoords="offset points")
        xs = np.linspace(MINI.logP.min(), MINI.logP.max(), 200)
        if len(FITS):
            f = FITS[(FITS.family == "BERT miniatures") & (FITS.pair == "QK")]["fit"].iloc[0]
            ax.plot(10 ** xs, np.column_stack([np.ones_like(xs), xs]) @ f["beta_lin"], "k-", lw=0.8, label="linear")
            ax.plot(10 ** xs, _seg_design(xs, f["psi"]) @ f["beta_seg"], "k--", lw=0.8, label="segmented")
        ax.set_xscale("log"); ax.set_xlabel("parameters"); ax.set_ylabel(r"$\rho_{QK}$"); ax.legend(frameon=False, fontsize=6.5)
        ax.set_title("(b) capacity-only fit and distilled models")
        ax = fig.add_subplot(gs[1, 0])
        for p in PAIRS:
            ax.errorbar(PYS.params, PYS[f"rho_{p}"], yerr=[PYS[f"rho_{p}"] - PYS[f"lo_{p}"], PYS[f"hi_{p}"] - PYS[f"rho_{p}"]],
                        marker="o", ms=3, lw=0.8, color=PAIR_COLORS[p], label=p)
        ax.set_xscale("log"); ax.set_xlabel("parameters"); ax.set_ylabel(r"$\rho$"); ax.set_title("(c) Pythia sizes"); ax.legend(frameon=False, fontsize=6.5)
        ax = fig.add_subplot(gs[1, 1:])
        if len(CK):
            _sz = lambda z: float(z[:-1]) * (1e9 if z.endswith("b") else 1e6)
            for sz, g in sorted(CK.groupby("size"), key=lambda kv: _sz(kv[0])):
                g = g.sort_values("step")
                ax.plot(g.step + 1, g.rho_QK, marker="o", ms=2.5, lw=0.9, label=f"Pythia-{sz} Q–K")
                ax.plot(g.step + 1, g.rho_KV, ls=":", lw=0.9, label=f"Pythia-{sz} K–V")
            ax.set_xscale("log"); ax.set_xlabel("training step + 1"); ax.set_ylabel(r"$\rho$"); ax.axhline(0, color="k", lw=0.5)
            ax.legend(frameon=False, fontsize=6, ncol=2); ax.set_title("(d) emergence during pre-training")
        fig.tight_layout()
        savefig(fig, "fig_rq2_capacity")

## B3. Positional encoding
### B3.1 Static analysis of the effective bilinear forms $B_\delta=W_QR_{-\delta}W_K^\top$
For $\delta\in\{0,1,4,16,64,256\}$: the full-rank principal angles (Proposition 3 predicts no
dependence on $\delta$), the spectral self-alignment $s(\delta)=\operatorname{tr}B_\delta/\|B_\delta\|_*$,
and the overlap of the dominant rank-$r$ left/right singular subspaces of $B_\delta$; per
frequency plane, the 2-D Q–K overlaps, which are gauge invariants only under RoPE.

In [ ]:
ROPE_MODELS = ["EleutherAI/pythia-160m", "EleutherAI/pythia-1.4b", "microsoft/phi-2",
               "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", "Qwen/Qwen2.5-0.5B", "meta-llama/Llama-3.2-1B",
               "mistralai/Mistral-7B-v0.1", "meta-llama/Llama-3.1-8B"]

def rope_model_analysis(mid, ws=None):
    key = hashlib.sha1(f"rope|{mid}|{GEOM_VERSION}".encode()).hexdigest()[:16]
    cp = CACHE / "geometry" / f"{key}.pkl"
    if ws is None and cp.exists():
        return pd.read_pickle(cp)
    ws_given = ws is not None
    try:
        ws = ws or open_weights(mid)
    except Exception as e:
        record_missing(mid, e); return None
    D, P = [], []
    for blk in iter_blocks(ws):
        if blk.rope is None:
            continue
        d_, p_ = rope_block_analysis(blk, CFG["rope_offsets"], CFG["topr"])
        d_["n"] = blk.d; D.append(d_); P.append(p_)
    if not ws_given:
        ws.close()
    out = dict(delta=pd.concat(D, ignore_index=True), planes=pd.concat(P, ignore_index=True))
    out["delta"]["model"] = mid; out["planes"]["model"] = mid
    if not ws_given:
        pd.to_pickle(out, cp)
    return out

if section_enabled("B3"):
    with guard():
        R.cell("B3.1")
        RA = {m: rope_model_analysis(m) for m in ROPE_MODELS}
        RA = {m: v for m, v in RA.items() if v is not None}
        Dl = pd.concat([v["delta"] for v in RA.values()], ignore_index=True)
        Pl = pd.concat([v["planes"] for v in RA.values()], ignore_index=True)
        R.set("b3.n-models", len(RA), "{:d}")
        R.set("b3.n-heads", int(Dl.groupby("model").apply(lambda g: g[["layer", "head"]].drop_duplicates().shape[0]).sum()), "{:,d}")
        R.set("b3.prop2-maxdev", fmt_sci(float(Dl.prop2_dev.max())))
        sa = Dl.groupby(["model", "delta"])["s_align"].mean().unstack()
        R.set("b3.salign0-range", f"{sa[0].min():.3f} to {sa[0].max():.3f}")
        last = max(CFG["rope_offsets"])
        R.set("b3.salign-last-range", f"{sa[last].min():.3f} to {sa[last].max():.3f}")
        tr = Dl.groupby(["model", "delta"])["top_r_rho"].mean().unstack()
        R.set("b3.topr-maxchange", float((tr.sub(tr[0], axis=0)).abs().max().max()), "{:.3f}")
        # frequency dependence of per-plane overlap (rotary planes only)
        fr = []
        for m, g in Pl[Pl.plane >= 0].groupby("model"):
            gm = g.groupby("plane").agg(omega=("omega", "first"), rho=("rho", "mean"))
            r_, lo_, hi_, p_ = spearman_ci(np.log(gm.omega), gm.rho, seed=stable_hash(m) % 999)
            fr.append(dict(model=m, r=r_, lo=lo_, hi=hi_, p=p_))
        FR = pd.DataFrame(fr)
        if len(FR):
            FR["p_holm"] = holm(FR.p.values)
            neg = FR[(FR.r < 0) & (FR.p_holm < 0.05)]; pos = FR[(FR.r > 0) & (FR.p_holm < 0.05)]
            R.set("b3.freq-verdict", f"per-plane Q–K excess decreases significantly with frequency (Holm $p<0.05$) in {len(neg)} of "
                  f"{len(FR)} RoPE models, increases significantly in {len(pos)}, and shows no significant monotone trend in "
                  f"{len(FR) - len(neg) - len(pos)}")
            R.set("b3.freq-range", f"{FR.r.min():+.2f} to {FR.r.max():+.2f}")
        pas = Pl[Pl.plane == -1].groupby("model")["rho"].mean()
        rot = Pl[Pl.plane >= 0].groupby("model")["rho"].mean()
        if len(pas):
            R.set("b3.pass-vs-rot", "; ".join(f"{short(m)}: {pas[m]:.3f} vs. {rot[m]:.3f}" for m in pas.index))
        tab = []
        for m in RA:
            d_ = Dl[Dl.model == m]
            tab.append({"Model": short(m), "rotary fraction": f"{geo(m)[1]['rot_frac']:.2f}" if geo(m)[1] else "--",
                        r"$\rho_{QK}$ (static)": geo(m)[0].query("pair=='QK'").rho.mean() if geo(m)[0] is not None else np.nan,
                        **{rf"$s({dl})$": d_[d_.delta == dl].s_align.mean() for dl in [0, 16, last]},
                        rf"$\rho^{{(r)}}(0)$": d_[d_.delta == 0].top_r_rho.mean(), rf"$\rho^{{(r)}}({last})$": d_[d_.delta == last].top_r_rho.mean()})
        save_table(pd.DataFrame(tab), "tab_rq3_static", float_fmt="{:.3f}")
        fig, ax = plt.subplots(1, 3, figsize=(7.4, 2.5))
        for m in RA:
            g = Dl[Dl.model == m].groupby("delta")
            ax[0].plot(g.s_align.mean().index + 1, g.s_align.mean().values, marker="o", ms=2.5, lw=0.9, label=short(m))
            ax[1].plot(g.top_r_rho.mean().index + 1, g.top_r_rho.mean().values, marker="o", ms=2.5, lw=0.9)
            gp = Pl[(Pl.model == m) & (Pl.plane >= 0)].groupby("plane").agg(omega=("omega", "first"), rho=("rho", "mean"))
            ax[2].plot(gp.omega, gp.rho, lw=0.8)
        ax[0].set_xscale("log"); ax[0].set_xlabel(r"$\delta+1$"); ax[0].set_ylabel(r"$s(\delta)$"); ax[0].set_title("(a) self-alignment of $B_\\delta$")
        ax[1].set_xscale("log"); ax[1].set_xlabel(r"$\delta+1$"); ax[1].set_ylabel(r"$\rho^{(r)}(\delta)$"); ax[1].set_title(f"(b) rank-{CFG['topr']} singular subspaces")
        ax[2].set_xscale("log"); ax[2].set_xlabel(r"plane frequency $\omega_j$"); ax[2].set_ylabel(r"per-plane $\rho_{QK}$"); ax[2].set_title("(c) frequency dependence")
        ax[0].legend(frameon=False, fontsize=5.5)
        fig.tight_layout(); savefig(fig, "fig_rq3_bilinear")

### B3.2 Activation-level analysis (~10k WikiText-103 tokens)
Per head: the canonical-correlation overlap between $q$ and $k$ of the same tokens (a
gauge-invariant, data-weighted analogue of the weight overlap), its shuffled-token null, and the
overlap of the 90%-variance PCA subspaces of $q$ and $k$ before and after the RoPE rotation
(descriptive: PCA angles in head coordinates are not gauge invariant).

In [ ]:
ACT_MODELS = ["openai-community/gpt2", "EleutherAI/pythia-160m", "microsoft/phi-2", "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
              "Qwen/Qwen2.5-0.5B", "meta-llama/Llama-3.2-1B"]

def activation_analysis(mid, model=None, tok=None, ws=None, name=None):
    name = name or mid
    cp_ = None
    if model is None:                   # checkpoint loaded here: the result is cached on disk
        cp_ = result_cache("activations", mid, name, CFG["ctx_eval"], CFG["act_tokens"])
        if cp_.exists():
            print(f"[cache] activations of {mid}")
            return pd.read_pickle(cp_)
        try:
            model, tok = load_model(mid, task="causal")
        except Exception as e:
            record_missing(mid, e); return None
    out_ = _activation_rows(mid, model, tok, ws, name)
    if cp_ is not None:
        out_.to_pickle(cp_)
    return out_

def _activation_rows(mid, model, tok, ws, name):
    ws = ws or ws_from_model(model, mid)
    b0 = next(iter_blocks(ws))          # blocks are re-created lazily below (one layer in memory at a time)
    n_blocks = _n_layers_cfg(ws.config) or sum(1 for _ in iter_blocks(ws))
    ctx = CFG["ctx_eval"]; ntok = CFG["act_tokens"]
    ids = corpus_tokens(tok, "wikitext103", ntok)
    w = ctx - 1
    X = ids[: (len(ids) // w) * w].view(-1, w)
    bos = _bos(tok)
    if bos is not None:
        X = torch.cat([torch.full((X.shape[0], 1), bos), X], 1)
    mt = ws.model_type
    acts, pos = capture_qk(model, mt, n_blocks, b0.n_q, b0.n_kv, b0.dh, X)
    rows = []
    gshuf = torch.Generator().manual_seed(0)
    for blk, (q, k) in zip(iter_blocks(ws), acts):
        for h in range(blk.n_q):
            qh, kh = q[:, h].double(), k[:, blk.kv_of_q[h]].double()
            r = dict(model=name, layer=blk.layer, head=h, cca=cca_overlap(qh, kh),
                     cca_null=cca_overlap(qh, kh[torch.randperm(len(kh), generator=gshuf)]))
            r["pca_pre"], r["kq"], r["kk"] = pca_overlap(qh, kh)
            if blk.rope is not None:
                qr = apply_rope_acts(qh.unsqueeze(1), pos, blk.rope["inv_freq"], blk.rope["rot_dim"])[:, 0]
                kr = apply_rope_acts(kh.unsqueeze(1), pos, blk.rope["inv_freq"], blk.rope["rot_dim"])[:, 0]
                r["pca_post"], _, _ = pca_overlap(qr, kr)
            rows.append(r)
    del acts
    return pd.DataFrame(rows)

if section_enabled("B3"):
    with guard():
        R.cell("B3.2")
        AC = []
        for m in ACT_MODELS:
            a = activation_analysis(m)
            if a is None:
                continue
            g_, _ = geo(m)
            w_ = g_[g_.pair == "QK"][["layer", "head", "rho", "overlap"]].rename(columns={"rho": "w_rho", "overlap": "w_overlap"})
            AC.append(a.merge(w_, on=["layer", "head"]))
            gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
        AC = pd.concat(AC, ignore_index=True)
        AC.to_csv(TAB / "activations_per_head.csv", index=False)
        r_, lo_, hi_, _ = spearman_ci(AC.w_rho, AC.cca)
        R.set("b3.act-weight-cca", fmt_ci(r_, lo_, hi_, 2))
        per = AC.groupby("model").apply(lambda g: st.spearmanr(g.w_rho, g.cca)[0])
        R.set("b3.act-weight-cca-range", f"{per.min():.2f} to {per.max():.2f}")
        R.set("b3.act-cca-excess", float((AC.cca - AC.cca_null).mean()), "{:.3f}")
        R.set("b3.act-cca-null", float(AC.cca_null.mean()), "{:.4f}")
        if "pca_post" in AC:
            R.set("b3.act-pca-change", fmt_ci(*hier_boot(AC.dropna(subset=["pca_post"]).assign(dd=lambda d: d.pca_post - d.pca_pre), "dd")))
        fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.6))
        for m, g in AC.groupby("model"):
            ax[0].scatter(g.w_rho, g.cca, s=4, alpha=0.6, label=short(m))
            if "pca_post" in g and g.pca_post.notna().any():
                ax[1].scatter(g.pca_pre, g.pca_post, s=4, alpha=0.6, label=short(m))
        ax[0].set_xlabel(r"weight-level $\rho_{QK}$"); ax[0].set_ylabel("activation CCA overlap (q, k)"); ax[0].legend(frameon=False, fontsize=5.5, markerscale=2)
        ax[0].set_title("(a) weights vs. activations")
        lim = [0, 1]; ax[1].plot(lim, lim, "k--", lw=0.6)
        ax[1].set_xlabel("PCA overlap before rotation"); ax[1].set_ylabel("PCA overlap after RoPE rotation"); ax[1].set_title("(b) effect of the rotation")
        fig.tight_layout(); savefig(fig, "fig_rq3_activations")

## B6. Broader coverage: encoder–decoder, mixture-of-experts and 7–8B GQA models
Large checkpoints are read layer by layer from safetensors (no model instantiation). For
encoder–decoder models, encoder self-attention, decoder self-attention and cross-attention are
reported separately; in cross-attention Q reads the decoder stream and K/V read the encoder
output, so Q–K and Q–V compare subspaces of two different residual streams (flagged).
For GQA, each query head is compared with its shared key head, and the span of each group's
query heads is compared with the shared key head (`QKgroup`, null $=r\,d_h/d$).

In [ ]:
ALL_MAIN = list(dict.fromkeys(ORIG12 + BROAD + DISTILLED + [f"EleutherAI/pythia-{s}" for s in PYTHIA_SIZES] +
                              ["meta-llama/Llama-3.2-1B", "meta-llama/Llama-2-7b-hf", "allenai/OLMo-2-0425-1B",
                               "google-bert/bert-base-cased"]))

if section_enabled("B6"):
    with guard():
        R.cell("B6.1")
        dfa, infoa = geo_many(ALL_MAIN)
        SA = summarize(dfa)
        SA["regime"] = SA.model.map(regime_of)
        SA.to_csv(TAB / "summary_all_models.csv", index=False)
        infoa.to_csv(TAB / "model_inventory.csv", index=False)
        selfk = SA[SA.kind.isin(["self", "enc-self", "dec-self"])]
        qk = selfk[selfk.pair == "QK"]
        R.set("b6.n-models", SA.model.nunique(), "{:d}")
        R.set("b6.qk-above-null", f"{int((qk.lo > 0).sum())} of {len(qk)}")
        R.set("b6.coupling-pos", f"{int((qk.coupling_lo > 0).sum())} of {int(qk.coupling.notna().sum())}")
        HA = hierarchy_tests(dfa[dfa.kind.isin(["self"])])
        HA["regime"] = HA.model.map(regime_of)
        HA.to_csv(TAB / "hierarchy_all.csv", index=False)
        R.set("b6.hierarchy-verdict", hier_sentence(HA))
        by_reg = HA.groupby("regime").agg(n=("model", "size"), full=("full_hierarchy", "sum"), viol=("violated", "sum"))
        R.set("b6.hier-by-regime", "; ".join(f"{REGIME_LABEL[r]}, complete ordering in {int(v.full)} of {int(v.n)} and at least one significant reversal in {int(v.viol)} of {int(v.n)}"
                                            for r, v in by_reg.reindex([x for x in REGIME_ORDER if x in by_reg.index]).iterrows()))
        # encoder-decoder
        for m, tag in [("google-t5/t5-base", "t5"), ("facebook/bart-base", "bart")]:
            s = SA[SA.model == m]
            for kind in ["enc-self", "dec-self", "cross"]:
                r = s[(s.kind == kind) & (s.pair == "QK")]
                if len(r):
                    R.set(f"b6.{tag}-{kind}-qk", fmt_ci(r["mean"].iloc[0], r["lo"].iloc[0], r["hi"].iloc[0]))
                r = s[(s.kind == kind) & (s.pair == "KV")]
                if len(r):
                    R.set(f"b6.{tag}-{kind}-kv", fmt_ci(r["mean"].iloc[0], r["lo"].iloc[0], r["hi"].iloc[0]))
        # GQA: per-head vs group-level
        gq = dfa[dfa.pair == "QKgroup"].groupby("model").rho.mean()
        ph = dfa[(dfa.pair == "QK") & dfa.model.isin(gq.index)].groupby("model").rho.mean()
        if len(gq):
            R.set("b6.gqa-group-vs-head", "; ".join(f"{short(m)}: {gq[m]:.3f} vs. {ph[m]:.3f}" for m in gq.index))
            R.set("b6.gqa-n", len(gq), "{:d}")
        # MoE and large models
        for m, tag in [("allenai/OLMoE-1B-7B-0924", "olmoe"), ("Qwen/Qwen1.5-MoE-A2.7B", "qwenmoe"), ("meta-llama/Llama-3.1-8B", "llama31"),
                       ("Qwen/Qwen2.5-7B", "qwen7b"), ("mistralai/Mistral-7B-v0.1", "mistral")]:
            r = qk[qk.model == m]
            if len(r):
                R.set(f"b6.{tag}-qk", fmt_ci(r["mean"].iloc[0], r["lo"].iloc[0], r["hi"].iloc[0]))
        # inventory table (appendix) and overview table
        inv_tab = infoa.assign(Model=infoa.model.map(short), Regime=infoa.model.map(regime_of),
                               **{"params (M)": infoa.params / 1e6})[["Model", "Regime", "params (M)", "n_layers", "d_model", "d_head", "n_q", "n_kv", "rot_frac"]]
        inv_tab.columns = ["Model", "Regime", "Params (M)", "$L$", "$d$", "$d_h$", "$n_q$", "$n_{kv}$", "rot."]
        save_table(inv_tab, "tab_inventory", float_fmt="{:.2f}")
        ov = []
        for (m, kind), g in SA.groupby(["model", "kind"], sort=False):
            row = {"Model": short(m), "Attention": kind, "Regime": regime_of(m)}
            for p in PAIRS:
                r = g[g.pair == p]
                row[rf"$\rho_{{{p}}}$"] = fmt_ci(r["mean"].iloc[0], r["lo"].iloc[0], r["hi"].iloc[0], 3) if len(r) else "--"
            r = g[g.pair == "QK"]
            row["coupling"] = f"{r.coupling.iloc[0]:+.3f}" if len(r) and np.isfinite(r.coupling.iloc[0]) else "--"
            ov.append(row)
        save_table(pd.DataFrame(ov), "tab_rq1_all")
        # overview figure, sorted by regime
        order = [m for reg_ in REGIME_ORDER for m in SA[(SA.regime == reg_) & SA.kind.isin(["self", "enc-self"])].model.unique()]
        sel = SA[SA.kind.isin(["self", "enc-self"])].drop_duplicates(["model", "pair"])
        fig, ax = plt.subplots(figsize=(7.4, 3.0))
        bar_ci(ax, sel, order, title="Null-calibrated Q/K/V excess across architectures (self-attention; encoder for enc–dec)")
        if FLOOR or (OUT / "floor.json").exists():
            fl = floor_for("google-bert/bert-base-uncased")[0]
            ax.axhspan(-fl / 2, fl / 2, color="0.85", zorder=0, label="BERT seed floor (width)")
        bounds = np.cumsum([sum(regime_of(m) == r for m in order) for r in REGIME_ORDER])
        for b in bounds[:-1]:
            ax.axvline(b - 0.5, color="0.5", lw=0.6, ls=":")
        savefig(fig, "fig_rq1_overview")

### B6.2 Family-level inference for regime comparisons
Checkpoints of the same family (sizes, instruction-tuned variants, seeds) are not independent, so
regime-level statements are tested with the *family* as the unit: (i) the number of independent
families per regime; (ii) family means of model-level $\rho_{QK}$ and of the ordering contrasts
$\rho_{QK}-\rho_{KV}$ and $\rho_{KV}-\rho_{QV}$, compared between each regime and absolute-PE MHA
with exact permutation tests over families (Holm over regimes and statistics); (iii) a linear mixed
model at the model level with a random intercept per family. Regimes with a single family are
reported descriptively only: their difference is confounded with that family. Note that positional
encoding and grouped-query attention are confounded in pre-trained models (every GQA model uses
RoPE); these tests cannot separate them.

In [ ]:
if section_enabled("B6"):
    with guard():
        R.cell("B6.2")
        sa_ = SA[SA.kind.isin(["self", "enc-self"])]
        ML = sa_.pivot_table(index="model", columns="pair", values="mean").reset_index()
        ML["d1"], ML["d2"] = ML["QK"] - ML["KV"], ML["KV"] - ML["QV"]
        ML["regime"], ML["family"] = ML.model.map(regime_of), ML.model.map(family_of)
        ML.to_csv(TAB / "model_level_by_family.csv", index=False)
        FAM = ML.groupby(["regime", "family"], as_index=False)[["QK", "d1", "d2"]].mean()
        nfam = FAM.groupby("regime").family.nunique()
        R.set("b6.families-per-regime", "; ".join(f"{REGIME_LABEL[r]}: {int(nfam[r])} famil{'ies' if nfam[r] > 1 else 'y'}"
                                                 for r in REGIME_ORDER if r in nfam.index))
        ref = "abs-MHA"
        ft = []
        for stat in ["QK", "d1", "d2"]:
            xa = FAM[FAM.regime == ref][stat].values
            for r in REGIME_ORDER:
                if r == ref or nfam.get(r, 0) < 2 or len(xa) < 2:
                    continue
                xb = FAM[FAM.regime == r][stat].values
                p_, pmin = exact_perm_test(xb, xa)
                ft.append(dict(stat=stat, regime=r, n_fam=len(xb), n_ref=len(xa), diff=xb.mean() - xa.mean(), p=p_, p_min=pmin))
        FT_ = pd.DataFrame(ft)
        if len(FT_):
            FT_["p_holm"] = holm(FT_.p.values)
            FT_.to_csv(TAB / "family_level_tests.csv", index=False)
            lab = {"QK": r"$\rho_{QK}$", "d1": r"$\rho_{QK}-\rho_{KV}$", "d2": r"$\rho_{KV}-\rho_{QV}$"}
            sig = FT_[FT_.p_holm < 0.05]
            R.set("b6.family-verdict", ("at the family level, " + "; ".join(
                f"{lab[r.stat]} differs between {REGIME_LABEL[r.regime]} and absolute-PE MHA families ($\\Delta={r.diff:+.3f}$, Holm {fmt_p(r.p_holm)})"
                for r in sig.itertuples())) if len(sig) else
                  f"at the family level, no regime differs significantly from absolute-PE MHA in $\\rho_{{QK}}$ or in the ordering contrasts "
                  f"after Holm correction ({len(FT_)} tests; smallest attainable $p$ between {FT_.p_min.min():.3f} and {FT_.p_min.max():.3f})")
            tab = FT_.assign(stat=FT_.stat.map(lab), regime=FT_.regime.map(REGIME_LABEL))[["stat", "regime", "n_fam", "n_ref", "diff", "p", "p_min", "p_holm"]]
            save_table(tab.rename(columns={"stat": "Statistic", "regime": "Regime", "n_fam": "families", "n_ref": "ref. families",
                                           "diff": r"$\Delta$ vs. abs.-PE MHA", "p": "$p$", "p_min": r"$p_{\min}$", "p_holm": r"$p_{\mathrm{Holm}}$"}),
                       "tab_family_tests", float_fmt="{:.3f}")
        # mixed model with a random intercept per family (regimes with >= 2 families)
        keep = [r for r in REGIME_ORDER if nfam.get(r, 0) >= 2]
        MM = ML[ML.regime.isin(keep)].copy()
        mm_rows = []
        for stat in ["QK", "d1", "d2"]:
            if ref not in keep:
                continue
            form = f"{stat} ~ C(regime, Treatment('{ref}'))"
            mm = mixed_model(form, MM, "family")
            if mm is None:      # fallback: OLS with family-clustered standard errors
                print(f"[B6.2] mixed model for {stat} failed; using OLS with family-clustered SEs")
                mm = smf.ols(form, MM).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(MM.family)[0]})
            ci = mm.conf_int()
            for t in mm.params.index:
                if t.startswith("C(regime"):
                    mm_rows.append(dict(stat=stat, term=t.split("T.")[-1].rstrip("]"), coef=mm.params[t], lo=ci.loc[t, 0], hi=ci.loc[t, 1], p=mm.pvalues[t]))
        MMR = pd.DataFrame(mm_rows)
        if len(MMR):
            MMR["p_holm"] = holm(MMR.p.values)
            MMR.to_csv(TAB / "family_mixed_model.csv", index=False)
            q = MMR[MMR.stat == "QK"]
            R.set("b6.family-mixed-qk", "; ".join(f"{REGIME_LABEL[r.term]} {r.coef:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}], Holm {fmt_p(r.p_holm)}" for r in q.itertuples()))
        # ordering at the family level: share of each family's models with the complete ordering
        HF = HA.assign(family=HA.model.map(family_of)).groupby(["regime", "family"]).full_hierarchy.mean().reset_index()
        R.set("b6.hier-by-family", "; ".join(
            f"{REGIME_LABEL[r]}: complete ordering in the majority of models of {int((g.full_hierarchy > 0.5).sum())} of {len(g)} famil{'ies' if len(g) > 1 else 'y'}"
            for r in REGIME_ORDER for g in [HF[HF.regime == r]] if len(g)))

## Z. Results summary and fill of the paper source
Lists every result with its producing cell and archives `outputs/` (caches excluded) as `outputs_results.zip`.
If `paper/qkv_geometry_pretrained.tex` exists, every `[[RESULT:id]]` in it is replaced by the LaTeX text stored by the cell
that produced `id` (ids without a value become `\pending{id}`) and figures and tables are copied to
`paper/figures` and `paper/tables` (smoke runs write to `paper_smoke/` instead).

In [ ]:
if section_enabled("Z"):
    with guard():
        R.cell("Z.1")
        env = dict(python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
                   numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__, device=DEVICE,
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, smoke=SMOKE, seed=SEED,
                   missing_checkpoints=MISSING, date=time.strftime("%Y-%m-%d"))
        (OUT / "environment.json").write_text(json.dumps(env, indent=1, default=str))
        src = PAPER_DIR / PAPER_TEX
        filled_name = Path(PAPER_TEX).stem + "_filled.tex"
        dst_dir = PAPER_DIR if not SMOKE else PROJECT_DIR / "paper_smoke"
        dst_dir.mkdir(exist_ok=True, parents=True)
        if src.exists():
            tex = src.read_text()
            ids = sorted(set(re.findall(r"\[\[RESULT:([a-z0-9.\-]+)\]\]", tex)))
            missing = [i for i in ids if i not in R.store]
            unused = sorted(set(R.store) - set(ids))
            def rep(m_):
                rid = m_.group(1)
                return R.text(rid) if rid in R.store else f"\\pending{{{rid}}}"
            filled = re.sub(r"\[\[RESULT:([a-z0-9.\-]+)\]\]", rep, tex)
            (dst_dir / filled_name).write_text(filled)
            for sub, srcdir in [("figures", FIG), ("tables", TAB)]:
                (dst_dir / sub).mkdir(exist_ok=True)
                for f in srcdir.glob("*.pdf" if sub == "figures" else "*.tex"):
                    shutil.copy(f, dst_dir / sub / f.name)
            for extra in ["references.bib", "elsarticle-harv.bst"]:
                if (PAPER_DIR / extra).exists() and dst_dir != PAPER_DIR:
                    shutil.copy(PAPER_DIR / extra, dst_dir / extra)
            report = dict(n_placeholders=len(ids), n_filled=len(ids) - len(missing), missing=missing, produced_but_unused=unused)
            (OUT / "fill_report.json").write_text(json.dumps(report, indent=1))
            print(f"{len(ids) - len(missing)}/{len(ids)} placeholders filled; {len(missing)} pending")
            if missing:
                print("pending:", ", ".join(missing[:60]), "..." if len(missing) > 60 else "")
            if shutil.which("latexmk") and os.environ.get("QKV_COMPILE", "1") == "1":
                r_ = subprocess.run(["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", filled_name],
                                    cwd=dst_dir, capture_output=True, text=True)
                print("latexmk:", "ok" if r_.returncode == 0 else r_.stdout[-2000:])
        else:
            print(PAPER_TEX, "not found in", PAPER_DIR, "(results are in", OUT / "results.json)")
        # make every result recoverable from the notebook itself (e.g. when only the .ipynb is shared)
        print("\n===== RESULTS (id | producing cell | LaTeX text) =====")
        for rid in sorted(R.store):
            print(f"{rid} | {R.store[rid]['cell']} | {R.store[rid]['text']}")
        try:
            import zipfile
            arch = PROJECT_DIR / (("outputs_smoke" if SMOKE else "outputs") + "_results.zip")
            with zipfile.ZipFile(arch, "w", zipfile.ZIP_DEFLATED) as zf:
                for f in OUT.rglob("*"):
                    if f.is_file() and "cache" not in f.relative_to(OUT).parts:
                        zf.write(f, f.relative_to(OUT))
            print("outputs archived ->", arch, "(results.json, figures, tables, csv; caches excluded)")
        except Exception as e:
            print("[zip] failed:", repr(e)[:150])
        if FAILED:
            print("Cells that raised an error (skipped; their results are pending):")
            for k, v in FAILED.items():
                print("  ", k, "->", v[:300])
        if MISSING:
            print("Checkpoints that could not be loaded (analyses use the available subset):")
            for k, v in MISSING.items():
                print("  ", k, "->", v[:120])
        todo = {s_: [c for c in cs if c not in CELLS_DONE] for s_, cs in SECTION_CELLS.items()
                if s_ != "Z" and ("ALL" in SECTIONS or s_ in SECTIONS)}
        todo = {s_: c for s_, c in todo.items() if c}
        status = ("ALL SECTIONS OF THIS NOTEBOOK ARE COMPLETE (results: outputs/results.json, archive: outputs_results.zip)." if not todo else
                  "STILL TO RUN: " + "; ".join(f"{s_} ({', '.join(c)})" for s_, c in todo.items()) + ". Next session: "
                  "attach the output of this version with Add input (Your Work -> this notebook) and run the same "
                  "notebook again (Save & Run All); completed sections are skipped and partial work is reused.")
        (OUT / "STATUS.txt").write_text(status + "\n")
        print("\n===== STATUS =====\n" + status)